# DASO-Net Phase 5B — Kaggle GPU training




In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/README_PHASE5B_KAGGLE.md
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/PACKAGE_MANIFEST.json
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/scripts/verify_phase5b_inputs.py
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/scripts/smoke_test_phase5b_core.py
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/scripts/train_phase5b.py
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/scripts/discover_kaggle_project.py
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/configs/phase5b_kaggle_v1.json
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/notebooks/DASO_Net_Phase5B_Kaggle.ipynb
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/src/dasonet/__init__.py
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/src/dasonet/model/dasonet.py
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/src/dasonet/model/decoding.py
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b/src/dasonet/model/blocks.py
/kaggle

In [1]:
import os, sys, subprocess, json, shutil, zipfile
from pathlib import Path

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


Sun Sep 13 08:07:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# ============================================================
# CELL 2 — Locate/copy Phase 5B code from Kaggle input
# Kaggle has already unpacked the uploaded package.
# ============================================================

import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# Find the mounted Phase5B package by a file that uniquely identifies it.
candidates = list(INPUT.rglob("README_PHASE5B_KAGGLE.md"))

if len(candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one README_PHASE5B_KAGGLE.md, "
        f"found {len(candidates)}:\n{candidates}"
    )

PHASE5B_INPUT_ROOT = candidates[0].parent

print("Mounted Phase5B input:")
print(PHASE5B_INPUT_ROOT)

# Copy the code package to writable Kaggle working storage.
CODE_ROOT = WORK / "DASO-Net-Phase5B-KaggleTraining"

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

shutil.copytree(PHASE5B_INPUT_ROOT, CODE_ROOT)

required = [
    CODE_ROOT / "configs/phase5b_kaggle_v1.json",
    CODE_ROOT / "scripts/verify_phase5b_inputs.py",
    CODE_ROOT / "scripts/smoke_test_phase5b_core.py",
    CODE_ROOT / "scripts/train_phase5b.py",
    CODE_ROOT / "src/dasonet/model/dasonet.py",
    CODE_ROOT / "src/dasonet/datasets/fruitmp_syn.py",
]

missing = [p for p in required if not p.exists()]

if missing:
    raise RuntimeError(
        "Phase5B package copy is incomplete:\n"
        + "\n".join(str(p) for p in missing)
    )

print()
print("CODE_ROOT =", CODE_ROOT)
print("Phase5B code copy: PASS")
print("Files copied:", sum(1 for p in CODE_ROOT.rglob("*") if p.is_file()))

Mounted Phase5B input:
/kaggle/input/datasets/abinavprabhuv/dasonet-phase5b

CODE_ROOT = /kaggle/working/DASO-Net-Phase5B-KaggleTraining
Phase5B code copy: PASS
Files copied: 24


In [6]:
# ============================================================
# CELL 3 — Locate the already-unpacked slim DASO-Net project
# No extraction is needed because Kaggle mounted it as a dataset.
# ============================================================

from pathlib import Path

INPUT = Path("/kaggle/input")

# Find the project root by its frozen recipe manifest.
recipe_hits = list(
    INPUT.rglob(
        "DASO-Net-Phase1A/data/fruitmp_syn/"
        "v1_1_recipes/fruitmp_syn_v1_1_train_recipes.csv"
    )
)

if len(recipe_hits) != 1:
    raise RuntimeError(
        f"Expected exactly one frozen training recipe manifest, "
        f"found {len(recipe_hits)}:\n{recipe_hits}"
    )

PROJECT_ROOT = recipe_hits[0].parents[3]

required = [
    PROJECT_ROOT
    / "data/fruitmp_syn/v1_1_recipes/"
      "fruitmp_syn_v1_1_train_recipes.csv",

    PROJECT_ROOT
    / "data/fruitmp_syn/v1_1_recipes/"
      "fruitmp_syn_v1_1_val_recipes.csv",

    PROJECT_ROOT
    / "data/public_real_mp/v1_processed/"
      "almeida_crpa_split_v1_candidate.csv",

    PROJECT_ROOT
    / "data/fruit_bg/v1_final/"
      "fruitbg_rgb_v1_manifest.csv",

    PROJECT_ROOT
    / "data/mp_morph/v1/"
      "morphology_manifest_v1.csv",

    PROJECT_ROOT
    / "configs/renderer_v3_1.json",

    PROJECT_ROOT
    / "src/dasonet/synth/renderer.py",
]

missing = [p for p in required if not p.exists()]

if missing:
    raise RuntimeError(
        "Slim project input is incomplete:\n"
        + "\n".join(str(p) for p in missing)
    )

print("PROJECT_ROOT =", PROJECT_ROOT)
print("Slim project discovery: PASS")
print(
    "Files visible:",
    sum(1 for p in PROJECT_ROOT.rglob("*") if p.is_file())
)

print()
print("Important:")
print("  Project data remains read-only in /kaggle/input.")
print("  Training outputs will go to /kaggle/working.")

PROJECT_ROOT = /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Slim project discovery: PASS
Files visible: 4348

Important:
  Project data remains read-only in /kaggle/input.
  Training outputs will go to /kaggle/working.


In [7]:
# Verify immutable synthetic hashes + Almeida leakage-safe split before any GPU training.
cmd = [
    sys.executable, str(CODE_ROOT / "scripts" / "verify_phase5b_inputs.py"),
    "--project-root", str(PROJECT_ROOT),
]
subprocess.run(cmd, check=True)


=== DASO-Net Phase 5B input verification ===
Project root: /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
FruitMP-Syn train recipes: 30000
FruitMP-Syn val recipes:   5000
Frozen integrity audit:    PASS
Almeida rows:              1140
Almeida perceptual leakage:0
Almeida exact leakage:     0
Hash mismatches:           0
Phase 5B inputs:           PASS


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/verify_phase5b_inputs.py', '--project-root', '/kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A'], returncode=0)

In [10]:
%%writefile /kaggle/working/DASO-Net-Phase5B-KaggleTraining/src/dasonet/losses/instance.py
from __future__ import annotations

import math
from typing import Iterable

import torch
from torch import nn
import torch.nn.functional as F


def gaussian2d(
    radius: int,
    device: torch.device,
    dtype: torch.dtype,
) -> torch.Tensor:
    diameter = 2 * radius + 1
    y = torch.arange(
        -radius,
        radius + 1,
        device=device,
        dtype=dtype,
    )
    x = torch.arange(
        -radius,
        radius + 1,
        device=device,
        dtype=dtype,
    )
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    sigma = max(diameter / 6.0, 1e-3)
    return torch.exp(-(xx * xx + yy * yy) / (2.0 * sigma * sigma))


def draw_gaussian(
    heatmap: torch.Tensor,
    center_x: int,
    center_y: int,
    radius: int,
) -> None:
    h, w = heatmap.shape[-2:]
    g = gaussian2d(radius, heatmap.device, heatmap.dtype)

    left = min(center_x, radius)
    right = min(w - center_x - 1, radius)
    top = min(center_y, radius)
    bottom = min(h - center_y - 1, radius)

    if left < 0 or right < 0 or top < 0 or bottom < 0:
        return

    hm_patch = heatmap[
        center_y - top:center_y + bottom + 1,
        center_x - left:center_x + right + 1,
    ]
    g_patch = g[
        radius - top:radius + bottom + 1,
        radius - left:radius + right + 1,
    ]
    torch.maximum(hm_patch, g_patch, out=hm_patch)


def modified_focal_loss(
    logits: torch.Tensor,
    target: torch.Tensor,
) -> torch.Tensor:
    pred = torch.sigmoid(logits).clamp(1e-5, 1.0 - 1e-5)

    pos = target.eq(1.0).float()
    neg = target.lt(1.0).float()
    neg_weights = torch.pow(1.0 - target, 4.0)

    pos_loss = -torch.log(pred) * torch.pow(1.0 - pred, 2.0) * pos
    neg_loss = (
        -torch.log(1.0 - pred)
        * torch.pow(pred, 2.0)
        * neg_weights
        * neg
    )

    num_pos = pos.sum()
    if num_pos < 1:
        return neg_loss.sum()
    return (pos_loss.sum() + neg_loss.sum()) / num_pos


def dice_loss_from_logits(
    logits: torch.Tensor,
    target: torch.Tensor,
) -> torch.Tensor:
    prob = torch.sigmoid(logits)
    prob = prob.reshape(-1)
    target = target.reshape(-1).float()

    inter = (prob * target).sum()
    denom = prob.sum() + target.sum()
    return 1.0 - (2.0 * inter + 1.0) / (denom + 1.0)


def boundary_from_union_mask(mask: torch.Tensor) -> torch.Tensor:
    """
    mask: [1,H,W] float/bool union mask
    """
    x = mask.float().unsqueeze(0)
    dilated = F.max_pool2d(x, 3, stride=1, padding=1)
    eroded = -F.max_pool2d(-x, 3, stride=1, padding=1)
    boundary = (dilated - eroded).clamp(0, 1)
    return boundary[0]


class InstanceSegLoss(nn.Module):
    def __init__(
        self,
        stride: int = 4,
        center_weight: float = 1.0,
        size_weight: float = 0.10,
        offset_weight: float = 1.0,
        mask_bce_weight: float = 1.0,
        mask_dice_weight: float = 1.0,
        boundary_weight: float = 0.25,
    ) -> None:
        super().__init__()
        self.stride = int(stride)
        self.center_weight = float(center_weight)
        self.size_weight = float(size_weight)
        self.offset_weight = float(offset_weight)
        self.mask_bce_weight = float(mask_bce_weight)
        self.mask_dice_weight = float(mask_dice_weight)
        self.boundary_weight = float(boundary_weight)

    def forward(
        self,
        outputs: dict,
        targets: list[dict[str, torch.Tensor]],
        input_size: tuple[int, int],
    ) -> dict[str, torch.Tensor]:
        center_logits = outputs["center_logits"]
        size_pred = outputs["size"]
        offset_pred = outputs["offset"]
        coeff_pred = outputs["coeff"]
        prototypes = outputs["prototypes"]
        boundary_logits = outputs["boundary_logits"]

        bsz, _, fh, fw = center_logits.shape

        if len(targets) != bsz:
            raise ValueError(
                f"Expected {bsz} target dictionaries, got {len(targets)}."
            )

        heatmap_target = torch.zeros_like(center_logits)
        boundary_target = torch.zeros_like(boundary_logits)

        size_losses = []
        offset_losses = []
        mask_bce_losses = []
        mask_dice_losses = []
        instance_count = 0

        for b, target in enumerate(targets):
            boxes = target["boxes"].to(center_logits.device).float()
            masks = target["masks"].to(center_logits.device).float()

            if boxes.numel() == 0:
                continue

            if masks.ndim != 3:
                raise ValueError("target['masks'] must have shape [N,H,W].")

            union = masks.amax(dim=0, keepdim=True)
            union_small = F.interpolate(
                union.unsqueeze(0),
                size=(fh, fw),
                mode="nearest",
            )[0]
            boundary_target[b] = boundary_from_union_mask(union_small)

            for n in range(boxes.shape[0]):
                x1, y1, x2, y2 = boxes[n]

                cx_in = 0.5 * (x1 + x2)
                cy_in = 0.5 * (y1 + y2)

                cx = cx_in / self.stride
                cy = cy_in / self.stride

                ix = int(torch.round(cx).item())
                iy = int(torch.round(cy).item())

                if not (0 <= ix < fw and 0 <= iy < fh):
                    continue

                w_feat = (x2 - x1).clamp(min=1.0) / self.stride
                h_feat = (y2 - y1).clamp(min=1.0) / self.stride

                radius = max(
                    1,
                    int(
                        min(
                            float(w_feat.item()),
                            float(h_feat.item()),
                        ) / 3.0
                    ),
                )
                radius = min(radius, 6)

                draw_gaussian(
                    heatmap_target[b, 0],
                    ix,
                    iy,
                    radius,
                )

                size_losses.append(
                    F.l1_loss(
                        size_pred[b, :, iy, ix],
                        torch.stack([w_feat, h_feat]),
                        reduction="mean",
                    )
                )

                offset_target = torch.stack([
                    cx - float(ix),
                    cy - float(iy),
                ])

                offset_losses.append(
                    F.l1_loss(
                        offset_pred[b, :, iy, ix],
                        offset_target,
                        reduction="mean",
                    )
                )

                coeff = coeff_pred[b, :, iy, ix]

                mask_logits = torch.einsum(
                    "k,khw->hw",
                    coeff,
                    prototypes[b],
                )

                gt_small = F.interpolate(
                    masks[n][None, None],
                    size=(fh, fw),
                    mode="nearest",
                )[0, 0]

                fx1 = max(
                    0,
                    int(math.floor(float(x1.item()) / self.stride)) - 1,
                )
                fy1 = max(
                    0,
                    int(math.floor(float(y1.item()) / self.stride)) - 1,
                )
                fx2 = min(
                    fw,
                    int(math.ceil(float(x2.item()) / self.stride)) + 2,
                )
                fy2 = min(
                    fh,
                    int(math.ceil(float(y2.item()) / self.stride)) + 2,
                )

                if fx2 <= fx1 or fy2 <= fy1:
                    continue

                pred_roi = mask_logits[
                    fy1:fy2,
                    fx1:fx2,
                ]
                gt_roi = gt_small[
                    fy1:fy2,
                    fx1:fx2,
                ]

                mask_bce_losses.append(
                    F.binary_cross_entropy_with_logits(
                        pred_roi,
                        gt_roi,
                    )
                )

                mask_dice_losses.append(
                    dice_loss_from_logits(
                        pred_roi,
                        gt_roi,
                    )
                )

                instance_count += 1

        center_loss = modified_focal_loss(
            center_logits,
            heatmap_target,
        )

        # DDP-safe zero-particle handling.
        #
        # FruitMP-Syn intentionally contains negative scenes. Under DDP one
        # rank may receive a negative scene while another receives a positive
        # scene. Every prediction head therefore remains connected to the
        # graph even when its supervised term is numerically zero.
        size_loss = (
            torch.stack(size_losses).mean()
            if size_losses
            else size_pred.sum() * 0.0
        )

        offset_loss = (
            torch.stack(offset_losses).mean()
            if offset_losses
            else offset_pred.sum() * 0.0
        )

        mask_zero = (
            coeff_pred.sum()
            + prototypes.sum()
        ) * 0.0

        mask_bce = (
            torch.stack(mask_bce_losses).mean()
            if mask_bce_losses
            else mask_zero
        )

        mask_dice = (
            torch.stack(mask_dice_losses).mean()
            if mask_dice_losses
            else mask_zero
        )

        boundary = F.binary_cross_entropy_with_logits(
            boundary_logits,
            boundary_target,
        )

        total = (
            self.center_weight * center_loss
            + self.size_weight * size_loss
            + self.offset_weight * offset_loss
            + self.mask_bce_weight * mask_bce
            + self.mask_dice_weight * mask_dice
            + self.boundary_weight * boundary
        )

        return {
            "total": total,
            "center": center_loss,
            "size": size_loss,
            "offset": offset_loss,
            "mask_bce": mask_bce,
            "mask_dice": mask_dice,
            "boundary": boundary,
            "instances_used": torch.tensor(
                float(instance_count),
                device=center_logits.device,
            ),
        }

Overwriting /kaggle/working/DASO-Net-Phase5B-KaggleTraining/src/dasonet/losses/instance.py


In [11]:
# Lightweight code-level smoke test.
subprocess.run([sys.executable, str(CODE_ROOT / "scripts" / "smoke_test_phase5b_core.py")], check=True)


=== Phase 5B core smoke ===
Parameters: 1,160,679
Synthetic embeddings: (1, 64)
Instance loss: 6.918230
PCC loss:      0.000007
CRPA loss:     0.635036
Zero-scene anchor BCE: 0.693067
Backward: PASS
Phase 5B core smoke: PASS


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/smoke_test_phase5b_core.py'], returncode=0)

## GPU pipeline smoke

This uses real frozen 1024×1024 virtual recipes, PCC clean/composite pairing, and Almeida CRPA anchors. On T4×2 it launches DDP with one sample per GPU. On a single P100/T4 it runs one process.


In [12]:
OUTPUT_ROOT = Path("/kaggle/working")
trainer = str(CODE_ROOT / "scripts" / "train_phase5b.py")
base_args = [
    "--project-root", str(PROJECT_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--smoke",
    "--log-every", "1",
]

if torch.cuda.device_count() >= 2:
    cmd = ["torchrun", "--standalone", "--nproc_per_node=2", trainer] + base_args
else:
    cmd = [sys.executable, trainer] + base_args

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --output-root /kaggle/working --smoke --log-every 1


W0913 08:29:54.709000 221 torch/distributed/run.py:852] 
W0913 08:29:54.709000 221 torch/distributed/run.py:852] *****************************************
W0913 08:29:54.709000 221 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 08:29:54.709000 221 torch/distributed/run.py:852] *****************************************
[W913 08:29:54.055736742 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 08:29:57.813602833 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 08:29:57.836857009 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:576: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please

=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5b_pilot_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Gradient accumulation:      2
Effective synthetic batch:  4
AMP fp16:                   True
Train steps/epoch/rank:     12
Optimizer steps total:      6
Smoke mode:                 True


/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:653: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:653: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:653: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):


epoch=00 step=00000/00012 loss=27.7665 inst=27.1653 pcc=0.0000 crpa=0.6012 lr=6.00e-05 vram_alloc=8.84GB vram_res=9.10GB


/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:653: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:673: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:673: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimize

... (21 outputs left out)

/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:462: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:462: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:462: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:462: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('

EPOCH 0 COMPLETE | train=56.8992 val=24.7232 best=24.7232 time=1.9m
Phase 5B run finished successfully.
Status: /kaggle/working/dasonet_phase5b/dasonet_phase5b_pilot_v1/run_status.json


/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py', '--project-root', '/kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A', '--output-root', '/kaggle/working', '--smoke', '--log-every', '1'], returncode=0)

In [13]:
# Inspect persistent output footprint.
out = Path("/kaggle/working/dasonet_phase5b/dasonet_phase5b_pilot_v1")
for p in sorted(out.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(out)}  {p.stat().st_size/1024/1024:.2f} MB")

status = out / "run_status.json"
if status.exists():
    print(json.dumps(json.loads(status.read_text()), indent=2))


best.pt  4.58 MB
history.csv  0.00 MB
last.pt  4.58 MB
resolved_config.json  0.00 MB
run_status.json  0.00 MB
{
  "status": "smoke_passed",
  "epochs_completed": 1,
  "best_val_total_loss": 24.723157703876495,
  "best_checkpoint": "/kaggle/working/dasonet_phase5b/dasonet_phase5b_pilot_v1/best.pt",
  "last_checkpoint": "/kaggle/working/dasonet_phase5b/dasonet_phase5b_pilot_v1/last.pt",
  "history": "/kaggle/working/dasonet_phase5b/dasonet_phase5b_pilot_v1/history.csv",
  "storage_policy": {
    "rendered_cache_written": false,
    "prediction_dump_written": false,
    "persistent_checkpoints": [
      "best.pt",
      "last.pt"
    ]
  }
}


## Optional one-epoch pilot

Run this only after the smoke cell passes. This is the first meaningful throughput/VRAM test, not the final experiment.


In [14]:
# ============================================================
# PHASE 5B — bounded T4x2 throughput/stability pilot
#
# Purpose:
#   - exercise the real DataLoader worker configuration
#   - use the intended accumulation=4
#   - measure realistic runtime and VRAM
#   - avoid launching a full 30,000-scene epoch yet
#
# This is an engineering pilot only, NOT a paper-result model.
# ============================================================

import copy
import json
import subprocess
import sys
import time
from pathlib import Path

# IMPORTANT:
# Do not rerun the Phase5B package-copy cell before this.
# The working tree currently contains the DDP-safe instance-loss fix.

BASE_CONFIG = (
    CODE_ROOT
    / "configs"
    / "phase5b_kaggle_v1.json"
)

PILOT_CONFIG = Path(
    "/kaggle/working/phase5b_throughput_pilot_v1.json"
)

with BASE_CONFIG.open("r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg = copy.deepcopy(cfg)

# Separate output directory so the successful smoke artifacts
# are not overwritten.
cfg["experiment_name"] = "dasonet_phase5b_throughput_v1"

# Intended T4x2 starting setup.
cfg["data"]["per_rank_synthetic_batch"] = 1
cfg["data"]["per_rank_anchor_batch"] = 2
cfg["data"]["synthetic_workers_per_rank"] = 2
cfg["data"]["anchor_workers_per_rank"] = 1

cfg["optimization"]["gradient_accumulation_steps"] = 4
cfg["optimization"]["amp"] = True

with PILOT_CONFIG.open("w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

trainer = str(
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

base_args = [
    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(PILOT_CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "1",

    # Deliberately bounded.
    "--max-train-steps",
    "100",

    "--max-val-steps",
    "20",

    "--accum",
    "4",

    "--log-every",
    "20",

    "--resume",
    "none",
]

if torch.cuda.device_count() >= 2:
    cmd = [
        "torchrun",
        "--standalone",
        "--nproc_per_node=2",
        trainer,
    ] + base_args
else:
    cmd = [
        sys.executable,
        trainer,
    ] + base_args

print("=== Phase 5B bounded throughput pilot ===")
print("Command:")
print(" ".join(cmd))
print()

start = time.time()

result = subprocess.run(
    cmd,
    check=True,
)

elapsed = time.time() - start

print()
print("========================================")
print("THROUGHPUT PILOT COMPLETE")
print("Return code:", result.returncode)
print(
    "Wall time:",
    f"{elapsed / 60.0:.2f} minutes",
)

out = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5b_throughput_v1"
)

print()
print("Output:", out)

if out.exists():
    total_bytes = sum(
        p.stat().st_size
        for p in out.rglob("*")
        if p.is_file()
    )

    print(
        "Output footprint:",
        f"{total_bytes / 1024**2:.2f} MB",
    )

    status = out / "run_status.json"

    if status.exists():
        print()
        print("run_status.json:")
        print(status.read_text())

print()
print(
    "Approx train-step wall time:",
    f"{elapsed / 100.0:.2f} s/step",
    "(includes validation/startup overhead, so this is conservative)"
)

=== Phase 5B bounded throughput pilot ===
Command:
torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5b_throughput_pilot_v1.json --output-root /kaggle/working --epochs 1 --max-train-steps 100 --max-val-steps 20 --accum 4 --log-every 20 --resume none



W0913 08:35:12.616000 269 torch/distributed/run.py:852] 
W0913 08:35:12.616000 269 torch/distributed/run.py:852] *****************************************
W0913 08:35:12.616000 269 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 08:35:12.616000 269 torch/distributed/run.py:852] *****************************************
[W913 08:35:12.030777885 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 08:35:15.882187248 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 08:35:15.899037752 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:576: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please

=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5b_throughput_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     100
Optimizer steps total:      25
Smoke mode:                 False


/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:653: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:653: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):


epoch=00 step=00000/00100 loss=27.7644 inst=27.1603 pcc=0.0000 crpa=0.6041 lr=1.25e-05 vram_alloc=8.84GB vram_res=9.11GB


/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:673: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:673: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch=00 step=00020/00100 loss=27.3389 inst=26.7295 pcc=0.0000 crpa=0.6094 lr=7.50e-05 vram_alloc=8.95GB vram_res=9.11GB
epoch=00 step=00040/00100 loss=26.9589 inst=26.3559 pcc=0.0001 crpa=0.6029 lr=1.37e-04 vram_alloc=8.96GB vram_res=9.11GB
epoch=00 step=00060/00100 loss=12.4180 inst=11.8313 pcc=0.0003 crpa=0.5864 lr=2.00e-04 vram_alloc=8.96GB vram_res=9.11GB
epoch=00 step=00080/00100 loss=8.3982 inst=7.8168 pcc=0.0006 crpa=0.5808 lr=2.62e-04 vram_alloc=8.97GB vram_res=9.11GB
epoch=00 step=00099/00100 loss=6.7922 inst=6.1959 pcc=0.0000 crpa=0.5963 lr=3.00e-04 vram_alloc=8.97GB vram_res=9.11GB


/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:462: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py:462: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):


EPOCH 0 COMPLETE | train=41.5686 val=6.3576 best=6.3576 time=3.9m
Phase 5B run finished successfully.
Status: /kaggle/working/dasonet_phase5b/dasonet_phase5b_throughput_v1/run_status.json


/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)



THROUGHPUT PILOT COMPLETE
Return code: 0
Wall time: 4.05 minutes

Output: /kaggle/working/dasonet_phase5b/dasonet_phase5b_throughput_v1
Output footprint: 27.25 MB

run_status.json:
{
  "status": "training_complete",
  "epochs_completed": 1,
  "best_val_total_loss": 6.35759861767292,
  "best_checkpoint": "/kaggle/working/dasonet_phase5b/dasonet_phase5b_throughput_v1/best.pt",
  "last_checkpoint": "/kaggle/working/dasonet_phase5b/dasonet_phase5b_throughput_v1/last.pt",
  "history": "/kaggle/working/dasonet_phase5b/dasonet_phase5b_throughput_v1/history.csv",
  "storage_policy": {
    "rendered_cache_written": false,
    "prediction_dump_written": false,
    "persistent_checkpoints": [
      "best.pt",
      "last.pt"
    ]
  }
}

Approx train-step wall time: 2.43 s/step (includes validation/startup overhead, so this is conservative)


In [15]:
# ============================================================
# PHASE 5B — COMPLETE hardened train_phase5b.py replacement
#
# Fixes:
#   1. modern torch.amp API
#   2. scheduler advances only after a real optimizer update
#   3. tracks AMP-skipped optimizer updates
#   4. explicit NCCL device/barrier handling
#   5. correct tail gradient-accumulation divisor
#   6. records actual CLI/runtime overrides for reproducibility
#   7. reports real DataLoader worker counts
#   8. inference_mode validation
#
# This payload contains the COMPLETE rewritten file.
# ============================================================

from pathlib import Path
import base64
import hashlib
import subprocess
import sys
import zlib

TARGET = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining/"
    "scripts/train_phase5b.py"
)

EXPECTED_SHA256 = (
    "e24de3c724da25adb01a45b03c5ae3de"
    "617014a07953fc5fb23a40ba845105f4"
)

PAYLOAD = (
"eNrtPWtv3EaS3/UreFwswrGp0SO2kcztBPDaTm4vji3YCQKcIBCcmR4NV3wdyZGl1eq/X1X1q5psUrLjPeCAExJrpruqurq6urq6urq1baoiSJLtvts3IkmC"
"rKirpgvSsqy6tMuqsj040GXNZZ02rdDf11XZiZsuz1ampL3WH3dpu2M1f2+rUn8u0m6nP1et/tSk5aYq9Lf21lR0WSEOtshmDYhAU/N4hnSooruts/JSl/+t"
"E03aVc3BwdnLVz+//OlN8uH9+1+DJSFE0Ncsh57O5o1oq/xaRLM5dEuUXXt+cnHw8cMrA86xj4KwbdbhQbYN2q6JNNgsADkFWYkMz5G/xUEAP/rbPCtb0XTR"
"cexizYxQy31R3wZpG5S16XDVrHfOl/kmA/Rste/EBmHxq+o5VZcldiHNc5FrIby2GK/TLj3T1YD9+vUZR953Wd7ONwBkcOHz2yrdiMYP57AzbO9jWtQ54Erk"
"TQpDL7p5UW0Ydy8/vn8nOhcir9pWtGYUy7ZLy7X4KC7fQkUcnL16JT+8+nD2Un66BBGLJlnD4MEvUazEZgOK0Lp0uybNSqYfEQ3Rj80+6345+3hb/or1Lzdp"
"DURiqnuZFyLbgNCyRmxelutd1aBMWtHJ+nWV52knkva2BAa6bC2LWyE2yaequdJ0VDepgb/uy00u4gMY+oON2AY0lRKYU200Cw5/MLNr/i4tRFunayFViQob"
"UEcD8LK53BfQ5TOqiTaiXTdZjXN1GWKLh9BkcAYTUATP/xr8nF5e5iIgIYgmnDGi83SzQQ6IWhQeHtZN9Xex7g6bqurCOGjEf+9RAstfm70YR6QK/AEKYBO2"
"2WUYmzLoarrPuyWqf39CSeA2xM818vt8lVwRu8n1yRxNRjiTlCa5rvZdvTdM6wbDI0nqCAcERn+652AL4DNHL0F34PtO5LX8EvwzSPddBb+OcGofddXReifW"
"V3WVld287qYbaIvqCumlazlQLUwqkXQg2Gk8UVfrXQuIYOPEElqyLL4DpiZxi/TmkMb9sO1E7SdyrLt4BkpGjf17cAzatt3neZCTEQgfbOQ6zSeaQD7/aCvp"
"er0vvkwKtWgOYXG5Olyl3Xr3ZTRSsgF/hALY+UNpGr5wMMvqEOzqF2hQXl0eimvR3HrbPX0ukRsBDkCpaXDjpOwVs/kJGMJ9Lc0WfMjF+aqq8jggwvYftXSJ"
"62wtLqQxAwnkgJ79Q8DgA1BUtXNRXmdNVc4vBbD7+/sPb18nH//2X2+A2/AknEnu8mqd5gmO4gje2/evXr5NPrx89zPiHWu8CYwBLF/UlpzTH4KTA4KAtR9X"
"e9mv9X6TzrM2Sa/TLE9XOfgRC2P1YNKB9f2wL9F5edM0VROFxiLb9ahV1vnwp7PfgqrMb/89ePXb65dYgQ0Z0nMYYc0BY9O2x1iCsUmkzCMrtRmzx1gFHeSj"
"E4WICtLwYkCD4MdkXQLLw1q0bXLZVDD8BgB/Vun6SpQbsJTrdc7MPzGNuIXodhXUwxgsjo56EJKLJNss5SdbK9kQeSsWj+zC8ewBqchfSqBK7ZlQY1IaLoqY"
"KUOsWleTYp2LtNzXCcOP+AAFcmJInIXDMM0enPULz8iC/71Rgm8TlF+W5tD6husYVa/SpsnICVASbJfnqO3yOwzbRtzMLnqDCR5D11S3vfHUEx1FhZ5MhP8s"
"5FRGMdDHHtdg+Pd6KiP4LHhKn0mDgifByTH8fKunIvj3c6JMaJKpsp4PaoI/B9HpkyffnqqZKcVWgKRhRPoE2CAziAS8XQ2lurVLT5+/IOc/IkedtgPUH5CG"
"7M4OuqI2LnMJH8lGPmXdjrYf86oWZRQ2q3CGrvTWDse2aoIVKM0V7gYycCWjPC1Wm3QRbGGnkW6ik+PTZySS02ezOFiFIRtLany+r8G7FhFRcazybr4TN5vs"
"EsbNDBP50wm6/TC10AmuwNpEVLqw24Jfqs0+l7oG47LgRMHhieo5rEsiB1OO7NfIOlGYG7qwAJgW0yuR1Luqq6CmydbJddpkKTR6oxv8VZRt1ch1gRXIZsMw"
"BMU5BNcWtTnI8nxfZCXtMQOiRJ+QjxaaPgQOwLEGzxX8/jngEo0VDNANDk0tzo8vqOgSjKmxBaKou9soWsXBifxvpiff8kbPu/m+zKCVIomO5999BzDzk1Mp"
"7VUGQ/plpA6P58ffPgfrg79mirGiSL+Ys+9PkbPj7yStW0NnDVpVRzegSNTxp8Q0tUvwffC6+hRxxNs4AKEvT8Th81ksWXQ0rQdryWqluwaXA/wC0bWR+r0I"
"cjAq55ts3Z3DTIqdob+4mLB+k3hSaf4U4E4P/MSSvJ42SHOcTLfESdARbAuYAezDsiaQWwHV4jz4WQgKDECloiZ5BlVur1pYcYNXsPDiKp0HJVgNMLx59QmU"
"PgCnsQXuwIUKcG0uRFE1t0EN24R234i5IzIpBr0i3ILOJtJfTKRnGykXBj8v2OZabRrlbnnh2UHDVoN8Z7kGotcszfEBSU+HOUiCSlzUOgz+sbRaOzB2AW7e"
"rKXB+IWirrwMa831jwKgRZOajehfZc9B56iZmWv60DVG+6H66RC8zUS+kSCmXLL6dInOFUkONpp7WMJq2HRLy52AhQLrCQRhBSLzplYjn0LFwWDdZf6pIgLY"
"SlBSc8w0kV+jc9NuLNuNqNmZUeKlaYw8aYlMkC+ezSY9NFp4YVFKZDcj2WIcVPWSqj5Q8ft6/vG3X5wZKfmQ4GDz5rC0FGCTY7nOAnubyAE5MSDGcK/2Gbgv"
"ZNij9fZyEaDGkHSGAQrJcQGSAcjzkJDCCxX1aFC3FI71/2CaFMl6l5Yl+Gm4u4iK89ApDC9m1qVrnw2Bn42AfjcE/W4E9OTFEJaVOcDbuhwA8zIHGFZI9JS6"
"CofcgLulDoKJQ4FbWGh4p9CAOyM9HIwIRe6OogyS2WEcMa/KqSzbLgFQPZqIHF6ch5mKramBrddrHxQU65Fv6tQHgeUAYtqC+l7YjmtJk23E8pmVkwrbfRLZ"
"5a5bSiXWHANtXusONXjCY1iszsGptlu0ZiNYTq2DhwtFslqPtterH+JusgeQGYCDvcKpnTa3Y8i9eqtS865y9zgwjhjNluHT/oC0yzvp2UfX0gmE7c412nGl"
"FDiTCS6UdgX8wXvL5OU+bTbJVqR0elBnNzihHiLoQ/JSh9W8haZdCRgybq0jO+07jmD2qidEhwoOstMRZyu8AvZ9LnU9R2BYbZ3DFUxm2J2MIPFK1/yIS/CM"
"we0qwM/IRhv1gzmUlFMy0GZLZADh52RdtVmpWxrnxAHzC5lZP9TsGJUkJqnzPYd0KZT0VYw6wXCv3MOpmLyxiP1ojtqFm4CQcqLceBQrpFAtRwJnkAIBCXkw"
"DFLLq1eMa4OK9WmHTdpjPDzhdhS/a/tJUSE8UYBK/9lExLseByFhhDFuwjPcOCbbpvqHKBPcwcK8tkH7a9wUfwZhgH8MWcmxlAHQHj80sdPGachaojrPuqXu"
"j3VVi/RS0ODQIqqFZ9TU1jtqqsNrxClrBPx7oiONGnzrrcIopa/XGZLhV+vKj2nePrYv04FCrWWJ3gUsPRsPN7ZncGLS7EZAD9dpu+RRMZweSxksa3f77TYX"
"UvwjrAabpqqTPG07AuuF+5jOfgabCuPzmJSCZexQgYcfruyfLTuJ9tXEBzuwk5NHyvCzObZI/wJJuoFcnzIOx31E8t7O4V72QFv2Kx0Vww0xbXzRsBMP5J8w"
"Mz3j5lyVebCKrIy881ehJHqlIB3nLMz0QqKiCsoeO1z+oLbtDMrYox5rCKr2ZEVR6WUDVz87mLQqSZvjLmDu1kaRXHJWmCcFopfBjyWuh6zntsYxWYoMxfy3"
"5XJwJM6Pz6F+5CS9JwbNoys/dsjsKh236jCAfSEvGCNaetCfRmwFSmybrmErFV6osDbrcR9i1lu5laOCk8wEeaKhtjM7LhV3OZgIDERNr+FcyVoWHXKl4K58"
"T56gR37tOOO259rvRkldBf+2DEJDJ7zvr48PdlIb4UEXe/N62MHeEvfkiWXRWdlUsZkZo0rPnTO/yruT6l+s9NPqrCzu5FgahR4ag4FKS5A/rtVOIHNSsfs9"
"cHXbtdRj6t2z5wMNN0rBG+t7bw8r6BirwyXl8WrqcsQ3Nnc2P8ZM4nAxsBqWZqjmCgC5cy7uU5KNGmJO/3v0DOhAULGHP91/h8+BTMLepGYMD2F9I9zn3N/C"
"AGdqnEKzkuj5AQj+ta23eAOczyDc68B+VdTgNOkAnDwwoihdUu07vvXE/qstoS2kwA3igoFYDJLbTBjOQOhsNxt/M1Um/+3AiXWsq1Jak/Wt3ryyCDg/XImD"
"6UMX7Crs3TvqGcwj202MU6g6FRpUpx8AZbp9HqpCBZKVIDid9EHckBN1I8NDNzJ70uCyBtRB3+HpQhslFWTkwowcbmPNUMyaXdqPig62Jwo8TRxLIXTJ8vht"
"kdbhBWtHBza1HaIzUUWd/FFltX0DRef8XLwMnTfnP6tx2xrtySPoj3ZH8c7Q5KCgo6CEKAsWvU3EQAosFMkU3RpmSqXQUcF26eicUwXc8tVODZJB7I2bD0ts"
"t7hvx/DrkmkeK+bQZtyszNKlViBXNnyY5I7BXb88pFZLhu8sH9CL3tyjfmFkTRlzOb1KcdmDMxE4B07FMrktYcFgYzWt4uhOWv4bAZbXcMEgoYxNES49QjEM"
"MRQom0IxreTVZdapFuTncfIKlki7sNqpqbo0VwYEzBR+g8n1FDWSf0UZme/yQIwyHnBA+HJOAAtJla0r5nhl4TbEQPBsZeE061YmKrqtgfTXPpjWKA1nvnNA"
"OqZZuL3qVScYtDYw9GUAQiFqAyO/DYB6cWgD3i8fINrYs8FhRRce8bbJvhUbI+Re8YVevp3sBhwoPZg6MSjFWLXJ5FVRZpMhFLOVfpBYIyurussKWFwaXU8F"
"8/e6WLkF0Abi9KBycGJ0zfzth4/6s0ZKGQY4O/OfmnTzkUqHeQEOMwlm5LKKlWgpDQ+6rk/V+4HzXnZXQX00xmWuvoPNy1otbbW6xHirYCZtnixRYrxFJ9Od"
"NsRxKM+I6DPbSYUu+wrKLeTgrFOh6lXEyhxQaSGBbVgbsL+AIHs0t2URx8DTVy88VowhcV4ZpikewzNK4OJZ3RjFSz1I6QSGytFf4Miz4gZWI8KAmjtnNQ/r"
"225X4VxWqXroKSAkJ0uAdJsF4GxW3yVmE3phSaVDrdoIZzgYwFJu54Ln+jnwlO3HcO753O+KGn0OTN7DNL6khY1cdkM5gHP5GSx+OAewkGcUolmIlP7GSERW"
"Vu0cI6IppnAUdUx09Qk9gv4ftiTetBq+h7Cp3TTRLha9OS4R8QtJFwxtWieYR0sJ6/zM749bFjUTSeRMzxUz557ZfiF923XHjrDsrBwl5J3PKqPWinsU3Tut"
"DXo6gSvT1IfTOw7u7mfO0al141DrTNPSzsrjgtgHIlvoWVzMoOapMvLonCNwm6vzpGD13YazmT2EogmR1rUoN8kuw7sKtyztNg6a6hPLQ7JrjrgB4FZPWPlt"
"JAkXs71L8SnPSrEM4TNsqCr0JZfhvtsefjfI0P3UYFouusDt9fw1NP07FURb6AQmqJV48WqJSYkRcDe/EreY/Orsg9C5lzy5+x1Jek6/doIiTrMDXyUQRuJa"
"QPqMNdI67UnejVn6YOsccXvn68Thd99V+JfFJIr0hpSp5WflRZ2IEm8zbJzwhA1FcKMis5AFyCcyYU8lgvNB2GlkezwO389s9DfRizs9ppU+yrAhftCk1XEs"
"ZXSEPFqyyVZjOdKDjQ9tX64WmNUrDwMw9KJ3BDHbt8RygxKrrUMcOK74vQoSqZO1Azs5pSaCKRANTEZBKYf8ygIld4NexDbggywITEJvcBr0hwyslzvRsq1V"
"Lgx8UwAFvwU/LJnaOTi06sJG8coplT1CkZTipovYkCiROYEJN75FRTC6JpMFDFEFtZi4rzMPZi4RHZXoEWLBrscTk4EKl5ITvPgMWjLKoIRhQwyfRUNGIAwN"
"vdX7LBpjXQHC8rs7KEzd0JXBi5nrtNX3XhLKztU3gpTVWTIL5EngPXkxG6qNjH+qzFkc9NiOZIyywxXoMg7s7T3+k8QsbuCLH3uai70VRjr+am7I/RDMjvsB"
"uDkfIeGJX/aPQ+xpv5sFQdaHRViWym1Q4jE1sIp1KRjL2cyLb/NWPSRs5QNUKK/VQ4DKH8CVGa8eZFnxADa3oiM0eBxkjJw0vTJ1XrqB7T5HRb2Tpplsu7iN"
"1fWsTAU+Wn3Ua/Uc9BcDsoPUe5Vvr5OLneT6GQu+YbvnW7LWd9DifSIzg9E0BUdoj6M1XnXhzqpEMul9WRn1PEC8dUr+H7uC2nNqHndHjw6HB5dW5VLfJoW8"
"NyRviJplrGtu2VkqZerhPXTy0duIHpBAluZyDz2jm10JvoIRDbzPGUvXUjfqhnlQ7lVLaY7UXcoWN7rgBa6A8q5IG7zI6iT6wFq4S9u06xp5xQeWaWyJlOrb"
"U+hgV+zzpG7EGjZVeJd+0VNK2uZOYEThLrvchWwy85Q3/aAGyYNXsJc1+r4zBzPOvcuWvDr7Y5aLd1X3IyY5y/uzTguWI9i98IxKtZmhHacK3U4kXtpU6KrB"
"BDwGoYoY0GSWvD1boPv66pSdRKNK8Nyef+2fFKiOqG1bG7JLm3T5nZOUBZqi+jZK8BL2+RklGSDgPifhSEeJt+ImCfHmejW63X5xnwGWY6Ah7TGDvErPO8nS"
"NZy+8nLTZafwEQ1zBN4qSwfhjfJi3aZTNt7k4PyZJ4WxuwE3iTpZJ1eWtd2rcnHIL2YYrkur0Q1U33ceQvCuuE4mdot6pDapNKPw0oJysqlfKsZw4Fn4dXMy"
"ndqlrSfJiVM8FEq/BBa2k9MBDhfKs/Fu2tEal5GeapjpR5/j4LTn7Doac2z7y9xLDGxh4pCZgvhyw8WMNit0sZ/0qUqg2OL/KTiT+UB4XVE5veBFB418RSCQ"
"S86+kfdkGzrtBVDYm+DLAeaOoyS1L8FpzbYZ8PKfH9+/0889tPPg1x2IAtPq28CGKltiTT0VAw22jFQjwPSCf5BBz4JPOyGTxwD+ECMvQXUtGrogAr0SAR69"
"zPkCqm3tlBHA1d6xJQ8QcCezzY6cwJ2albgFtIPq4rvriTXOuC2hj9Pgk6YXGcfiaRKkOghq1ctFUPpBQL24PXN/woXjQLlw1neSyTLakXKhepPRzn8ry0V/"
"xnpImIk3TsCA+I4MkmucJnQcIV2YRJckiefYINHy0eAKmI4RevBq94iBwOFpA6vU91d66PKNHxndiqzt4wcTB/xyEyalbLLGcaRUMXk54EIzz4QOgSmPZb8C"
"LBjvI6kA4gbEl+FzL5LzC26Clafr2l/b9ry4gn8j9QCaSjonxyyprjzb9IixfRSEytHbJNI4qTebZKhTusXkPW/2RU13FvH4YAMtLU9nnnjtQT/qRStA/wLQ"
"9PULNS1674uMaT5tHdzAwoj22+F0y8dSq/suTTy2iMS+tVOeXtD5iHN1d3B5auquh0Po9eszc6DCHwuxW6iL2OiFvOvMd1erBoYAIyzJag8rU9PKnT9f7nkM"
"wok3OLEF0yd7kVUz5Hr18kzCnCrJI66Xm7T4PfL00HmwwpV13qgramY1zkXa4LWaBGOOTiIvaQDdfAMhrNPbPiavGyCuYLPeLvHtBnw44fvvv595x1Yqs3FY"
"clHasKdNEL3oXz3pmUzlo3Bicd/4znS4lLtQGDQlJ4ijDgXvaw+PXtYiyyM/R0dyOePb2Q4UqEdSenXRSTzR1hO1uLIzlLQp9rXqtEoJUENCNf1WaHON7RzH"
"I2wcBidsiw5kE/KsAh2VMQ1gVd7IWpMdrZ62A+VK5DMvkcm2oHAG0RiErlUvnLj1X1TpMPioAiYkLXH4TB+1EdZT4B4ELkskgV6QCJrzd/wvS80HEBlt9WR+"
"3Le2l/j6BchHcnCoqFg2xuSs4MbI4YhiH3URvToyY6+Z2LAX3idFp3v+HFQkAgDogtTKqo3oQ51BjabUk4iWpxnqp5LGoS3CN4tkM3aYzQluzxa5x+2kBG8/"
"2FSV2OrG0nxiMSE61zUk3fP4aCJ0zRSw7dKmM9PzeGQSO1Xs7NZoujy7tfYJrTzsCzDA4672WIMPHbrUvKBYQ6BOsLDAGGO344mxfKcoQTAaF2JAP6RJYtgZ"
"ixc5dA20DcjkPfLqwdY7/cKjbOt+iqp1z2S5u8vlsONvqtix6lu+uDcq3jSSEebiQC/qTPGMWsZKzcyrZW6MvmcuvK6inKzIxjb8QO1uAnxddRHcMTbuQ1ck"
"XlJ/POnDZShcLuVrJOMPnuLjPwAVzg58/TmTHmQgr5K7P3fcvbwfIfBePTeUNYvBccmdnRBj6L+jr0nPWcCU+enst5ZRubOO6Bj6a3X+7/u5e3jzMkb2zHhS"
"nk6NvjqGr4wt4jGa7+hAMOiyfMjvnYmdfVPKFFEEo55/c3F/M1091gf15qfNaw7UjQwcWcdlf5CEummGhyOayB337McIfDRNK0//SD4+QH3WHt83g8DEeKfk"
"NXQPNYege4dlnNpPKjQR8NCEondHZWOYb0xoaijdnnBhOWUPaT4JJum+/OUs2NYnL3wafcdWwDF8etKAnKr2iAwtF7jXZR2d12b7IX0Y8mwWZmZ53JxRJaDL"
"wzhrPJPJ7ikd+6nSpfTi4y6sqnK+bq/tOox5VjAlMeWCXnq1PjZle+l3TxgYO8OVKIk9EZg5Ic32KgO0jXHeWcQTzxml95FhQLK8FJGzzCkn3rcjlY9yR4M1"
"aLgTeij/ZpAo48EcScPp7dMfk44z0ponMcd7381JzRk5LLc5ifB/lWAIMUL+uypBj8WXD0IxNNxQk3N87Km0o+ergzFDTzQrYFzgn6jH0ePTdvj+dThEI7vR"
"fgoPq194MxFkMs+gaiqjZ5Du8DWSer56Ys/XTu75Wgk+XyvJ5wsSffBn34rEe3tO70X/PLCAM3mm36ek7lrpuKLvMVMzchQ9GW3a3O7y6CGP9etnupBXtnX/"
"sz5pAi5neKxl6rDIu1gNRUnv9prpKykc6SiMXm1HsES5UXtvTuVpoA69HsmB01VFSN7zVOuPbeyQszs78OnFOlF/UoTSpfQfF5mX+zxXX6OZz6j0H06mIzbf"
"ICzGkqt4y2qBKiu0Y+vIwyoln3EsP93/pRw1N1dtpR4OnEhWU5o+G6XkXpN6ZO7agzlsj8xle3xO26Nz2x6f4zaZ6zZmC8ZJjYuY9uUbvWbLa4k6jjYyqTyK"
"yC4R0K+IkZ1RHtGntNn4VBhjIp8xQ2Qb+5I+JDbI5e+gSWCXfzJmnWc1uTHgv+DjxqNCeTCU7zglbpTWnLRSa9jQID4/4m85nUxWYiufOVXBVtlzusVEEp7N"
"JkcBDOZD0tHClI99T8Ak6VZ6pJ/HyuP8Ry/qnwIbgwzQ8W+de2sC/+YDZQD8eHbyItAib0dooXHEHVlWbo/epe/mwcvNNa7QlN9go6posU0r9GdERunt8Wq7"
"THcgNlK6/xsYhQmkUIMKVLtpeCaCV0C0viXtfr2Wbz/DGkpLBwu2yqWZjQb4qFxR/EMAs2uskcWESbA3/UQdjduOXnj3aT+NhmfwjDc33NvZhNKRzYXJWCV7"
"NZKc2t9z+KnaGCGNv3bjlNdAG+O8uuS+XKA2IdM+kr+/aU630IS9pKb+XsCNes0nMRA6OgbWmF7sf/LkW/9INKIVzfUESQ3waIo5znU730irpf1vz48vzsO8"
"YUf7w2DDuKWUWSvLO3n/5/h0c0+SXN7RmdXx8839kT8+QnVBOEEZ9WJ559WKxfzZ9p6WcQ2gE52/0dHeby764FNtwTo/oARlQyK4zg8gsfDz2sub5V3eLOan"
"4j64xrEgJVneGV2Bqu39T3+Vlfj2xJ0ec1UTPmb9kcfshdyPstkm86fZVDKZ1MZi7AsCjemTvpDDk7ktuThwKE3mddtDZ8WVbkjxZFojjgavB1q/0b3eNlzq"
"47G4yrCi/4dhHkztMHZoWDztWk66k9MupMmNXE5kMfVyBR2XfyouJPK0bqW9sYEa2FaxIM5gKPR7F3IusMGRN6xkKFNmMvdCcUXdVNK6WTp/4QdVg9MjheG5"
"e+WcbhlyB48+f2qqT4PctsFrAiNj7X9UoHcA58eSU4DJaMFmhR/nyRMm4xGybvbJ4iGLP0KFL9/93IfFcHGPJ4SXtAJctQ3iKR0bgaaB7OnNgg/vEO9+qPru"
"nWAe7KZbwf7NipMHdyWEfByPnZKGFyNbF9/rHt7VV58cxw/sT+KHHbP4YScvnt4aTtRPKPrQO5zozNSwjWbSje9smQlQEd3+iMlsga83Yib74P9H7FEjdvAZ"
"fuM2fHP2/tV/BNJnvA9evf/l7O2bX98E/5Q2cHlnTeGUJ0V3w5Z3xmhIWOzJ8o4/SDNJQlszcLAGhu2eVkPwbaXlOnpxvJifbO+L8KE7iVNZk9qxePSfV3sw"
"9wHfb9i33lUslHVgSWUCMZjCFh8wci9QyIBzqLMaEgzJ5bDnDYejrZLUDchGr46eVeBLbHrYn8gL+uPGZjpS4Nx8M2k7vgfR7KKY+kgam0wkh5lAkyTVyqJI"
"8XXGExf64+vp1BWDhT0WHD5PeeHjZuRZaUao93ilj4o2wXWVZ+vbwbM+Bq6BBVk0mMUNWyORYPZ2JzC/fuR2LyHVgJDRH+VKML37sVj2yVt2+wSQzk3KWGwT"
"zTyduneL7iez0/dlIifXdGK6hHkgN92Tf2RSjqChYAvzsoWlIqBQT9vin7a9nY+lBVCLC54kBAx/02P4G5MWAMRh18mvpk7/4UmWUo1/sTyhxJ8kodS6hMxT"
"koTqpQ15Dv8/Hs99Bg=="
)

data = zlib.decompress(base64.b64decode(PAYLOAD))
sha = hashlib.sha256(data).hexdigest()

if sha != EXPECTED_SHA256:
    raise RuntimeError(
        f"Hardened trainer payload SHA mismatch:\n"
        f"expected={EXPECTED_SHA256}\n"
        f"actual={sha}"
    )

TARGET.parent.mkdir(parents=True, exist_ok=True)
TARGET.write_bytes(data)

subprocess.run(
    [sys.executable, "-m", "py_compile", str(TARGET)],
    check=True,
)

print("Complete trainer replacement: PASS")
print("Path:", TARGET)
print("Bytes:", TARGET.stat().st_size)
print("SHA256:", sha)
print("Syntax check: PASS")

Complete trainer replacement: PASS
Path: /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py
Bytes: 32264
SHA256: e24de3c724da25adb01a45b03c5ae3de617014a07953fc5fb23a40ba845105f4
Syntax check: PASS


In [16]:
# ============================================================
# HARDENED TRAINER SANITY SMOKE
# 4 train steps + 2 val steps, T4x2
# ============================================================

import json
import shutil
import subprocess
from pathlib import Path

BASE_CONFIG = (
    CODE_ROOT
    / "configs"
    / "phase5b_kaggle_v1.json"
)

SMOKE_CONFIG = Path(
    "/kaggle/working/phase5b_hardened_smoke_v1.json"
)

with BASE_CONFIG.open("r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["experiment_name"] = "dasonet_phase5b_hardened_smoke_v1"

with SMOKE_CONFIG.open("w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

smoke_out = Path(
    "/kaggle/working/dasonet_phase5b/"
    "dasonet_phase5b_hardened_smoke_v1"
)

if smoke_out.exists():
    shutil.rmtree(smoke_out)

# Confirm the DDP-safe zero-particle instance-loss fix is still present.
instance_file = (
    CODE_ROOT
    / "src/dasonet/losses/instance.py"
)

instance_text = instance_file.read_text(encoding="utf-8")

if "DDP-safe zero-particle handling" not in instance_text:
    raise RuntimeError(
        "The DDP-safe instance-loss replacement is no longer present. "
        "Do not continue."
    )

trainer = str(
    CODE_ROOT
    / "scripts/train_phase5b.py"
)

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    trainer,

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(SMOKE_CONFIG),

    "--output-root",
    "/kaggle/working",

    "--smoke",

    "--max-train-steps",
    "4",

    "--max-val-steps",
    "2",

    "--log-every",
    "1",

    "--resume",
    "none",
]

print("Running hardened sanity smoke:")
print(" ".join(cmd))
print()

subprocess.run(cmd, check=True)

print()
print("========================================")
print("HARDENED TRAINER DDP SANITY: PASS")

Running hardened sanity smoke:
torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5b_hardened_smoke_v1.json --output-root /kaggle/working --smoke --max-train-steps 4 --max-val-steps 2 --log-every 1 --resume none



W0913 08:46:54.492000 417 torch/distributed/run.py:852] 
W0913 08:46:54.492000 417 torch/distributed/run.py:852] *****************************************
W0913 08:46:54.492000 417 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 08:46:54.492000 417 torch/distributed/run.py:852] *****************************************
[W913 08:46:54.045476183 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 08:46:57.756336468 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 08:46:57.785225919 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5b_hardened_smoke_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     0
Anchor workers/rank:        0
Gradient accumulation:      2
Effective synthetic batch:  4
AMP fp16:                   True
Train steps/epoch/rank:     4
Optimizer steps total:      2
Smoke mode:                 True
epoch=00 step=00000/00004 loss=27.7631 inst=27.1619 pcc=0.0000 crpa=0.6012 lr=3.00e-04 vram_alloc=8.84GB vram_res=9.11GB
epoch=00 step=00001/00004 loss=141.3904 inst=140.7333 pcc=0.0000 crpa=0.6572 lr=3.00e-04 vram_alloc=8.84GB vram_res=9.11GB
epoch=00 step=00002/00004 loss=8.5386 inst=7.7397 pcc=0.0000 crpa=0.7989 

In [17]:
# ============================================================
# PHASE 5B — T4x2 WORKER THROUGHPUT BENCHMARK
#
# Baseline already measured:
#   synthetic workers/GPU = 2
#   train steps          = 100
#   val steps            = 20
#   wall time            = 4.05 min
#
# Candidate:
#   synthetic workers/GPU = 4
#   anchor workers/GPU     = 1
#   same model/batch/steps/seed
# ============================================================

import csv
import json
import shutil
import subprocess
import time
from pathlib import Path

BASELINE_WALL_MIN = 4.05

BASE_CONFIG = (
    CODE_ROOT
    / "configs"
    / "phase5b_kaggle_v1.json"
)

BENCH_CONFIG = Path(
    "/kaggle/working/"
    "phase5b_workers4_benchmark_v1.json"
)

with BASE_CONFIG.open("r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["experiment_name"] = "dasonet_phase5b_workers4_v1"

cfg["data"]["per_rank_synthetic_batch"] = 1
cfg["data"]["per_rank_anchor_batch"] = 2
cfg["data"]["synthetic_workers_per_rank"] = 4
cfg["data"]["anchor_workers_per_rank"] = 1
cfg["data"]["prefetch_factor"] = 2

cfg["optimization"]["gradient_accumulation_steps"] = 4
cfg["optimization"]["amp"] = True

with BENCH_CONFIG.open("w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

new_out = Path(
    "/kaggle/working/dasonet_phase5b/"
    "dasonet_phase5b_workers4_v1"
)

if new_out.exists():
    shutil.rmtree(new_out)

trainer = str(
    CODE_ROOT
    / "scripts/train_phase5b.py"
)

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    trainer,

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(BENCH_CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "1",

    "--max-train-steps",
    "100",

    "--max-val-steps",
    "20",

    "--accum",
    "4",

    "--num-workers",
    "4",

    "--log-every",
    "20",

    "--resume",
    "none",
]

print("=== T4x2 WORKERS=4 BENCHMARK ===")
print("Command:")
print(" ".join(cmd))
print()

start = time.time()

result = subprocess.run(
    cmd,
    check=True,
)

elapsed_sec = time.time() - start
elapsed_min = elapsed_sec / 60.0

print()
print("========================================")
print("WORKERS=4 BENCHMARK COMPLETE")
print("Return code:", result.returncode)
print(f"Wall time: {elapsed_min:.2f} min")

wall_speedup = BASELINE_WALL_MIN / elapsed_min

print()
print("Wall-time comparison")
print(f"workers=2 baseline: {BASELINE_WALL_MIN:.2f} min")
print(f"workers=4 current:  {elapsed_min:.2f} min")
print(f"speedup ratio:      {wall_speedup:.3f}x")

baseline_history = Path(
    "/kaggle/working/dasonet_phase5b/"
    "dasonet_phase5b_throughput_v1/"
    "history.csv"
)

new_history = new_out / "history.csv"

def read_last_history(path: Path):
    if not path.exists():
        return None

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:
        rows = list(csv.DictReader(f))

    return rows[-1] if rows else None


old_row = read_last_history(baseline_history)
new_row = read_last_history(new_history)

if old_row is not None and new_row is not None:
    old_epoch_sec = float(old_row["epoch_seconds"])
    new_epoch_sec = float(new_row["epoch_seconds"])

    print()
    print("Trainer epoch-time comparison")
    print(
        "workers=2 epoch_seconds:",
        f"{old_epoch_sec:.2f}",
    )
    print(
        "workers=4 epoch_seconds:",
        f"{new_epoch_sec:.2f}",
    )
    print(
        "epoch speedup ratio:",
        f"{old_epoch_sec / new_epoch_sec:.3f}x",
    )

status_path = new_out / "run_status.json"

if status_path.exists():
    print()
    print("run_status.json:")
    print(status_path.read_text(encoding="utf-8"))

total_bytes = sum(
    p.stat().st_size
    for p in new_out.rglob("*")
    if p.is_file()
)

print()
print(
    "Output footprint:",
    f"{total_bytes / 1024**2:.2f} MB",
)

print()
if wall_speedup > 1.05:
    print(
        "DECISION SIGNAL: workers=4 is materially faster "
        "than the workers=2 baseline."
    )
elif wall_speedup < 0.95:
    print(
        "DECISION SIGNAL: workers=4 is slower; "
        "workers=2 should remain the default."
    )
else:
    print(
        "DECISION SIGNAL: difference is small; "
        "prefer workers=2 for lower CPU/RAM pressure."
    )

=== T4x2 WORKERS=4 BENCHMARK ===
Command:
torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5b_workers4_benchmark_v1.json --output-root /kaggle/working --epochs 1 --max-train-steps 100 --max-val-steps 20 --accum 4 --num-workers 4 --log-every 20 --resume none



W0913 08:48:51.434000 463 torch/distributed/run.py:852] 
W0913 08:48:51.434000 463 torch/distributed/run.py:852] *****************************************
W0913 08:48:51.434000 463 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 08:48:51.434000 463 torch/distributed/run.py:852] *****************************************
[W913 08:48:51.919592162 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 08:48:54.855404408 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 08:48:54.861176165 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5b_workers4_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     4
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     100
Optimizer steps total:      25
Smoke mode:                 False
epoch=00 step=00000/00100 loss=27.7644 inst=27.1603 pcc=0.0000 crpa=0.6041 lr=1.25e-05 vram_alloc=8.84GB vram_res=9.11GB
epoch=00 step=00020/00100 loss=27.3389 inst=26.7295 pcc=0.0000 crpa=0.6094 lr=1.25e-05 vram_alloc=8.95GB vram_res=9.11GB
epoch=00 step=00040/00100 loss=26.9589 inst=26.3559 pcc=0.0001 crpa=0.6029 lr

In [20]:
%%writefile /kaggle/working/launch_phase5b_amp32.py
from __future__ import annotations

import runpy
import sys
from pathlib import Path

import torch


TRAINER = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining/"
    "scripts/train_phase5b.py"
)

if not TRAINER.exists():
    raise FileNotFoundError(TRAINER)

source = TRAINER.read_text(encoding="utf-8")

# Make sure we are wrapping the successfully hardened trainer,
# not the original deprecated-AMP version.
required_markers = [
    "torch.amp.GradScaler",
    "amp_skipped_optimizer_steps",
    "Synthetic workers/rank",
]

missing = [m for m in required_markers if m not in source]

if missing:
    raise RuntimeError(
        "Current trainer is not the hardened Phase 5B version.\n"
        f"Missing markers: {missing}"
    )


# Preserve the real PyTorch class.
_OriginalGradScaler = torch.amp.GradScaler


class T4StableGradScaler(_OriginalGradScaler):
    """
    Identical to torch.amp.GradScaler except that, when the trainer does
    not explicitly specify an initial scale, T4 Phase 5B starts at 32
    rather than PyTorch's much larger default.
    """

    def __init__(self, *args, **kwargs):
        kwargs.setdefault("init_scale", 32.0)
        super().__init__(*args, **kwargs)


torch.amp.GradScaler = T4StableGradScaler

print("====================================================")
print("DASO-Net Phase 5B AMP wrapper")
print("Trainer:", TRAINER)
print("Forced AMP initial scale: 32.0")
print("Trainer modification: NONE")
print("====================================================")

# Make the wrapped trainer see exactly the original command-line arguments.
sys.argv[0] = str(TRAINER)

runpy.run_path(
    str(TRAINER),
    run_name="__main__",
)

Writing /kaggle/working/launch_phase5b_amp32.py


In [21]:
# ============================================================
# PHASE 5B FINAL STABILITY CHECK
#
# T4x2
# 1024x1024
# batch/GPU = 1
# anchors/GPU = 2
# accumulation = 4
# effective batch = 8
# synthetic workers/GPU = 2
# AMP initial scale = 32
#
# 100 train steps + 20 validation steps
# ============================================================

import json
import shutil
import subprocess
import time
from pathlib import Path

import torch


CODE_ROOT = Path(
    "/kaggle/working/DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/abinavprabhuv/"
    "dasonet-project/DASO-Net-Phase1A"
)

BASE_CONFIG = (
    CODE_ROOT
    / "configs"
    / "phase5b_kaggle_v1.json"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5b_amp32_workers2_finalcheck_v1.json"
)

WRAPPER = Path(
    "/kaggle/working/launch_phase5b_amp32.py"
)

OUT = Path(
    "/kaggle/working/dasonet_phase5b/"
    "dasonet_phase5b_amp32_workers2_finalcheck_v1"
)

if not WRAPPER.exists():
    raise RuntimeError(
        "AMP32 wrapper is missing. Run the wrapper cell first."
    )

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"Project root is missing: {PROJECT_ROOT}"
    )

with BASE_CONFIG.open("r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg["experiment_name"] = (
    "dasonet_phase5b_amp32_workers2_finalcheck_v1"
)

cfg["data"]["per_rank_synthetic_batch"] = 1
cfg["data"]["per_rank_anchor_batch"] = 2
cfg["data"]["synthetic_workers_per_rank"] = 2
cfg["data"]["anchor_workers_per_rank"] = 1
cfg["data"]["prefetch_factor"] = 2

cfg["optimization"]["gradient_accumulation_steps"] = 4
cfg["optimization"]["amp"] = True

with CONFIG.open("w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

if OUT.exists():
    shutil.rmtree(OUT)

# DDP-safe negative-scene fix must still be present.
INSTANCE_FILE = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "losses"
    / "instance.py"
)

instance_source = INSTANCE_FILE.read_text(
    encoding="utf-8"
)

if "DDP-safe zero-particle handling" not in instance_source:
    raise RuntimeError(
        "DDP-safe zero-particle loss fix is missing. "
        "Do not train."
    )

cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(WRAPPER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "1",

    "--max-train-steps",
    "100",

    "--max-val-steps",
    "20",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "20",

    "--resume",
    "none",
]

print("====================================================")
print("FINAL PHASE 5B AMP32 STABILITY CHECK")
print("====================================================")
print("GPU count:", torch.cuda.device_count())
print("Command:")
print(" ".join(cmd))
print()

start = time.time()

result = subprocess.run(
    cmd,
    check=True,
)

elapsed = time.time() - start

print()
print("====================================================")
print("RUN COMPLETE")
print("Return code:", result.returncode)
print(f"Wall time: {elapsed / 60.0:.2f} min")

status_path = OUT / "run_status.json"

if not status_path.exists():
    raise RuntimeError(
        f"run_status.json was not created: {status_path}"
    )

status = json.loads(
    status_path.read_text(encoding="utf-8")
)

print()
print("run_status.json:")
print(json.dumps(status, indent=2))

amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)

# Verify the saved GradScaler state directly from the final checkpoint.
last_checkpoint = OUT / "last.pt"

if not last_checkpoint.exists():
    raise RuntimeError(
        f"Missing final checkpoint: {last_checkpoint}"
    )

checkpoint = torch.load(
    last_checkpoint,
    map_location="cpu",
    weights_only=False,
)

scaler_state = checkpoint.get(
    "scaler_state_dict",
    {},
)

saved_scale = scaler_state.get(
    "scale",
    None,
)

growth_tracker = scaler_state.get(
    "_growth_tracker",
    None,
)

print()
print("Saved AMP scaler state:")
print("  scale:", saved_scale)
print("  growth_tracker:", growth_tracker)
print("  skipped optimizer steps:", amp_skips)

print()
print("====================================================")
print("PHASE 5B FINAL CHECK")
print("====================================================")

if amp_skips == 0:
    print("DDP stability: PASS")
    print("AMP skipped optimizer steps: 0 — PASS")
    print("Workers/GPU: 2 — FROZEN")
    print("Native tile: 1024x1024 — FROZEN")
    print("Batch/GPU: 1 — FROZEN")
    print("Gradient accumulation: 4 — FROZEN")
    print("Effective synthetic batch: 8 — FROZEN")
    print("AMP initial scale: 32 — FROZEN")
    print()
    print(
        "Phase 5B training infrastructure: "
        "READY TO FREEZE"
    )
else:
    print(
        f"AMP skipped optimizer steps: {amp_skips}"
    )
    print(
        "Phase 5B requires another AMP-scale adjustment "
        "before Phase 5C."
    )

FINAL PHASE 5B AMP32 STABILITY CHECK
GPU count: 2
Command:
torchrun --standalone --nproc_per_node=2 /kaggle/working/launch_phase5b_amp32.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5b_amp32_workers2_finalcheck_v1.json --output-root /kaggle/working --epochs 1 --max-train-steps 100 --max-val-steps 20 --accum 4 --num-workers 2 --log-every 20 --resume none



W0913 09:08:57.348000 670 torch/distributed/run.py:852] 
W0913 09:08:57.348000 670 torch/distributed/run.py:852] *****************************************
W0913 09:08:57.348000 670 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 09:08:57.348000 670 torch/distributed/run.py:852] *****************************************
[W913 09:08:57.734239050 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5B AMP wrapper
Trainer: /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py
Forced AMP initial scale: 32.0
Trainer modification: NONE
DASO-Net Phase 5B AMP wrapper
Trainer: /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py
Forced AMP initial scale: 32.0
Trainer modification: NONE


[W913 09:09:00.407145073 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 09:09:00.437447908 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5b_amp32_workers2_finalcheck_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     100
Optimizer steps total:      25
Smoke mode:                 False
epoch=00 step=00000/00100 loss=27.7718 inst=27.1677 pcc=0.0000 crpa=0.6041 lr=1.25e-05 vram_alloc=8.84GB vram_res=9.10GB
epoch=00 step=00020/00100 loss=12.6640 inst=12.0591 pcc=0.0000 crpa=0.6048 lr=7.50e-05 vram_alloc=8.96GB vram_res=9.10GB
epoch=00 step=00040/00100 loss=7.2829 inst=6.6913 pcc=0.0001

In [22]:
# ============================================================
# PHASE 5B — FREEZE FINAL TRAINING INFRASTRUCTURE
# ============================================================

from pathlib import Path
import hashlib
import json
import platform
import shutil
import sys
import zipfile

import torch

CODE_ROOT = Path(
    "/kaggle/working/DASO-Net-Phase5B-KaggleTraining"
)

RUN_ROOT = Path(
    "/kaggle/working/dasonet_phase5b/"
    "dasonet_phase5b_amp32_workers2_finalcheck_v1"
)

WRAPPER = Path(
    "/kaggle/working/launch_phase5b_amp32.py"
)

FINAL_CONFIG = Path(
    "/kaggle/working/"
    "phase5b_amp32_workers2_finalcheck_v1.json"
)

FREEZE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-FROZEN-v1"
)

FREEZE_ZIP = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-FROZEN-v1.zip"
)

if FREEZE_ROOT.exists():
    shutil.rmtree(FREEZE_ROOT)

if FREEZE_ZIP.exists():
    FREEZE_ZIP.unlink()

FREEZE_ROOT.mkdir(parents=True)

required = {
    "src/dasonet/losses/instance.py":
        CODE_ROOT / "src/dasonet/losses/instance.py",

    "scripts/train_phase5b.py":
        CODE_ROOT / "scripts/train_phase5b.py",

    "launch_phase5b_amp32.py":
        WRAPPER,

    "configs/phase5b_amp32_workers2_finalcheck_v1.json":
        FINAL_CONFIG,

    "run/best.pt":
        RUN_ROOT / "best.pt",

    "run/last.pt":
        RUN_ROOT / "last.pt",

    "run/history.csv":
        RUN_ROOT / "history.csv",

    "run/resolved_config.json":
        RUN_ROOT / "resolved_config.json",

    "run/run_status.json":
        RUN_ROOT / "run_status.json",
}

missing = [
    str(src)
    for src in required.values()
    if not src.exists()
]

if missing:
    raise RuntimeError(
        "Cannot freeze Phase 5B. Missing:\n"
        + "\n".join(missing)
    )

# Verify the essential scientific fixes before freezing.
instance_text = (
    required["src/dasonet/losses/instance.py"]
    .read_text(encoding="utf-8")
)

trainer_text = (
    required["scripts/train_phase5b.py"]
    .read_text(encoding="utf-8")
)

if "DDP-safe zero-particle handling" not in instance_text:
    raise RuntimeError(
        "DDP-safe zero-particle loss fix is missing."
    )

if "amp_skipped_optimizer_steps" not in trainer_text:
    raise RuntimeError(
        "Hardened AMP trainer is missing."
    )

status = json.loads(
    (RUN_ROOT / "run_status.json").read_text(
        encoding="utf-8"
    )
)

if status.get("status") != "training_complete":
    raise RuntimeError(
        f"Unexpected run status: {status}"
    )

if int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
) != 0:
    raise RuntimeError(
        "Final Phase 5B run did not have zero AMP skips."
    )


def sha256(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


file_hashes = {}

for archive_rel, src in required.items():
    dst = FREEZE_ROOT / archive_rel
    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    shutil.copy2(src, dst)
    file_hashes[archive_rel] = sha256(dst)


freeze_manifest = {
    "freeze_name":
        "DASO-Net Phase5B Training Infrastructure v1",

    "phase":
        "5B",

    "status":
        "FROZEN_PASS",

    "hardware": {
        "gpu_count":
            torch.cuda.device_count(),

        "gpu_names": [
            torch.cuda.get_device_name(i)
            for i in range(
                torch.cuda.device_count()
            )
        ],

        "pytorch":
            torch.__version__,

        "cuda_runtime":
            torch.version.cuda,
    },

    "frozen_training_configuration": {
        "native_tile":
            1024,

        "per_rank_synthetic_batch":
            1,

        "per_rank_anchor_pair_batch":
            2,

        "gradient_accumulation":
            4,

        "effective_synthetic_batch":
            8,

        "synthetic_workers_per_rank":
            2,

        "anchor_workers_per_rank":
            1,

        "amp":
            "fp16",

        "amp_initial_scale":
            32.0,

        "final_amp_skipped_optimizer_steps":
            0,

        "peak_reserved_vram_gb_per_gpu":
            9.10,

        "rendered_cache":
            False,

        "persistent_checkpoints": [
            "best.pt",
            "last.pt",
        ],
    },

    "final_stability_check": {
        "train_steps":
            100,

        "validation_steps":
            20,

        "wall_time_minutes":
            4.06,

        "best_val_total_loss":
            status.get(
                "best_val_total_loss"
            ),
    },

    "scientific_scope": {
        "result_type":
            "engineering/infrastructure validation",

        "paper_performance_result":
            False,

        "note":
            (
                "Loss values and checkpoints in this freeze "
                "are infrastructure diagnostics only and are "
                "not manuscript performance claims."
            ),
    },

    "python":
        sys.version,

    "platform":
        platform.platform(),

    "files_sha256":
        file_hashes,
}

manifest_path = (
    FREEZE_ROOT
    / "PHASE5B_FREEZE_MANIFEST.json"
)

manifest_path.write_text(
    json.dumps(
        freeze_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

manifest_sha = sha256(manifest_path)

with zipfile.ZipFile(
    FREEZE_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as z:
    for p in sorted(
        FREEZE_ROOT.rglob("*")
    ):
        if p.is_file():
            z.write(
                p,
                arcname=str(
                    p.relative_to(
                        FREEZE_ROOT
                    )
                ),
            )

zip_sha = sha256(FREEZE_ZIP)

print("========================================")
print("PHASE 5B FREEZE: PASS")
print("========================================")
print("Freeze ZIP:")
print(FREEZE_ZIP)
print()
print(
    "ZIP size:",
    f"{FREEZE_ZIP.stat().st_size / 1024**2:.2f} MB",
)
print()
print("Manifest SHA256:")
print(manifest_sha)
print()
print("ZIP SHA256:")
print(zip_sha)
print()
print("Frozen configuration:")
print("  GPUs:                  2 x Tesla T4")
print("  native tile:           1024 x 1024")
print("  batch/GPU:             1")
print("  anchor pairs/GPU:      2")
print("  accumulation:          4")
print("  effective batch:       8")
print("  synthetic workers/GPU: 2")
print("  anchor workers/GPU:    1")
print("  AMP:                   FP16")
print("  initial scale:         32")
print("  AMP skips:             0")
print("  peak reserved VRAM:    ~9.10 GB/GPU")
print()
print(
    "Phase 5B is now formally frozen."
)

PHASE 5B FREEZE: PASS
Freeze ZIP:
/kaggle/working/DASO-Net-Phase5B-FROZEN-v1.zip

ZIP size: 24.99 MB

Manifest SHA256:
ecac8f980b5a26f30a5e5e46a70f64d30f17df8b09c452dc68c448ec467eb512

ZIP SHA256:
74338288f033a1dec3e0f1d66c271f6580be3c1f1a4caf5e1e986a11ffde7983

Frozen configuration:
  GPUs:                  2 x Tesla T4
  native tile:           1024 x 1024
  batch/GPU:             1
  anchor pairs/GPU:      2
  accumulation:          4
  effective batch:       8
  synthetic workers/GPU: 2
  anchor workers/GPU:    1
  AMP:                   FP16
  initial scale:         32
  AMP skips:             0
  peak reserved VRAM:    ~9.10 GB/GPU

Phase 5B is now formally frozen.


## Phase 5C

In [23]:
%%writefile /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/evaluate_phase5c.py
from __future__ import annotations

import argparse
import csv
import json
import random
import sys
import time
from collections import defaultdict
from pathlib import Path

PACKAGE_ROOT = Path(__file__).resolve().parents[1]
SRC_ROOT = PACKAGE_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

from dasonet.model import DASONet
from dasonet.model.decoding import decode_instances
from dasonet.training import (
    FruitMPSynTrainAdapter,
    collate_synthetic,
    seed_worker,
)


def parse_args():
    p = argparse.ArgumentParser(
        "DASO-Net Phase 5C synthetic instance evaluator"
    )

    p.add_argument("--project-root", required=True)
    p.add_argument("--checkpoint", required=True)
    p.add_argument("--output-dir", required=True)

    p.add_argument(
        "--split",
        choices=["val", "test"],
        default="val",
    )

    p.add_argument(
        "--max-scenes",
        type=int,
        default=0,
    )

    p.add_argument(
        "--seed",
        type=int,
        default=20260917,
    )

    p.add_argument(
        "--num-workers",
        type=int,
        default=2,
    )

    p.add_argument(
        "--decode-min-score",
        type=float,
        default=0.05,
    )

    p.add_argument(
        "--operating-score",
        type=float,
        default=0.25,
    )

    p.add_argument(
        "--topk",
        type=int,
        default=100,
    )

    p.add_argument(
        "--nms-iou",
        type=float,
        default=0.5,
    )

    p.add_argument(
        "--mask-threshold",
        type=float,
        default=0.5,
    )

    p.add_argument(
        "--match-iou",
        type=float,
        default=0.5,
    )

    p.add_argument(
        "--no-amp",
        action="store_true",
    )

    return p.parse_args()


def safe_div(a, b):
    return float(a / b) if b else 0.0


def dice_from_iou(iou):
    iou = float(iou)

    if iou <= 0.0:
        return 0.0

    return 2.0 * iou / (1.0 + iou)


@torch.no_grad()
def exact_mask_iou_matrix(
    pred_masks,
    gt_masks,
    gt_boxes,
):
    """
    Exact full-resolution mask IoU.

    Intersections are evaluated only inside the exact GT box because
    the GT mask is zero everywhere outside that box. Prediction area
    is still computed over the complete image, so the IoU remains exact.

    pred_masks: [P,H,W]
    gt_masks:   [G,H,W]
    gt_boxes:   [G,4], xyxy
    """

    p = int(pred_masks.shape[0])
    g = int(gt_masks.shape[0])

    device = pred_masks.device

    if p == 0 or g == 0:
        return torch.zeros(
            (p, g),
            dtype=torch.float32,
            device=device,
        )

    pred = pred_masks.bool()
    gt = gt_masks.to(device).bool()
    boxes = gt_boxes.to(device)

    pred_area = (
        pred.flatten(1)
        .sum(1)
        .float()
    )

    gt_area = (
        gt.flatten(1)
        .sum(1)
        .float()
    )

    ious = torch.zeros(
        (p, g),
        dtype=torch.float32,
        device=device,
    )

    h, w = gt.shape[-2:]

    for j in range(g):
        x1, y1, x2, y2 = boxes[j]

        x1 = max(
            0,
            min(
                w,
                int(torch.floor(x1).item()),
            ),
        )

        y1 = max(
            0,
            min(
                h,
                int(torch.floor(y1).item()),
            ),
        )

        x2 = max(
            0,
            min(
                w,
                int(torch.ceil(x2).item()),
            ),
        )

        y2 = max(
            0,
            min(
                h,
                int(torch.ceil(y2).item()),
            ),
        )

        if x2 <= x1 or y2 <= y1:
            continue

        inter = (
            pred[:, y1:y2, x1:x2]
            & gt[
                j,
                y1:y2,
                x1:x2,
            ].unsqueeze(0)
        )

        inter = (
            inter.flatten(1)
            .sum(1)
            .float()
        )

        union = (
            pred_area
            + gt_area[j]
            - inter
        )

        ious[:, j] = (
            inter
            / union.clamp(min=1.0)
        )

    return ious


def match_image(
    scores,
    ious,
    score_thr,
    iou_thr=0.5,
):
    """
    Score-ordered one-to-one matching.

    Each GT instance can be matched once.
    """

    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    ious = np.asarray(
        ious,
        dtype=np.float64,
    )

    p, g = ious.shape

    ids = np.flatnonzero(
        scores >= score_thr
    )

    if ids.size:
        ids = ids[
            np.argsort(
                -scores[ids],
                kind="stable",
            )
        ]

    used = np.zeros(
        g,
        dtype=bool,
    )

    matches = []
    fp = 0

    for pi in ids.tolist():

        if g == 0:
            fp += 1
            continue

        row = ious[pi].copy()

        row[used] = -1.0

        gi = int(
            np.argmax(row)
        )

        best = float(
            row[gi]
        )

        if best >= iou_thr:
            used[gi] = True

            matches.append(
                (
                    pi,
                    gi,
                    best,
                )
            )

        else:
            fp += 1

    return {
        "tp": len(matches),
        "fp": fp,
        "fn": int(
            g - used.sum()
        ),
        "pred_count": int(
            ids.size
        ),
        "gt_count": g,
        "matches": matches,
    }


def operating_summary(
    records,
    score_thr,
    iou_thr=0.5,
):
    tp = 0
    fp = 0
    fn = 0

    images = 0

    empty_images = 0
    empty_fp = 0

    abs_err = 0.0
    signed_err = 0.0

    matched = []

    for r in records:

        m = match_image(
            r["scores"],
            r["ious"],
            score_thr,
            iou_thr,
        )

        tp += m["tp"]
        fp += m["fp"]
        fn += m["fn"]

        images += 1

        abs_err += abs(
            m["pred_count"]
            - m["gt_count"]
        )

        signed_err += (
            m["pred_count"]
            - m["gt_count"]
        )

        if m["gt_count"] == 0:
            empty_images += 1
            empty_fp += m["pred_count"]

        matched.extend(
            x[2]
            for x in m["matches"]
        )

    precision = safe_div(
        tp,
        tp + fp,
    )

    recall = safe_div(
        tp,
        tp + fn,
    )

    f1 = safe_div(
        2.0
        * precision
        * recall,
        precision + recall,
    )

    return {
        "score_threshold":
            float(score_thr),

        "iou_threshold":
            float(iou_thr),

        "tp":
            tp,

        "fp":
            fp,

        "fn":
            fn,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "matched_mask_iou_mean":
            (
                float(
                    np.mean(matched)
                )
                if matched
                else 0.0
            ),

        "matched_mask_dice_mean":
            (
                float(
                    np.mean(
                        [
                            dice_from_iou(x)
                            for x
                            in matched
                        ]
                    )
                )
                if matched
                else 0.0
            ),

        "count_mae":
            safe_div(
                abs_err,
                images,
            ),

        "count_bias":
            safe_div(
                signed_err,
                images,
            ),

        "empty_scene_count":
            empty_images,

        "empty_scene_fp_per_image":
            safe_div(
                empty_fp,
                empty_images,
            ),

        "images":
            images,
    }


def average_precision(
    records,
    iou_thr,
):
    """
    101-point interpolated mask AP.
    """

    total_gt = sum(
        int(r["gt_count"])
        for r in records
    )

    if total_gt == 0:
        return float("nan")

    ranked = []

    for image_index, r in enumerate(
        records
    ):
        ranked.extend(
            (
                float(score),
                image_index,
                pred_index,
            )
            for pred_index, score
            in enumerate(
                r["scores"].tolist()
            )
        )

    if not ranked:
        return 0.0

    ranked.sort(
        key=lambda x: x[0],
        reverse=True,
    )

    used = [
        np.zeros(
            int(r["gt_count"]),
            dtype=bool,
        )
        for r in records
    ]

    tp = np.zeros(
        len(ranked),
        dtype=np.float64,
    )

    fp = np.zeros(
        len(ranked),
        dtype=np.float64,
    )

    for k, (
        _,
        image_index,
        pred_index,
    ) in enumerate(ranked):

        r = records[
            image_index
        ]

        if r["gt_count"] == 0:
            fp[k] = 1.0
            continue

        row = (
            r["ious"][
                pred_index
            ]
            .copy()
        )

        row[
            used[image_index]
        ] = -1.0

        gt_index = int(
            np.argmax(row)
        )

        best = float(
            row[gt_index]
        )

        if best >= iou_thr:
            used[
                image_index
            ][gt_index] = True

            tp[k] = 1.0

        else:
            fp[k] = 1.0

    tp = np.cumsum(tp)
    fp = np.cumsum(fp)

    recall = (
        tp
        / float(total_gt)
    )

    precision = (
        tp
        / np.maximum(
            tp + fp,
            1e-12,
        )
    )

    values = []

    for level in np.linspace(
        0.0,
        1.0,
        101,
    ):
        mask = (
            recall >= level
        )

        if np.any(mask):
            values.append(
                float(
                    precision[
                        mask
                    ].max()
                )
            )
        else:
            values.append(
                0.0
            )

    return float(
        np.mean(values)
    )


def ap_suite(records):
    thresholds = [
        round(
            0.50
            + 0.05 * i,
            2,
        )
        for i
        in range(10)
    ]

    aps = {
        t:
            average_precision(
                records,
                t,
            )
        for t
        in thresholds
    }

    finite = [
        v
        for v
        in aps.values()
        if np.isfinite(v)
    ]

    return {
        "mask_ap50":
            float(
                aps[0.50]
            ),

        "mask_ap75":
            float(
                aps[0.75]
            ),

        "mask_ap50_95":
            (
                float(
                    np.mean(finite)
                )
                if finite
                else None
            ),

        "per_iou": {
            f"{k:.2f}":
                float(v)
            for k, v
            in aps.items()
        },
    }


def threshold_sweep(
    records,
    iou_thr,
):
    rows = [
        operating_summary(
            records,
            float(t),
            iou_thr,
        )
        for t
        in np.arange(
            0.05,
            0.951,
            0.05,
        )
    ]

    best = max(
        rows,
        key=lambda x: (
            x["f1"],
            x["precision"],
            x["score_threshold"],
        ),
    )

    return {
        "best_threshold_by_f1":
            best[
                "score_threshold"
            ],

        "best_f1":
            best["f1"],

        "rows":
            rows,
    }


def build_model(
    checkpoint,
    device,
):
    cfg = dict(
        checkpoint
        .get(
            "config",
            {},
        )
        .get(
            "model",
            {},
        )
    )

    cfg.pop(
        "max_stride",
        None,
    )

    allowed = {
        "stem_channels",
        "s4_channels",
        "s8_channels",
        "s16_channels",
        "fpn_channels",
        "num_prototypes",
        "embedding_dim",
    }

    model = DASONet(
        **{
            k: v
            for k, v
            in cfg.items()
            if k in allowed
        }
    ).to(device)

    state = checkpoint.get(
        "core_state_dict"
    )

    if state is None:

        bundle = checkpoint.get(
            "bundle_state_dict"
        )

        if bundle is None:
            raise RuntimeError(
                "Checkpoint lacks "
                "core_state_dict/"
                "bundle_state_dict."
            )

        state = {
            k[5:]: v
            for k, v
            in bundle.items()
            if k.startswith(
                "core."
            )
        }

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    return model


def select_indices(
    dataset,
    max_scenes,
    seed,
):
    n = len(dataset)

    if (
        max_scenes <= 0
        or max_scenes >= n
    ):
        return list(
            range(n)
        )

    rng = np.random.default_rng(
        seed
    )

    ids = rng.choice(
        np.arange(n),
        size=max_scenes,
        replace=False,
    )

    ids = np.sort(
        ids
    ).astype(int).tolist()

    # Ensure that the engineering smoke includes
    # at least one true zero-particle scene.
    recipes = (
        dataset
        .base
        .recipes
    )

    has_empty = any(
        int(
            recipes
            .iloc[i][
                "requested_particle_count"
            ]
        ) == 0
        for i in ids
    )

    if not has_empty:

        empty = np.flatnonzero(
            recipes[
                "requested_particle_count"
            ].to_numpy()
            == 0
        )

        if empty.size:
            ids[0] = int(
                empty[0]
            )

            ids = sorted(
                set(ids)
            )

    return ids


def aggregate_rows(rows):
    tp = sum(
        r["tp"]
        for r in rows
    )

    fp = sum(
        r["fp"]
        for r in rows
    )

    fn = sum(
        r["fn"]
        for r in rows
    )

    precision = safe_div(
        tp,
        tp + fp,
    )

    recall = safe_div(
        tp,
        tp + fn,
    )

    f1 = safe_div(
        2.0
        * precision
        * recall,
        precision + recall,
    )

    return {
        "images":
            len(rows),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "count_mae":
            (
                float(
                    np.mean(
                        [
                            x[
                                "count_abs_error"
                            ]
                            for x
                            in rows
                        ]
                    )
                )
                if rows
                else 0.0
            ),
    }


def main():
    args = parse_args()

    random.seed(
        args.seed
    )

    np.random.seed(
        args.seed
    )

    torch.manual_seed(
        args.seed
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required."
        )

    device = torch.device(
        "cuda:0"
    )

    torch.cuda.set_device(
        device
    )

    project_root = Path(
        args.project_root
    ).resolve()

    checkpoint_path = Path(
        args.checkpoint
    ).resolve()

    output_dir = Path(
        args.output_dir
    ).resolve()

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = build_model(
        checkpoint,
        device,
    )

    dataset = FruitMPSynTrainAdapter(
        project_root,
        args.split,
        validate_frozen_hashes=True,
    )

    indices = select_indices(
        dataset,
        args.max_scenes,
        args.seed,
    )

    loader_kwargs = {
        "dataset":
            Subset(
                dataset,
                indices,
            ),

        "batch_size":
            1,

        "shuffle":
            False,

        "num_workers":
            max(
                0,
                args.num_workers,
            ),

        "pin_memory":
            True,

        "collate_fn":
            collate_synthetic,

        "worker_init_fn":
            seed_worker,
    }

    if args.num_workers > 0:
        loader_kwargs[
            "prefetch_factor"
        ] = 2

        loader_kwargs[
            "persistent_workers"
        ] = True

    loader = DataLoader(
        **loader_kwargs
    )

    records = []
    scene_rows = []

    size_stats = defaultdict(
        lambda: [
            0,
            0,
            0.0,
            0.0,
        ]
    )

    fruit_rows = defaultdict(
        list
    )

    start = time.time()

    torch.cuda.reset_peak_memory_stats(
        device
    )

    with torch.inference_mode():

        for step, batch in enumerate(
            loader
        ):

            image = (
                batch[
                    "composite"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
            )

            target = (
                batch[
                    "targets"
                ][0]
            )

            gt_masks = (
                target[
                    "masks"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
                .bool()
            )

            gt_boxes = (
                target[
                    "boxes"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
            )

            with torch.amp.autocast(
                "cuda",
                enabled=not args.no_amp,
                dtype=torch.float16,
            ):
                output = model(
                    image
                )

            pred = decode_instances(
                output,

                (
                    image.shape[-2],
                    image.shape[-1],
                ),

                stride=4,

                score_threshold=
                    args.decode_min_score,

                topk=
                    args.topk,

                nms_iou=
                    args.nms_iou,

                mask_threshold=
                    args.mask_threshold,
            )[0]

            scores = (
                pred[
                    "scores"
                ]
                .float()
                .cpu()
                .numpy()
            )

            ious = (
                exact_mask_iou_matrix(
                    pred[
                        "masks"
                    ],
                    gt_masks,
                    gt_boxes,
                )
                .float()
                .cpu()
                .numpy()
            )

            gt_count = int(
                gt_masks.shape[0]
            )

            record = {
                "scores":
                    scores,

                "ious":
                    ious,

                "gt_count":
                    gt_count,
            }

            records.append(
                record
            )

            match = match_image(
                scores,
                ious,
                args.operating_score,
                args.match_iou,
            )

            metadata = (
                batch[
                    "metadata"
                ][0]
            )

            particles = list(
                metadata.get(
                    "particles",
                    [],
                )
            )

            for gt_index in range(
                gt_count
            ):

                if gt_index < len(
                    particles
                ):
                    size_name = str(
                        particles[
                            gt_index
                        ].get(
                            "size_bin",
                            "unknown",
                        )
                    )
                else:
                    size_name = (
                        "unknown"
                    )

                size_stats[
                    size_name
                ][0] += 1

            for (
                _,
                gt_index,
                matched_iou,
            ) in match[
                "matches"
            ]:

                if gt_index < len(
                    particles
                ):
                    size_name = str(
                        particles[
                            gt_index
                        ].get(
                            "size_bin",
                            "unknown",
                        )
                    )
                else:
                    size_name = (
                        "unknown"
                    )

                size_stats[
                    size_name
                ][1] += 1

                size_stats[
                    size_name
                ][2] += matched_iou

                size_stats[
                    size_name
                ][3] += dice_from_iou(
                    matched_iou
                )

            matched_ious = [
                x[2]
                for x
                in match[
                    "matches"
                ]
            ]

            row = {
                "scene_id":
                    batch[
                        "scene_id"
                    ][0],

                "fruit":
                    batch[
                        "fruit"
                    ][0],

                "gt_count":
                    gt_count,

                "pred_count":
                    match[
                        "pred_count"
                    ],

                "tp":
                    match["tp"],

                "fp":
                    match["fp"],

                "fn":
                    match["fn"],

                "count_abs_error":
                    abs(
                        match[
                            "pred_count"
                        ]
                        - gt_count
                    ),

                "matched_iou_mean":
                    (
                        float(
                            np.mean(
                                matched_ious
                            )
                        )
                        if matched_ious
                        else 0.0
                    ),

                "matched_dice_mean":
                    (
                        float(
                            np.mean(
                                [
                                    dice_from_iou(
                                        x
                                    )
                                    for x
                                    in matched_ious
                                ]
                            )
                        )
                        if matched_ious
                        else 0.0
                    ),

                "max_score":
                    (
                        float(
                            scores.max()
                        )
                        if scores.size
                        else 0.0
                    ),

                "decoded_predictions":
                    int(
                        scores.size
                    ),
            }

            scene_rows.append(
                row
            )

            fruit_rows[
                row["fruit"]
            ].append(
                row
            )

            if (
                step % 10 == 0
                or step + 1
                == len(loader)
            ):
                print(
                    f"eval "
                    f"{step + 1:04d}/"
                    f"{len(loader):04d} "
                    f"scene="
                    f"{row['scene_id']} "
                    f"gt={gt_count} "
                    f"decoded="
                    f"{scores.size} "
                    f"op_pred="
                    f"{match['pred_count']} "
                    f"elapsed="
                    f"{(time.time() - start) / 60:.1f}m"
                )

    elapsed = (
        time.time()
        - start
    )

    operating = operating_summary(
        records,
        args.operating_score,
        args.match_iou,
    )

    aps = ap_suite(
        records
    )

    sweep = threshold_sweep(
        records,
        args.match_iou,
    )

    size_summary = {}

    for size_name, values in sorted(
        size_stats.items()
    ):
        (
            gt,
            matched,
            iou_sum,
            dice_sum,
        ) = values

        size_summary[
            size_name
        ] = {
            "gt_particles":
                gt,

            "matched_particles":
                matched,

            "recall_at_operating_point":
                safe_div(
                    matched,
                    gt,
                ),

            "matched_mask_iou_mean":
                safe_div(
                    iou_sum,
                    matched,
                ),

            "matched_mask_dice_mean":
                safe_div(
                    dice_sum,
                    matched,
                ),
        }

    recipes = (
        dataset
        .base
        .recipes
        .iloc[
            indices
        ]
    )

    summary = {
        "phase":
            "5C",

        "paper_result":
            False,

        "note":
            (
                "Metric-pipeline engineering smoke; "
                "do not cite as manuscript performance."
            ),

        "checkpoint":
            str(
                checkpoint_path
            ),

        "checkpoint_epoch":
            int(
                checkpoint.get(
                    "epoch",
                    -1,
                )
            ),

        "checkpoint_optimizer_step":
            int(
                checkpoint.get(
                    "optimizer_step",
                    -1,
                )
            ),

        "split":
            args.split,

        "scenes_evaluated":
            len(records),

        "sample_particle_count_distribution": {
            str(k):
                int(v)
            for k, v
            in (
                recipes[
                    "requested_particle_count"
                ]
                .value_counts()
                .sort_index()
                .items()
            )
        },

        "sample_fruit_distribution": {
            str(k):
                int(v)
            for k, v
            in (
                recipes[
                    "fruit"
                ]
                .value_counts()
                .sort_index()
                .items()
            )
        },

        "decode": {
            "decode_min_score":
                args.decode_min_score,

            "operating_score":
                args.operating_score,

            "topk":
                args.topk,

            "nms_iou":
                args.nms_iou,

            "mask_threshold":
                args.mask_threshold,

            "match_iou":
                args.match_iou,
        },

        "operating_point":
            operating,

        "mask_ap":
            aps,

        "validation_threshold_sweep":
            sweep,

        "apparent_size_bins":
            size_summary,

        "by_fruit": {
            fruit:
                aggregate_rows(
                    rows
                )
            for fruit, rows
            in sorted(
                fruit_rows.items()
            )
        },

        "runtime": {
            "minutes":
                elapsed
                / 60.0,

            "scenes_per_second":
                safe_div(
                    len(records),
                    elapsed,
                ),

            "peak_allocated_gb":
                (
                    torch.cuda
                    .max_memory_allocated(
                        device
                    )
                    / (1024 ** 3)
                ),

            "peak_reserved_gb":
                (
                    torch.cuda
                    .max_memory_reserved(
                        device
                    )
                    / (1024 ** 3)
                ),
        },
    }

    summary_path = (
        output_dir
        / "phase5c_metric_summary.json"
    )

    summary_path.write_text(
        json.dumps(
            summary,
            indent=2,
        ),
        encoding="utf-8",
    )

    scene_csv = (
        output_dir
        / "phase5c_per_scene.csv"
    )

    with scene_csv.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=list(
                scene_rows[0].keys()
            ),
        )

        writer.writeheader()
        writer.writerows(
            scene_rows
        )

    print()
    print(
        "=== PHASE 5C METRIC "
        "EVALUATION COMPLETE ==="
    )

    print(
        f"Scenes:                 "
        f"{len(records)}"
    )

    print(
        f"Mask AP50:              "
        f"{aps['mask_ap50']:.4f}"
    )

    print(
        f"Mask AP75:              "
        f"{aps['mask_ap75']:.4f}"
    )

    print(
        f"Mask AP50:95:           "
        f"{aps['mask_ap50_95']:.4f}"
    )

    print(
        f"Operating precision:    "
        f"{operating['precision']:.4f}"
    )

    print(
        f"Operating recall:       "
        f"{operating['recall']:.4f}"
    )

    print(
        f"Operating F1:           "
        f"{operating['f1']:.4f}"
    )

    print(
        f"Matched mask Dice:      "
        f"{operating['matched_mask_dice_mean']:.4f}"
    )

    print(
        f"Count MAE:              "
        f"{operating['count_mae']:.4f}"
    )

    print(
        f"Empty-scene FP/image:   "
        f"{operating['empty_scene_fp_per_image']:.4f}"
    )

    print(
        f"Best val F1 threshold:  "
        f"{sweep['best_threshold_by_f1']:.2f}"
    )

    print(
        f"Best val F1:            "
        f"{sweep['best_f1']:.4f}"
    )

    print(
        f"Runtime:                "
        f"{elapsed / 60.0:.2f} min"
    )

    print(
        f"Peak reserved VRAM:     "
        f"{summary['runtime']['peak_reserved_gb']:.2f} GB"
    )

    print(
        f"Summary:                "
        f"{summary_path}"
    )

    print(
        f"Per-scene CSV:          "
        f"{scene_csv}"
    )

    print(
        "Phase 5C metric pipeline: PASS"
    )


if __name__ == "__main__":
    main()

Writing /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/evaluate_phase5c.py


In [24]:
# ============================================================
# PHASE 5C — SYNTHETIC INSTANCE-METRIC SMOKE
#
# Uses the Phase-5B 100-step checkpoint ONLY to validate
# the evaluation machinery.
#
# These values are NOT manuscript performance results.
# ============================================================

import json
import shutil
import subprocess
import sys
from pathlib import Path

CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CHECKPOINT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5b_amp32_workers2_finalcheck_v1/"
    "best.pt"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "metric_smoke_v1"
)

if not CHECKPOINT.exists():
    raise RuntimeError(
        f"Missing Phase 5B checkpoint:\n{CHECKPOINT}"
    )

if not EVALUATOR.exists():
    raise RuntimeError(
        f"Missing Phase 5C evaluator:\n{EVALUATOR}"
    )

if OUTPUT_DIR.exists():
    shutil.rmtree(
        OUTPUT_DIR
    )

cmd = [
    sys.executable,
    str(EVALUATOR),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(CHECKPOINT),

    "--output-dir",
    str(OUTPUT_DIR),

    "--split",
    "val",

    "--max-scenes",
    "40",

    "--seed",
    "20260917",

    "--num-workers",
    "2",

    "--decode-min-score",
    "0.05",

    "--operating-score",
    "0.25",

    "--topk",
    "100",

    "--nms-iou",
    "0.5",

    "--mask-threshold",
    "0.5",

    "--match-iou",
    "0.5",
]

print(
    "=== PHASE 5C METRIC SMOKE ==="
)

print(
    " ".join(cmd)
)

print()

subprocess.run(
    cmd,
    check=True,
)

summary_path = (
    OUTPUT_DIR
    / "phase5c_metric_summary.json"
)

if not summary_path.exists():
    raise RuntimeError(
        "Phase 5C summary was not created."
    )

summary = json.loads(
    summary_path.read_text(
        encoding="utf-8"
    )
)

print()
print(
    "========================================"
)

print(
    "PHASE 5C METRIC SMOKE: PASS"
)

print(
    "========================================"
)

print(
    "Scenes:",
    summary[
        "scenes_evaluated"
    ],
)

print(
    "Particle-count distribution:",
    summary[
        "sample_particle_count_distribution"
    ],
)

print(
    "Fruit distribution:",
    summary[
        "sample_fruit_distribution"
    ],
)

print()
print(
    "IMPORTANT:"
)

print(
    "These metrics validate the evaluation pipeline only."
)

print(
    "They are NOT final DASO-Net performance results."
)

=== PHASE 5C METRIC SMOKE ===
/usr/bin/python3 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/evaluate_phase5c.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --checkpoint /kaggle/working/dasonet_phase5b/dasonet_phase5b_amp32_workers2_finalcheck_v1/best.pt --output-dir /kaggle/working/dasonet_phase5c/metric_smoke_v1 --split val --max-scenes 40 --seed 20260917 --num-workers 2 --decode-min-score 0.05 --operating-score 0.25 --topk 100 --nms-iou 0.5 --mask-threshold 0.5 --match-iou 0.5

eval 0001/0040 scene=27f8155ca8ca48c7d0b7 gt=3 decoded=27 op_pred=0 elapsed=0.1m
eval 0011/0040 scene=12013b728415e3061c07 gt=5 decoded=10 op_pred=0 elapsed=0.2m
eval 0021/0040 scene=eb57ddcce4de72461739 gt=2 decoded=3 op_pred=0 elapsed=0.3m
eval 0031/0040 scene=2e669e0527f9c6a85050 gt=6 decoded=30 op_pred=0 elapsed=0.4m
eval 0040/0040 scene=d71a86e5db8e7f866c1d gt=1 decoded=0 op_pred=0 elapsed=0.5m

=== PHASE 5C METRIC EVALUATION COMPLETE ===
Scenes:        

In [25]:
# ============================================================
# PHASE 5C — METRIC MATHEMATICS POSITIVE CONTROL
#
# This MUST produce perfect scores for perfect synthetic
# predictions before we trust the evaluator on learned models.
# ============================================================

import importlib.util
from pathlib import Path

import numpy as np
import torch


EVALUATOR = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining/"
    "scripts/evaluate_phase5c.py"
)

spec = importlib.util.spec_from_file_location(
    "phase5c_eval",
    EVALUATOR,
)

ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)


# ------------------------------------------------------------
# 1. Exact full-resolution mask IoU test
# ------------------------------------------------------------

gt = torch.zeros(
    (1, 16, 16),
    dtype=torch.bool,
)

gt[0, 4:10, 4:10] = True

pred = torch.zeros(
    (1, 16, 16),
    dtype=torch.bool,
)

pred[0, 4:10, 4:10] = True

boxes = torch.tensor(
    [[4.0, 4.0, 10.0, 10.0]],
    dtype=torch.float32,
)

iou = ev.exact_mask_iou_matrix(
    pred,
    gt,
    boxes,
)

print(
    "Perfect-mask IoU:",
    float(iou[0, 0]),
)

assert torch.allclose(
    iou,
    torch.ones_like(iou),
    atol=1e-7,
)


# ------------------------------------------------------------
# 2. Perfect instance-detection / mask-AP test
# ------------------------------------------------------------

perfect_records = [
    {
        "scores":
            np.array(
                [0.95],
                dtype=np.float64,
            ),

        "ious":
            np.array(
                [[1.0]],
                dtype=np.float64,
            ),

        "gt_count":
            1,
    },

    # Valid clean/empty scene.
    {
        "scores":
            np.empty(
                (0,),
                dtype=np.float64,
            ),

        "ious":
            np.empty(
                (0, 0),
                dtype=np.float64,
            ),

        "gt_count":
            0,
    },
]

op = ev.operating_summary(
    perfect_records,
    score_thr=0.5,
    iou_thr=0.5,
)

aps = ev.ap_suite(
    perfect_records
)

print()
print("Perfect-control operating metrics:")
print(
    "  precision:",
    op["precision"],
)
print(
    "  recall:",
    op["recall"],
)
print(
    "  F1:",
    op["f1"],
)
print(
    "  Dice:",
    op["matched_mask_dice_mean"],
)
print(
    "  count MAE:",
    op["count_mae"],
)
print(
    "  empty FP/image:",
    op["empty_scene_fp_per_image"],
)

print()
print("Perfect-control AP:")
print(
    "  AP50:",
    aps["mask_ap50"],
)
print(
    "  AP75:",
    aps["mask_ap75"],
)
print(
    "  AP50:95:",
    aps["mask_ap50_95"],
)

assert abs(
    op["precision"] - 1.0
) < 1e-7

assert abs(
    op["recall"] - 1.0
) < 1e-7

assert abs(
    op["f1"] - 1.0
) < 1e-7

assert abs(
    op["matched_mask_dice_mean"] - 1.0
) < 1e-7

assert abs(
    op["count_mae"]
) < 1e-7

assert abs(
    aps["mask_ap50"] - 1.0
) < 1e-7

assert abs(
    aps["mask_ap75"] - 1.0
) < 1e-7

assert abs(
    aps["mask_ap50_95"] - 1.0
) < 1e-7


# ------------------------------------------------------------
# 3. Empty-scene false-positive accounting test
# ------------------------------------------------------------

empty_fp_record = [
    {
        "scores":
            np.array(
                [0.90],
                dtype=np.float64,
            ),

        "ious":
            np.empty(
                (1, 0),
                dtype=np.float64,
            ),

        "gt_count":
            0,
    }
]

empty_op = ev.operating_summary(
    empty_fp_record,
    score_thr=0.5,
    iou_thr=0.5,
)

print()
print(
    "Empty-scene injected FP/image:",
    empty_op[
        "empty_scene_fp_per_image"
    ],
)

assert abs(
    empty_op[
        "empty_scene_fp_per_image"
    ] - 1.0
) < 1e-7

assert (
    empty_op["fp"] == 1
)

print()
print(
    "========================================"
)
print(
    "PHASE 5C METRIC POSITIVE CONTROL: PASS"
)
print(
    "========================================"
)

Perfect-mask IoU: 1.0

Perfect-control operating metrics:
  precision: 1.0
  recall: 1.0
  F1: 1.0
  Dice: 1.0
  count MAE: 0.0
  empty FP/image: 0.0

Perfect-control AP:
  AP50: 1.0
  AP75: 1.0
  AP50:95: 1.0

Empty-scene injected FP/image: 1.0

PHASE 5C METRIC POSITIVE CONTROL: PASS


In [26]:
# ============================================================
# PHASE 5C — BOUNDED LEARNING PILOT v1
#
# 500 train steps/rank
# 100 val-loss steps/rank
# 125 intended optimizer updates
#
# This is for learning-behavior diagnosis, NOT manuscript
# performance reporting.
# ============================================================

import json
import shutil
import subprocess
import time
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

WRAPPER = Path(
    "/kaggle/working/"
    "launch_phase5b_amp32.py"
)

BASE_CONFIG = (
    CODE_ROOT
    / "configs"
    / "phase5b_kaggle_v1.json"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_pilot500_v1.json"
)

RUN_NAME = (
    "dasonet_phase5c_pilot500_v1"
)

OUT = (
    Path(
        "/kaggle/working/"
        "dasonet_phase5b"
    )
    / RUN_NAME
)


if not WRAPPER.exists():
    raise RuntimeError(
        "AMP32 wrapper is missing."
    )

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"Project root missing:\n{PROJECT_ROOT}"
    )

instance_file = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "losses"
    / "instance.py"
)

if (
    "DDP-safe zero-particle handling"
    not in instance_file.read_text(
        encoding="utf-8"
    )
):
    raise RuntimeError(
        "DDP-safe instance loss is missing."
    )


with BASE_CONFIG.open(
    "r",
    encoding="utf-8",
) as f:
    cfg = json.load(f)


cfg["experiment_name"] = RUN_NAME

cfg["data"][
    "per_rank_synthetic_batch"
] = 1

cfg["data"][
    "per_rank_anchor_batch"
] = 2

cfg["data"][
    "synthetic_workers_per_rank"
] = 2

cfg["data"][
    "anchor_workers_per_rank"
] = 1

cfg["data"][
    "prefetch_factor"
] = 2

cfg["optimization"][
    "gradient_accumulation_steps"
] = 4

cfg["optimization"][
    "amp"
] = True


with CONFIG.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        cfg,
        f,
        indent=2,
    )


if OUT.exists():
    shutil.rmtree(
        OUT
    )


cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(WRAPPER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "1",

    "--max-train-steps",
    "500",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "50",

    "--resume",
    "none",
]


print(
    "========================================"
)
print(
    "PHASE 5C — 500-STEP LEARNING PILOT"
)
print(
    "========================================"
)
print(
    " ".join(cmd)
)
print()

start = time.time()

subprocess.run(
    cmd,
    check=True,
)

elapsed = (
    time.time()
    - start
)

status_file = (
    OUT
    / "run_status.json"
)

if not status_file.exists():
    raise RuntimeError(
        "Pilot run_status.json missing."
    )

status = json.loads(
    status_file.read_text(
        encoding="utf-8"
    )
)

print()
print(
    "========================================"
)
print(
    "PHASE 5C PILOT TRAINING COMPLETE"
)
print(
    "========================================"
)

print(
    f"Wall time: "
    f"{elapsed / 60.0:.2f} min"
)

print(
    "AMP skipped optimizer steps:",
    status.get(
        "amp_skipped_optimizer_steps"
    ),
)

print(
    "Best val total loss:",
    status.get(
        "best_val_total_loss"
    ),
)

print(
    "Best checkpoint:",
    status.get(
        "best_checkpoint"
    ),
)

print(
    "Last checkpoint:",
    status.get(
        "last_checkpoint"
    ),
)

if (
    int(
        status.get(
            "amp_skipped_optimizer_steps",
            -1,
        )
    )
    != 0
):
    raise RuntimeError(
        "Unexpected AMP optimizer skips "
        "during Phase 5C pilot."
    )

print()
print(
    "Phase 5C 500-step training pilot: PASS"
)

PHASE 5C — 500-STEP LEARNING PILOT
torchrun --standalone --nproc_per_node=2 /kaggle/working/launch_phase5b_amp32.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_pilot500_v1.json --output-root /kaggle/working --epochs 1 --max-train-steps 500 --max-val-steps 100 --accum 4 --num-workers 2 --log-every 50 --resume none



W0913 09:25:18.243000 863 torch/distributed/run.py:852] 
W0913 09:25:18.243000 863 torch/distributed/run.py:852] *****************************************
W0913 09:25:18.243000 863 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 09:25:18.243000 863 torch/distributed/run.py:852] *****************************************
[W913 09:25:18.788722271 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5B AMP wrapper
Trainer: /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py
Forced AMP initial scale: 32.0
Trainer modification: NONE
DASO-Net Phase 5B AMP wrapper
Trainer: /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py
Forced AMP initial scale: 32.0
Trainer modification: NONE


[W913 09:25:21.565874327 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 09:25:21.601493047 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_pilot500_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     500
Optimizer steps total:      125
Smoke mode:                 False
epoch=00 step=00000/00500 loss=27.7631 inst=27.1589 pcc=0.0000 crpa=0.6041 lr=2.42e-06 vram_alloc=8.84GB vram_res=9.11GB
epoch=00 step=00050/00500 loss=7.9469 inst=7.3414 pcc=0.0000 crpa=0.6055 lr=3.15e-05 vram_alloc=8.97GB vram_res=9.11GB
epoch=00 step=00100/00500 loss=7.4498 inst=6.8471 pcc=0.0001 crpa=0.6026 lr=6.

In [27]:
# ============================================================
# PHASE 5C — 100-step vs 500-step checkpoint comparison
#
# SAME:
#   validation scenes
#   random seed
#   decoder
#   IoU matching
#   score sweep
#
# Still engineering/pilot evidence, NOT final paper results.
# ============================================================

import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

BASELINE_CKPT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5b_amp32_workers2_finalcheck_v1/"
    "best.pt"
)

PILOT_CKPT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_pilot500_v1/"
    "best.pt"
)

COMPARE_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "pilot500_comparison_v1"
)


if COMPARE_ROOT.exists():
    shutil.rmtree(
        COMPARE_ROOT
    )

COMPARE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


runs = {
    "baseline_100":
        BASELINE_CKPT,

    "pilot_500":
        PILOT_CKPT,
}


summaries = {}


for name, checkpoint in runs.items():

    if not checkpoint.exists():
        raise RuntimeError(
            f"Missing checkpoint:\n{checkpoint}"
        )

    out = (
        COMPARE_ROOT
        / name
    )

    cmd = [
        sys.executable,
        str(EVALUATOR),

        "--project-root",
        str(PROJECT_ROOT),

        "--checkpoint",
        str(checkpoint),

        "--output-dir",
        str(out),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        # Keep AP truncation extremely low.
        "--decode-min-score",
        "0.001",

        # Engineering operating point only.
        "--operating-score",
        "0.05",

        "--topk",
        "100",

        "--nms-iou",
        "0.5",

        "--mask-threshold",
        "0.5",

        "--match-iou",
        "0.5",
    ]

    print()
    print(
        "========================================"
    )
    print(
        f"EVALUATING: {name}"
    )
    print(
        "========================================"
    )

    subprocess.run(
        cmd,
        check=True,
    )

    summary_file = (
        out
        / "phase5c_metric_summary.json"
    )

    summaries[name] = json.loads(
        summary_file.read_text(
            encoding="utf-8"
        )
    )


def metric_row(summary):
    op = summary[
        "operating_point"
    ]

    ap = summary[
        "mask_ap"
    ]

    sweep = summary[
        "validation_threshold_sweep"
    ]

    return {
        "AP50":
            ap[
                "mask_ap50"
            ],

        "AP75":
            ap[
                "mask_ap75"
            ],

        "AP50:95":
            ap[
                "mask_ap50_95"
            ],

        "Precision@0.05":
            op[
                "precision"
            ],

        "Recall@0.05":
            op[
                "recall"
            ],

        "F1@0.05":
            op[
                "f1"
            ],

        "MatchedDice":
            op[
                "matched_mask_dice_mean"
            ],

        "CountMAE":
            op[
                "count_mae"
            ],

        "EmptyFP/image":
            op[
                "empty_scene_fp_per_image"
            ],

        "BestThreshold":
            sweep[
                "best_threshold_by_f1"
            ],

        "BestF1":
            sweep[
                "best_f1"
            ],
    }


base = metric_row(
    summaries[
        "baseline_100"
    ]
)

pilot = metric_row(
    summaries[
        "pilot_500"
    ]
)


print()
print(
    "=============================================================="
)
print(
    "PHASE 5C — SAME-200-SCENE LEARNING COMPARISON"
)
print(
    "=============================================================="
)

print(
    f"{'Metric':<20}"
    f"{'100-step':>14}"
    f"{'500-step':>14}"
    f"{'Delta':>14}"
)

print(
    "-" * 62
)

for key in base:

    b = float(
        base[key]
    )

    p = float(
        pilot[key]
    )

    delta = (
        p - b
    )

    print(
        f"{key:<20}"
        f"{b:>14.4f}"
        f"{p:>14.4f}"
        f"{delta:>+14.4f}"
    )


print()
print(
    "200-scene particle-count distribution:"
)

print(
    summaries[
        "pilot_500"
    ][
        "sample_particle_count_distribution"
    ]
)

print()
print(
    "200-scene fruit distribution:"
)

print(
    summaries[
        "pilot_500"
    ][
        "sample_fruit_distribution"
    ]
)

print()
print(
    "IMPORTANT:"
)

print(
    "This comparison determines whether "
    "DASO-Net is learning instance localization/masks."
)

print(
    "It is not a final manuscript result."
)


EVALUATING: baseline_100
eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=100 op_pred=74 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=100 op_pred=0 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=100 op_pred=5 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=100 op_pred=1 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=100 op_pred=3 elapsed=0.5m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=100 op_pred=1 elapsed=0.6m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=100 op_pred=99 elapsed=0.7m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=100 op_pred=100 elapsed=0.8m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=100 op_pred=100 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=100 op_pred=0 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0d7548bc gt=0 decoded=100 op_pred=0 elapsed=1.1m
eval 0111/0200 scene=858baec99120971a3e47 gt=3 decoded=100 op_pred

In [28]:
%%writefile /kaggle/working/DASO-Net-Phase5B-KaggleTraining/src/dasonet/model/decoding.py
from __future__ import annotations

from typing import List

import torch
import torch.nn.functional as F


def box_iou(
    boxes1: torch.Tensor,
    boxes2: torch.Tensor,
) -> torch.Tensor:
    area1 = (
        (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0)
        * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    )
    area2 = (
        (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0)
        * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    )

    lt = torch.maximum(
        boxes1[:, None, :2],
        boxes2[None, :, :2],
    )
    rb = torch.minimum(
        boxes1[:, None, 2:],
        boxes2[None, :, 2:],
    )
    wh = (rb - lt).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    union = area1[:, None] + area2[None, :] - inter

    return inter / union.clamp(min=1e-6)


def nms(
    boxes: torch.Tensor,
    scores: torch.Tensor,
    iou_threshold: float,
) -> torch.Tensor:
    if boxes.numel() == 0:
        return torch.empty(
            (0,),
            dtype=torch.long,
            device=boxes.device,
        )

    order = scores.argsort(descending=True)
    keep = []

    while order.numel() > 0:
        i = int(order[0])
        keep.append(i)

        if order.numel() == 1:
            break

        rest = order[1:]

        ious = box_iou(
            boxes[i].unsqueeze(0),
            boxes[rest],
        ).squeeze(0)

        order = rest[
            ious <= iou_threshold
        ]

    return torch.tensor(
        keep,
        dtype=torch.long,
        device=boxes.device,
    )


def local_peak_suppression(
    heatmap: torch.Tensor,
    kernel: int = 3,
) -> torch.Tensor:
    """
    Keep only local maxima in the center heatmap.

    The center target is Gaussian. Without this step, several neighboring
    pixels belonging to one Gaussian peak can be decoded as independent
    particle centers.

    heatmap: [B,1,H,W] sigmoid probabilities.
    kernel: positive odd integer; kernel=1 disables suppression.
    """
    kernel = int(kernel)

    if kernel < 1 or kernel % 2 == 0:
        raise ValueError(
            "local peak kernel must be a positive odd integer."
        )

    if kernel == 1:
        return heatmap

    pad = (kernel - 1) // 2

    pooled = F.max_pool2d(
        heatmap,
        kernel_size=kernel,
        stride=1,
        padding=pad,
    )

    keep = heatmap.eq(
        pooled
    )

    return (
        heatmap
        * keep.to(
            heatmap.dtype
        )
    )


@torch.no_grad()
def decode_instances(
    outputs: dict,
    input_size: tuple[int, int],
    stride: int = 4,
    score_threshold: float = 0.25,
    topk: int = 100,
    nms_iou: float = 0.5,
    mask_threshold: float = 0.5,
    local_peak_kernel: int = 3,
) -> List[dict[str, torch.Tensor]]:

    center = torch.sigmoid(
        outputs[
            "center_logits"
        ]
    )

    center = local_peak_suppression(
        center,
        kernel=local_peak_kernel,
    )

    size = outputs[
        "size"
    ]

    offset = outputs[
        "offset"
    ]

    coeff = outputs[
        "coeff"
    ]

    prototypes = outputs[
        "prototypes"
    ]

    bsz, _, fh, fw = (
        center.shape
    )

    input_h, input_w = (
        input_size
    )

    results = []

    for b in range(
        bsz
    ):

        scores_flat = (
            center[
                b,
                0,
            ]
            .flatten()
        )

        k = min(
            topk,
            int(
                scores_flat.numel()
            ),
        )

        scores, indices = torch.topk(
            scores_flat,
            k=k,
        )

        valid = (
            scores
            >= score_threshold
        )

        scores = scores[
            valid
        ]

        indices = indices[
            valid
        ]

        if (
            scores.numel()
            == 0
        ):

            results.append(
                {
                    "scores":
                        scores,

                    "boxes":
                        torch.empty(
                            (0, 4),
                            device=
                                center.device,
                        ),

                    "masks":
                        torch.empty(
                            (
                                0,
                                input_h,
                                input_w,
                            ),
                            dtype=
                                torch.bool,
                            device=
                                center.device,
                        ),
                }
            )

            continue

        ys = torch.div(
            indices,
            fw,
            rounding_mode=
                "floor",
        )

        xs = (
            indices
            % fw
        )

        wh = (
            size[
                b,
                :,
                ys,
                xs,
            ]
            .T
        )

        off = (
            offset[
                b,
                :,
                ys,
                xs,
            ]
            .T
        )

        cx = (
            xs.float()
            + off[:, 0]
        )

        cy = (
            ys.float()
            + off[:, 1]
        )

        x1 = (
            cx
            - 0.5
            * wh[:, 0]
        ) * stride

        y1 = (
            cy
            - 0.5
            * wh[:, 1]
        ) * stride

        x2 = (
            cx
            + 0.5
            * wh[:, 0]
        ) * stride

        y2 = (
            cy
            + 0.5
            * wh[:, 1]
        ) * stride

        boxes = torch.stack(
            [
                x1,
                y1,
                x2,
                y2,
            ],
            dim=1,
        )

        boxes[
            :,
            [0, 2],
        ] = boxes[
            :,
            [0, 2],
        ].clamp(
            0,
            input_w - 1,
        )

        boxes[
            :,
            [1, 3],
        ] = boxes[
            :,
            [1, 3],
        ].clamp(
            0,
            input_h - 1,
        )

        keep = nms(
            boxes,
            scores,
            iou_threshold=
                nms_iou,
        )

        scores = scores[
            keep
        ]

        boxes = boxes[
            keep
        ]

        c = (
            coeff[
                b,
                :,
                ys,
                xs,
            ]
            .T[
                keep
            ]
        )

        proto = prototypes[
            b
        ]

        mask_logits = torch.einsum(
            "nk,khw->nhw",
            c,
            proto,
        )

        mask_probs = torch.sigmoid(
            mask_logits
        ).unsqueeze(
            1
        )

        mask_probs = F.interpolate(
            mask_probs,
            size=(
                input_h,
                input_w,
            ),
            mode=
                "bilinear",
            align_corners=
                False,
        )[:, 0]

        # Crop each prototype mask to its predicted box.
        yy = torch.arange(
            input_h,
            device=
                center.device,
        )[
            None,
            :,
            None,
        ]

        xx = torch.arange(
            input_w,
            device=
                center.device,
        )[
            None,
            None,
            :,
        ]

        x1b = boxes[
            :,
            0,
        ][
            :,
            None,
            None,
        ]

        y1b = boxes[
            :,
            1,
        ][
            :,
            None,
            None,
        ]

        x2b = boxes[
            :,
            2,
        ][
            :,
            None,
            None,
        ]

        y2b = boxes[
            :,
            3,
        ][
            :,
            None,
            None,
        ]

        inside = (
            (xx >= x1b)
            & (xx <= x2b)
            & (yy >= y1b)
            & (yy <= y2b)
        )

        masks = (
            (
                mask_probs
                >= mask_threshold
            )
            & inside
        )

        results.append(
            {
                "scores":
                    scores,

                "boxes":
                    boxes,

                "masks":
                    masks,
            }
        )

    return results

Overwriting /kaggle/working/DASO-Net-Phase5B-KaggleTraining/src/dasonet/model/decoding.py


In [30]:
# ============================================================
# PHASE 5C — DECODER v2 SEMANTIC + FUNCTIONAL VERIFICATION
#
# No precomputed SHA assumption.
# We record the actual Kaggle-written file SHA after verifying:
#   1. Python syntax
#   2. expected decoder-v2 code markers
#   3. 3x3 local-peak suppression
#   4. kernel=1 identity behavior
# ============================================================

import hashlib
import importlib.util
import subprocess
import sys
from pathlib import Path

import torch


DECODER = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining/"
    "src/dasonet/model/decoding.py"
)


def sha256(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


if not DECODER.exists():
    raise RuntimeError(
        f"Decoder file is missing:\n{DECODER}"
    )


# ------------------------------------------------------------
# 1. Syntax verification
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(DECODER),
    ],
    check=True,
)


# ------------------------------------------------------------
# 2. Semantic source verification
# ------------------------------------------------------------

source = DECODER.read_text(
    encoding="utf-8"
)

required_markers = [
    "def local_peak_suppression(",
    "kernel: int = 3",
    "F.max_pool2d(",
    "center = local_peak_suppression(",
    "local_peak_kernel: int = 3",
    "kernel=local_peak_kernel",
    "mask_probs = F.interpolate(",
    "masks =",
]

missing = [
    marker
    for marker in required_markers
    if marker not in source
]

if missing:
    raise RuntimeError(
        "Decoder-v2 semantic verification failed.\n"
        f"Missing markers: {missing}"
    )


# ------------------------------------------------------------
# 3. Import exact file directly
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "dasonet_decoder_v2_verify",
    DECODER,
)

decoder = importlib.util.module_from_spec(
    spec
)

spec.loader.exec_module(
    decoder
)


# ------------------------------------------------------------
# 4. Local-peak functional test
# ------------------------------------------------------------

heat = torch.zeros(
    (1, 1, 9, 9),
    dtype=torch.float32,
)

# Gaussian-like cluster: only the strongest point should survive.
heat[0, 0, 4, 4] = 0.90
heat[0, 0, 4, 5] = 0.80
heat[0, 0, 5, 4] = 0.70

# Independent peak far enough away to survive.
heat[0, 0, 1, 1] = 0.85


suppressed = decoder.local_peak_suppression(
    heat,
    kernel=3,
)

before_count = int(
    (heat > 0)
    .sum()
    .item()
)

after_count = int(
    (suppressed > 0)
    .sum()
    .item()
)


print("Positive pixels before:", before_count)
print("Local peaks after:     ", after_count)


if before_count != 4:
    raise RuntimeError(
        f"Unexpected test setup: {before_count}"
    )

if after_count != 2:
    raise RuntimeError(
        "3x3 peak suppression failed. "
        f"Expected 2 surviving peaks, got {after_count}."
    )


if not torch.isclose(
    suppressed[0, 0, 4, 4],
    torch.tensor(0.90),
    atol=1e-6,
):
    raise RuntimeError(
        "Main peak was not preserved correctly."
    )


if not torch.isclose(
    suppressed[0, 0, 1, 1],
    torch.tensor(0.85),
    atol=1e-6,
):
    raise RuntimeError(
        "Independent peak was not preserved correctly."
    )


if float(
    suppressed[0, 0, 4, 5]
) != 0.0:
    raise RuntimeError(
        "Neighboring 0.80 response was not suppressed."
    )


if float(
    suppressed[0, 0, 5, 4]
) != 0.0:
    raise RuntimeError(
        "Neighboring 0.70 response was not suppressed."
    )


# ------------------------------------------------------------
# 5. kernel=1 must be identity
# ------------------------------------------------------------

identity = decoder.local_peak_suppression(
    heat,
    kernel=1,
)

if not torch.equal(
    identity,
    heat,
):
    raise RuntimeError(
        "kernel=1 identity test failed."
    )


# ------------------------------------------------------------
# Record the ACTUAL verified file hash
# ------------------------------------------------------------

actual_sha = sha256(
    DECODER
)


print()
print("Decoder path:")
print(DECODER)

print()
print("Verified decoder SHA256:")
print(actual_sha)

print()
print("Syntax check:             PASS")
print("Semantic marker check:    PASS")
print("3x3 peak suppression:     PASS")
print("kernel=1 identity:        PASS")

print()
print("========================================")
print("PHASE 5C DECODER LOCAL-PEAK FIX: PASS")
print("========================================")

Positive pixels before: 4
Local peaks after:      2

Decoder path:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/src/dasonet/model/decoding.py

Verified decoder SHA256:
9799eb1be568e3d6e0ce82b7c99df7a0ed05d271cd44ff24b612526cc18537d1

Syntax check:             PASS
Semantic marker check:    PASS
3x3 peak suppression:     PASS
kernel=1 identity:        PASS

PHASE 5C DECODER LOCAL-PEAK FIX: PASS


In [31]:
# ============================================================
# PHASE 5C — SAME-200-SCENE DECODER COMPARISON
#
# SAME:
#   - 200 validation scenes
#   - seed
#   - checkpoints
#   - score thresholds
#   - NMS IoU
#   - mask threshold
#   - matching IoU
#
# ONLY intended change:
#   raw top-k center pixels
#       ->
#   3x3 center local-maximum suppression before top-k
#
# No retraining.
# ============================================================

import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

BASELINE_CKPT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5b_amp32_workers2_finalcheck_v1/"
    "best.pt"
)

PILOT_CKPT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_pilot500_v1/"
    "best.pt"
)

OLD_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "pilot500_comparison_v1"
)

NEW_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "pilot500_comparison_peaknms_v1"
)


def sha256(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Verify corrected decoder is still present
# ------------------------------------------------------------

source = DECODER.read_text(
    encoding="utf-8"
)

required_markers = [
    "def local_peak_suppression(",
    "local_peak_kernel: int = 3",
    "center = local_peak_suppression(",
    "kernel=local_peak_kernel",
]

missing = [
    marker
    for marker in required_markers
    if marker not in source
]

if missing:
    raise RuntimeError(
        "Corrected decoder is not active.\n"
        f"Missing markers: {missing}"
    )


decoder_sha = sha256(
    DECODER
)

print("Verified decoder SHA256:")
print(decoder_sha)


if NEW_ROOT.exists():
    shutil.rmtree(
        NEW_ROOT
    )

NEW_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


runs = {
    "baseline_100_peaknms":
        BASELINE_CKPT,

    "pilot_500_peaknms":
        PILOT_CKPT,
}


summaries = {}


# ------------------------------------------------------------
# Evaluate same checkpoints on same deterministic 200 scenes
# ------------------------------------------------------------

for name, checkpoint in runs.items():

    if not checkpoint.exists():
        raise RuntimeError(
            f"Missing checkpoint:\n{checkpoint}"
        )

    out = (
        NEW_ROOT
        / name
    )

    cmd = [
        sys.executable,
        str(EVALUATOR),

        "--project-root",
        str(PROJECT_ROOT),

        "--checkpoint",
        str(checkpoint),

        "--output-dir",
        str(out),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--decode-min-score",
        "0.001",

        "--operating-score",
        "0.05",

        "--topk",
        "100",

        "--nms-iou",
        "0.5",

        "--mask-threshold",
        "0.5",

        "--match-iou",
        "0.5",
    ]

    print()
    print(
        "========================================"
    )

    print(
        "EVALUATING WITH 3x3 PEAK NMS:",
        name,
    )

    print(
        "========================================"
    )

    subprocess.run(
        cmd,
        check=True,
    )

    summary_file = (
        out
        / "phase5c_metric_summary.json"
    )

    if not summary_file.exists():
        raise RuntimeError(
            f"Missing summary:\n{summary_file}"
        )

    summaries[name] = json.loads(
        summary_file.read_text(
            encoding="utf-8"
        )
    )


# ------------------------------------------------------------
# Metric extraction
# ------------------------------------------------------------

def metric_row(summary):

    op = summary[
        "operating_point"
    ]

    ap = summary[
        "mask_ap"
    ]

    sweep = summary[
        "validation_threshold_sweep"
    ]

    return {
        "AP50":
            ap["mask_ap50"],

        "AP75":
            ap["mask_ap75"],

        "AP50:95":
            ap["mask_ap50_95"],

        "Precision@0.05":
            op["precision"],

        "Recall@0.05":
            op["recall"],

        "F1@0.05":
            op["f1"],

        "MatchedDice":
            op[
                "matched_mask_dice_mean"
            ],

        "CountMAE":
            op["count_mae"],

        "EmptyFP/image":
            op[
                "empty_scene_fp_per_image"
            ],

        "BestThreshold":
            sweep[
                "best_threshold_by_f1"
            ],

        "BestF1":
            sweep["best_f1"],
    }


base = metric_row(
    summaries[
        "baseline_100_peaknms"
    ]
)

pilot = metric_row(
    summaries[
        "pilot_500_peaknms"
    ]
)


# ------------------------------------------------------------
# Corrected 100-step vs 500-step table
# ------------------------------------------------------------

print()
print(
    "=============================================================="
)

print(
    "PHASE 5C — CORRECTED 3x3 PEAK-NMS COMPARISON"
)

print(
    "=============================================================="
)

print(
    "Decoder SHA256:",
    decoder_sha,
)

print()

print(
    f"{'Metric':<20}"
    f"{'100-step':>14}"
    f"{'500-step':>14}"
    f"{'Delta':>14}"
)

print(
    "-" * 62
)


for key in base:

    b = float(
        base[key]
    )

    p = float(
        pilot[key]
    )

    print(
        f"{key:<20}"
        f"{b:>14.4f}"
        f"{p:>14.4f}"
        f"{(p - b):>+14.4f}"
    )


# ------------------------------------------------------------
# Compare 500-step raw decoder vs corrected decoder
# ------------------------------------------------------------

old_pilot_file = (
    OLD_ROOT
    / "pilot_500"
    / "phase5c_metric_summary.json"
)


if not old_pilot_file.exists():
    raise RuntimeError(
        "Previous raw-decoder 500-step summary is missing:\n"
        f"{old_pilot_file}"
    )


old_pilot_summary = json.loads(
    old_pilot_file.read_text(
        encoding="utf-8"
    )
)

old_pilot = metric_row(
    old_pilot_summary
)


print()
print(
    "=============================================================="
)

print(
    "500-STEP DECODER EFFECT: RAW TOP-K -> 3x3 PEAK NMS"
)

print(
    "=============================================================="
)

print(
    f"{'Metric':<20}"
    f"{'raw':>14}"
    f"{'peak-nms':>14}"
    f"{'Delta':>14}"
)

print(
    "-" * 62
)


for key in [
    "AP50",
    "AP75",
    "AP50:95",
    "Precision@0.05",
    "Recall@0.05",
    "F1@0.05",
    "MatchedDice",
    "CountMAE",
    "EmptyFP/image",
    "BestThreshold",
    "BestF1",
]:

    old = float(
        old_pilot[key]
    )

    new = float(
        pilot[key]
    )

    print(
        f"{key:<20}"
        f"{old:>14.4f}"
        f"{new:>14.4f}"
        f"{(new - old):>+14.4f}"
    )


# ------------------------------------------------------------
# Confirm exact sample equivalence
# ------------------------------------------------------------

new_dist = summaries[
    "pilot_500_peaknms"
][
    "sample_particle_count_distribution"
]

old_dist = old_pilot_summary[
    "sample_particle_count_distribution"
]

if new_dist != old_dist:
    raise RuntimeError(
        "Particle-count distribution changed. "
        "Comparison is not apples-to-apples."
    )


new_fruit = summaries[
    "pilot_500_peaknms"
][
    "sample_fruit_distribution"
]

old_fruit = old_pilot_summary[
    "sample_fruit_distribution"
]

if new_fruit != old_fruit:
    raise RuntimeError(
        "Fruit distribution changed. "
        "Comparison is not apples-to-apples."
    )


# ------------------------------------------------------------
# Save comparison provenance
# ------------------------------------------------------------

comparison_manifest = {
    "phase":
        "5C",

    "status":
        "decoder_comparison_complete",

    "paper_result":
        False,

    "decoder_change":
        (
            "3x3 local-maximum suppression "
            "on sigmoid center heatmap before top-k"
        ),

    "decoder_sha256":
        decoder_sha,

    "scenes":
        200,

    "seed":
        20260917,

    "particle_count_distribution":
        new_dist,

    "fruit_distribution":
        new_fruit,

    "pilot_500_raw":
        old_pilot,

    "pilot_500_peaknms":
        pilot,
}


manifest_path = (
    NEW_ROOT
    / "decoder_comparison_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        comparison_manifest,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "200-scene particle-count distribution:"
)

print(
    new_dist
)


print()
print(
    "200-scene fruit distribution:"
)

print(
    new_fruit
)


print()
print(
    "Comparison manifest:"
)

print(
    manifest_path
)


print()
print(
    "========================================"
)

print(
    "CORRECTED DECODER COMPARISON: PASS"
)

print(
    "========================================"
)

print(
    "Do not start longer training until these results are reviewed."
)

Verified decoder SHA256:
9799eb1be568e3d6e0ce82b7c99df7a0ed05d271cd44ff24b612526cc18537d1

EVALUATING WITH 3x3 PEAK NMS: baseline_100_peaknms
eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=100 op_pred=33 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=100 op_pred=0 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=100 op_pred=5 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=100 op_pred=1 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=100 op_pred=3 elapsed=0.5m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=100 op_pred=1 elapsed=0.6m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=100 op_pred=68 elapsed=0.7m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=100 op_pred=66 elapsed=0.8m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=100 op_pred=93 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=100 op_pred=0 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0

In [32]:
%%writefile /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/diagnose_phase5c_oracle.py
from __future__ import annotations

import argparse
import csv
import importlib.util
import json
import math
import random
import sys
import time
from pathlib import Path

PACKAGE_ROOT = Path(__file__).resolve().parents[1]
SRC_ROOT = PACKAGE_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from dasonet.model.decoding import local_peak_suppression
from dasonet.training import (
    FruitMPSynTrainAdapter,
    collate_synthetic,
    seed_worker,
)


def parse_args():
    p = argparse.ArgumentParser(
        "DASO-Net Phase 5C oracle-component diagnostic"
    )

    p.add_argument(
        "--project-root",
        required=True,
    )

    p.add_argument(
        "--checkpoint",
        required=True,
    )

    p.add_argument(
        "--evaluator",
        required=True,
    )

    p.add_argument(
        "--output-dir",
        required=True,
    )

    p.add_argument(
        "--split",
        default="val",
        choices=["val", "test"],
    )

    p.add_argument(
        "--max-scenes",
        type=int,
        default=200,
    )

    p.add_argument(
        "--seed",
        type=int,
        default=20260917,
    )

    p.add_argument(
        "--num-workers",
        type=int,
        default=2,
    )

    p.add_argument(
        "--stride",
        type=int,
        default=4,
    )

    p.add_argument(
        "--peak-kernel",
        type=int,
        default=3,
    )

    p.add_argument(
        "--topk",
        type=int,
        default=100,
    )

    p.add_argument(
        "--mask-threshold",
        type=float,
        default=0.5,
    )

    return p.parse_args()


def load_evaluator(path: Path):
    spec = importlib.util.spec_from_file_location(
        "phase5c_evaluator_for_oracle",
        path,
    )

    if spec is None or spec.loader is None:
        raise RuntimeError(
            f"Could not import evaluator: {path}"
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


def summary_stats(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    values = values[
        np.isfinite(values)
    ]

    if values.size == 0:
        return {
            "n": 0,
            "mean": None,
            "median": None,
            "p25": None,
            "p75": None,
            "p90": None,
        }

    return {
        "n":
            int(values.size),

        "mean":
            float(np.mean(values)),

        "median":
            float(np.median(values)),

        "p25":
            float(np.quantile(values, 0.25)),

        "p75":
            float(np.quantile(values, 0.75)),

        "p90":
            float(np.quantile(values, 0.90)),
    }


def fraction_ge(values, threshold):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    if values.size == 0:
        return 0.0

    return float(
        np.mean(
            values >= threshold
        )
    )


def fraction_le(values, threshold):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    if values.size == 0:
        return 0.0

    return float(
        np.mean(
            values <= threshold
        )
    )


def pair_box_iou(
    pred_box: torch.Tensor,
    gt_box: torch.Tensor,
) -> float:

    x1 = torch.maximum(
        pred_box[0],
        gt_box[0],
    )

    y1 = torch.maximum(
        pred_box[1],
        gt_box[1],
    )

    x2 = torch.minimum(
        pred_box[2],
        gt_box[2],
    )

    y2 = torch.minimum(
        pred_box[3],
        gt_box[3],
    )

    inter_w = (
        x2 - x1
    ).clamp(min=0)

    inter_h = (
        y2 - y1
    ).clamp(min=0)

    inter = (
        inter_w
        * inter_h
    )

    pred_area = (
        (
            pred_box[2]
            - pred_box[0]
        ).clamp(min=0)
        *
        (
            pred_box[3]
            - pred_box[1]
        ).clamp(min=0)
    )

    gt_area = (
        (
            gt_box[2]
            - gt_box[0]
        ).clamp(min=0)
        *
        (
            gt_box[3]
            - gt_box[1]
        ).clamp(min=0)
    )

    union = (
        pred_area
        + gt_area
        - inter
    )

    return float(
        (
            inter
            / union.clamp(min=1e-6)
        ).item()
    )


def binary_iou(
    pred: torch.Tensor,
    gt: torch.Tensor,
) -> float:

    pred = pred.bool()
    gt = gt.bool()

    inter = torch.logical_and(
        pred,
        gt,
    ).sum().float()

    union = torch.logical_or(
        pred,
        gt,
    ).sum().float()

    if float(union.item()) <= 0:
        return 0.0

    return float(
        (
            inter / union
        ).item()
    )


def crop_boolean_mask(
    mask: torch.Tensor,
    box: torch.Tensor,
) -> torch.Tensor:

    h, w = mask.shape[-2:]

    x1 = int(
        math.floor(
            float(box[0].item())
        )
    )

    y1 = int(
        math.floor(
            float(box[1].item())
        )
    )

    x2 = int(
        math.ceil(
            float(box[2].item())
        )
    )

    y2 = int(
        math.ceil(
            float(box[3].item())
        )
    )

    x1 = max(
        0,
        min(w, x1),
    )

    y1 = max(
        0,
        min(h, y1),
    )

    x2 = max(
        0,
        min(w, x2),
    )

    y2 = max(
        0,
        min(h, y2),
    )

    out = torch.zeros_like(
        mask,
        dtype=torch.bool,
    )

    if (
        x2 > x1
        and y2 > y1
    ):
        out[
            y1:y2,
            x1:x2,
        ] = mask[
            y1:y2,
            x1:x2,
        ].bool()

    return out


def main():
    args = parse_args()

    random.seed(
        args.seed
    )

    np.random.seed(
        args.seed
    )

    torch.manual_seed(
        args.seed
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required."
        )

    device = torch.device(
        "cuda:0"
    )

    torch.cuda.set_device(
        device
    )

    project_root = Path(
        args.project_root
    ).resolve()

    checkpoint_path = Path(
        args.checkpoint
    ).resolve()

    evaluator_path = Path(
        args.evaluator
    ).resolve()

    output_dir = Path(
        args.output_dir
    ).resolve()

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    evaluator = load_evaluator(
        evaluator_path
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = evaluator.build_model(
        checkpoint,
        device,
    )

    dataset = FruitMPSynTrainAdapter(
        project_root,
        args.split,
        validate_frozen_hashes=True,
    )

    indices = evaluator.select_indices(
        dataset,
        args.max_scenes,
        args.seed,
    )

    loader_kwargs = {
        "dataset":
            Subset(
                dataset,
                indices,
            ),

        "batch_size":
            1,

        "shuffle":
            False,

        "num_workers":
            max(
                0,
                args.num_workers,
            ),

        "pin_memory":
            True,

        "collate_fn":
            collate_synthetic,

        "worker_init_fn":
            seed_worker,
    }

    if args.num_workers > 0:
        loader_kwargs[
            "prefetch_factor"
        ] = 2

        loader_kwargs[
            "persistent_workers"
        ] = True

    loader = DataLoader(
        **loader_kwargs
    )

    particle_rows = []

    gt_center_scores = []
    nearest_peak_distances = []

    box_ious = []
    center_errors = []

    width_ratios = []
    height_ratios = []

    mask_raw_ious = []
    mask_gtcrop_ious = []
    mask_predcrop_ious = []

    empty_max_peak_scores = []

    empty_peak_counts = {
        "0.05": [],
        "0.10": [],
        "0.20": [],
        "0.30": [],
        "0.50": [],
    }

    scene_count = 0
    particle_count = 0
    empty_scene_count = 0

    start = time.time()

    torch.cuda.reset_peak_memory_stats(
        device
    )

    with torch.inference_mode():

        for step, batch in enumerate(
            loader
        ):

            image = (
                batch[
                    "composite"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
            )

            target = (
                batch[
                    "targets"
                ][0]
            )

            gt_boxes = (
                target[
                    "boxes"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
                .float()
            )

            gt_masks = (
                target[
                    "masks"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
                .bool()
            )

            with torch.amp.autocast(
                "cuda",
                enabled=True,
                dtype=torch.float16,
            ):
                outputs = model(
                    image
                )

            center_prob = torch.sigmoid(
                outputs[
                    "center_logits"
                ]
            ).float()

            peak_map = local_peak_suppression(
                center_prob,
                kernel=args.peak_kernel,
            )

            fmap_h = int(
                center_prob.shape[-2]
            )

            fmap_w = int(
                center_prob.shape[-1]
            )

            flat_peaks = (
                peak_map[
                    0,
                    0,
                ]
                .flatten()
            )

            k = min(
                args.topk,
                int(
                    flat_peaks.numel()
                ),
            )

            peak_scores, peak_indices = torch.topk(
                flat_peaks,
                k=k,
            )

            peak_y = torch.div(
                peak_indices,
                fmap_w,
                rounding_mode="floor",
            ).float()

            peak_x = (
                peak_indices
                % fmap_w
            ).float()

            gt_count = int(
                gt_boxes.shape[0]
            )

            scene_count += 1
            particle_count += gt_count

            scene_id = (
                batch[
                    "scene_id"
                ][0]
            )

            fruit = (
                batch[
                    "fruit"
                ][0]
            )

            if gt_count == 0:

                empty_scene_count += 1

                empty_max_peak_scores.append(
                    float(
                        flat_peaks.max().item()
                    )
                )

                for threshold_text in (
                    "0.05",
                    "0.10",
                    "0.20",
                    "0.30",
                    "0.50",
                ):
                    threshold = float(
                        threshold_text
                    )

                    empty_peak_counts[
                        threshold_text
                    ].append(
                        int(
                            (
                                flat_peaks
                                >= threshold
                            )
                            .sum()
                            .item()
                        )
                    )

            if gt_count > 0:

                gt_cx_px = (
                    gt_boxes[:, 0]
                    + gt_boxes[:, 2]
                ) * 0.5

                gt_cy_px = (
                    gt_boxes[:, 1]
                    + gt_boxes[:, 3]
                ) * 0.5

                gt_cx_f = (
                    gt_cx_px
                    / float(args.stride)
                )

                gt_cy_f = (
                    gt_cy_px
                    / float(args.stride)
                )

                gx = torch.round(
                    gt_cx_f
                ).long().clamp(
                    0,
                    fmap_w - 1,
                )

                gy = torch.round(
                    gt_cy_f
                ).long().clamp(
                    0,
                    fmap_h - 1,
                )

                center_scores_here = (
                    center_prob[
                        0,
                        0,
                        gy,
                        gx,
                    ]
                )

                pred_size = (
                    outputs[
                        "size"
                    ][
                        0,
                        :,
                        gy,
                        gx,
                    ]
                    .float()
                    .T
                )

                pred_offset = (
                    outputs[
                        "offset"
                    ][
                        0,
                        :,
                        gy,
                        gx,
                    ]
                    .float()
                    .T
                )

                pred_cx_f = (
                    gx.float()
                    + pred_offset[:, 0]
                )

                pred_cy_f = (
                    gy.float()
                    + pred_offset[:, 1]
                )

                pred_cx_px = (
                    pred_cx_f
                    * float(args.stride)
                )

                pred_cy_px = (
                    pred_cy_f
                    * float(args.stride)
                )

                pred_w_px = (
                    pred_size[:, 0]
                    * float(args.stride)
                )

                pred_h_px = (
                    pred_size[:, 1]
                    * float(args.stride)
                )

                pred_boxes = torch.stack(
                    [
                        pred_cx_px
                        - 0.5
                        * pred_w_px,

                        pred_cy_px
                        - 0.5
                        * pred_h_px,

                        pred_cx_px
                        + 0.5
                        * pred_w_px,

                        pred_cy_px
                        + 0.5
                        * pred_h_px,
                    ],
                    dim=1,
                )

                input_h = int(
                    image.shape[-2]
                )

                input_w = int(
                    image.shape[-1]
                )

                pred_boxes[
                    :,
                    [0, 2],
                ] = pred_boxes[
                    :,
                    [0, 2],
                ].clamp(
                    0,
                    input_w - 1,
                )

                pred_boxes[
                    :,
                    [1, 3],
                ] = pred_boxes[
                    :,
                    [1, 3],
                ].clamp(
                    0,
                    input_h - 1,
                )

                coeff = (
                    outputs[
                        "coeff"
                    ][
                        0,
                        :,
                        gy,
                        gx,
                    ]
                    .float()
                    .T
                )

                proto = (
                    outputs[
                        "prototypes"
                    ][0]
                    .float()
                )

                mask_logits = torch.einsum(
                    "nk,khw->nhw",
                    coeff,
                    proto,
                )

                mask_probs = torch.sigmoid(
                    mask_logits
                ).unsqueeze(
                    1
                )

                mask_probs = F.interpolate(
                    mask_probs,
                    size=(
                        input_h,
                        input_w,
                    ),
                    mode="bilinear",
                    align_corners=False,
                )[:, 0]

                mask_binary = (
                    mask_probs
                    >= args.mask_threshold
                )

                gt_width = (
                    gt_boxes[:, 2]
                    - gt_boxes[:, 0]
                ).clamp(
                    min=1e-6
                )

                gt_height = (
                    gt_boxes[:, 3]
                    - gt_boxes[:, 1]
                ).clamp(
                    min=1e-6
                )

                for j in range(
                    gt_count
                ):

                    score_here = float(
                        center_scores_here[
                            j
                        ].item()
                    )

                    gt_center_scores.append(
                        score_here
                    )

                    distances = torch.sqrt(
                        (
                            peak_x
                            - gt_cx_f[j]
                        ) ** 2
                        +
                        (
                            peak_y
                            - gt_cy_f[j]
                        ) ** 2
                    ) * float(
                        args.stride
                    )

                    nearest_distance = float(
                        distances.min().item()
                    )

                    nearest_peak_distances.append(
                        nearest_distance
                    )

                    center_error = math.sqrt(
                        (
                            float(
                                pred_cx_px[j].item()
                            )
                            - float(
                                gt_cx_px[j].item()
                            )
                        ) ** 2
                        +
                        (
                            float(
                                pred_cy_px[j].item()
                            )
                            - float(
                                gt_cy_px[j].item()
                            )
                        ) ** 2
                    )

                    center_errors.append(
                        center_error
                    )

                    b_iou = pair_box_iou(
                        pred_boxes[j],
                        gt_boxes[j],
                    )

                    box_ious.append(
                        b_iou
                    )

                    width_ratio = float(
                        (
                            pred_w_px[j]
                            / gt_width[j]
                        ).item()
                    )

                    height_ratio = float(
                        (
                            pred_h_px[j]
                            / gt_height[j]
                        ).item()
                    )

                    width_ratios.append(
                        width_ratio
                    )

                    height_ratios.append(
                        height_ratio
                    )

                    raw_mask = (
                        mask_binary[j]
                    )

                    gt_crop_mask = crop_boolean_mask(
                        raw_mask,
                        gt_boxes[j],
                    )

                    pred_crop_mask = crop_boolean_mask(
                        raw_mask,
                        pred_boxes[j],
                    )

                    gt_mask = (
                        gt_masks[j]
                    )

                    raw_iou = binary_iou(
                        raw_mask,
                        gt_mask,
                    )

                    gt_crop_iou = binary_iou(
                        gt_crop_mask,
                        gt_mask,
                    )

                    pred_crop_iou = binary_iou(
                        pred_crop_mask,
                        gt_mask,
                    )

                    mask_raw_ious.append(
                        raw_iou
                    )

                    mask_gtcrop_ious.append(
                        gt_crop_iou
                    )

                    mask_predcrop_ious.append(
                        pred_crop_iou
                    )

                    particle_rows.append(
                        {
                            "scene_id":
                                scene_id,

                            "fruit":
                                fruit,

                            "gt_index":
                                j,

                            "gt_center_score":
                                score_here,

                            "nearest_top100_peak_distance_px":
                                nearest_distance,

                            "oracle_gtcell_center_error_px":
                                center_error,

                            "oracle_gtcell_box_iou":
                                b_iou,

                            "pred_to_gt_width_ratio":
                                width_ratio,

                            "pred_to_gt_height_ratio":
                                height_ratio,

                            "oracle_gtcell_raw_mask_iou":
                                raw_iou,

                            "oracle_gtcell_gtbox_crop_mask_iou":
                                gt_crop_iou,

                            "oracle_gtcell_predbox_crop_mask_iou":
                                pred_crop_iou,
                        }
                    )

            if (
                step % 20 == 0
                or step + 1
                == len(loader)
            ):
                print(
                    f"diagnostic "
                    f"{step + 1:04d}/"
                    f"{len(loader):04d} "
                    f"scene={scene_id} "
                    f"gt={gt_count} "
                    f"elapsed="
                    f"{(time.time() - start) / 60.0:.1f}m"
                )

    elapsed = (
        time.time()
        - start
    )

    summary = {
        "phase":
            "5C",

        "diagnostic":
            "oracle_component_isolation",

        "paper_result":
            False,

        "checkpoint":
            str(
                checkpoint_path
            ),

        "checkpoint_epoch":
            int(
                checkpoint.get(
                    "epoch",
                    -1,
                )
            ),

        "checkpoint_optimizer_step":
            int(
                checkpoint.get(
                    "optimizer_step",
                    -1,
                )
            ),

        "scenes":
            scene_count,

        "particles":
            particle_count,

        "empty_scenes":
            empty_scene_count,

        "center_head": {
            "gt_cell_score_stats":
                summary_stats(
                    gt_center_scores
                ),

            "nearest_top100_peak_distance_px_stats":
                summary_stats(
                    nearest_peak_distances
                ),

            "top100_peak_recall_within_4px":
                fraction_le(
                    nearest_peak_distances,
                    4.0,
                ),

            "top100_peak_recall_within_8px":
                fraction_le(
                    nearest_peak_distances,
                    8.0,
                ),

            "top100_peak_recall_within_16px":
                fraction_le(
                    nearest_peak_distances,
                    16.0,
                ),

            "top100_peak_recall_within_32px":
                fraction_le(
                    nearest_peak_distances,
                    32.0,
                ),
        },

        "oracle_gt_cell_geometry": {
            "center_error_px_stats":
                summary_stats(
                    center_errors
                ),

            "box_iou_stats":
                summary_stats(
                    box_ious
                ),

            "box_iou_ge_0_50":
                fraction_ge(
                    box_ious,
                    0.50,
                ),

            "box_iou_ge_0_75":
                fraction_ge(
                    box_ious,
                    0.75,
                ),

            "pred_to_gt_width_ratio_stats":
                summary_stats(
                    width_ratios
                ),

            "pred_to_gt_height_ratio_stats":
                summary_stats(
                    height_ratios
                ),
        },

        "oracle_gt_cell_mask": {
            "raw_mask_iou_stats":
                summary_stats(
                    mask_raw_ious
                ),

            "raw_mask_iou_ge_0_50":
                fraction_ge(
                    mask_raw_ious,
                    0.50,
                ),

            "gt_box_crop_mask_iou_stats":
                summary_stats(
                    mask_gtcrop_ious
                ),

            "gt_box_crop_mask_iou_ge_0_50":
                fraction_ge(
                    mask_gtcrop_ious,
                    0.50,
                ),

            "pred_box_crop_mask_iou_stats":
                summary_stats(
                    mask_predcrop_ious
                ),

            "pred_box_crop_mask_iou_ge_0_50":
                fraction_ge(
                    mask_predcrop_ious,
                    0.50,
                ),
        },

        "empty_scene_center_behavior": {
            "max_peak_score_stats":
                summary_stats(
                    empty_max_peak_scores
                ),

            "mean_local_peak_count_by_threshold": {
                threshold:
                    (
                        float(
                            np.mean(
                                counts
                            )
                        )
                        if counts
                        else 0.0
                    )
                for threshold, counts
                in empty_peak_counts.items()
            },
        },

        "runtime": {
            "minutes":
                elapsed / 60.0,

            "peak_reserved_vram_gb":
                float(
                    torch.cuda
                    .max_memory_reserved(
                        device
                    )
                    / (1024 ** 3)
                ),
        },
    }

    output_json = (
        output_dir
        / "oracle_component_diagnostic.json"
    )

    output_json.write_text(
        json.dumps(
            summary,
            indent=2,
        ),
        encoding="utf-8",
    )

    output_csv = (
        output_dir
        / "oracle_particle_diagnostic.csv"
    )

    with output_csv.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as f:

        if particle_rows:
            writer = csv.DictWriter(
                f,
                fieldnames=list(
                    particle_rows[0].keys()
                ),
            )

            writer.writeheader()

            writer.writerows(
                particle_rows
            )

    center = summary[
        "center_head"
    ]

    geom = summary[
        "oracle_gt_cell_geometry"
    ]

    masks = summary[
        "oracle_gt_cell_mask"
    ]

    empty = summary[
        "empty_scene_center_behavior"
    ]

    print()
    print(
        "============================================================"
    )

    print(
        "PHASE 5C — ORACLE COMPONENT DIAGNOSTIC"
    )

    print(
        "============================================================"
    )

    print(
        "Scenes:",
        scene_count,
    )

    print(
        "GT particles:",
        particle_count,
    )

    print(
        "Empty scenes:",
        empty_scene_count,
    )

    print()
    print(
        "--- CENTER HEAD ---"
    )

    print(
        "GT-cell center score mean:",
        f"{center['gt_cell_score_stats']['mean']:.4f}",
    )

    print(
        "GT-cell center score median:",
        f"{center['gt_cell_score_stats']['median']:.4f}",
    )

    print(
        "Top100 peak recall <= 4 px:",
        f"{center['top100_peak_recall_within_4px']:.4f}",
    )

    print(
        "Top100 peak recall <= 8 px:",
        f"{center['top100_peak_recall_within_8px']:.4f}",
    )

    print(
        "Top100 peak recall <=16 px:",
        f"{center['top100_peak_recall_within_16px']:.4f}",
    )

    print(
        "Top100 peak recall <=32 px:",
        f"{center['top100_peak_recall_within_32px']:.4f}",
    )

    print()
    print(
        "--- ORACLE GT-CELL GEOMETRY ---"
    )

    print(
        "Center error mean px:",
        f"{geom['center_error_px_stats']['mean']:.3f}",
    )

    print(
        "Box IoU mean:",
        f"{geom['box_iou_stats']['mean']:.4f}",
    )

    print(
        "Box IoU >=0.50:",
        f"{geom['box_iou_ge_0_50']:.4f}",
    )

    print(
        "Box IoU >=0.75:",
        f"{geom['box_iou_ge_0_75']:.4f}",
    )

    print(
        "Width ratio median:",
        f"{geom['pred_to_gt_width_ratio_stats']['median']:.4f}",
    )

    print(
        "Height ratio median:",
        f"{geom['pred_to_gt_height_ratio_stats']['median']:.4f}",
    )

    print()
    print(
        "--- ORACLE GT-CELL MASK ---"
    )

    print(
        "Raw mask IoU mean:",
        f"{masks['raw_mask_iou_stats']['mean']:.4f}",
    )

    print(
        "Raw mask IoU >=0.50:",
        f"{masks['raw_mask_iou_ge_0_50']:.4f}",
    )

    print(
        "GT-box-crop mask IoU mean:",
        f"{masks['gt_box_crop_mask_iou_stats']['mean']:.4f}",
    )

    print(
        "GT-box-crop mask IoU >=0.50:",
        f"{masks['gt_box_crop_mask_iou_ge_0_50']:.4f}",
    )

    print(
        "Pred-box-crop mask IoU mean:",
        f"{masks['pred_box_crop_mask_iou_stats']['mean']:.4f}",
    )

    print(
        "Pred-box-crop mask IoU >=0.50:",
        f"{masks['pred_box_crop_mask_iou_ge_0_50']:.4f}",
    )

    print()
    print(
        "--- EMPTY-SCENE CENTER BEHAVIOR ---"
    )

    print(
        "Empty max peak score mean:",
        f"{empty['max_peak_score_stats']['mean']:.4f}",
    )

    print(
        "Empty max peak score median:",
        f"{empty['max_peak_score_stats']['median']:.4f}",
    )

    for threshold, count in (
        empty[
            "mean_local_peak_count_by_threshold"
        ].items()
    ):
        print(
            f"Empty local peaks >= {threshold}:",
            f"{count:.2f} / image",
        )

    print()
    print(
        "Runtime:",
        f"{elapsed / 60.0:.2f} min",
    )

    print(
        "Peak reserved VRAM:",
        f"{summary['runtime']['peak_reserved_vram_gb']:.2f} GB",
    )

    print(
        "JSON:",
        output_json,
    )

    print(
        "CSV:",
        output_csv,
    )

    print()
    print(
        "PHASE 5C ORACLE DIAGNOSTIC: PASS"
    )


if __name__ == "__main__":
    main()

Writing /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/diagnose_phase5c_oracle.py


In [33]:
# ============================================================
# PHASE 5C — RUN ORACLE COMPONENT DIAGNOSTIC
#
# Uses the 500-step checkpoint and exactly the same
# deterministic 200 validation scenes as the previous comparison.
#
# No training occurs here.
# ============================================================

import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CHECKPOINT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_pilot500_v1/"
    "best.pt"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

DIAGNOSTIC = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "oracle_component_diagnostic_v1"
)


for required in (
    PROJECT_ROOT,
    CHECKPOINT,
    EVALUATOR,
    DIAGNOSTIC,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


if OUTPUT_DIR.exists():
    shutil.rmtree(
        OUTPUT_DIR
    )


cmd = [
    sys.executable,
    str(DIAGNOSTIC),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(CHECKPOINT),

    "--evaluator",
    str(EVALUATOR),

    "--output-dir",
    str(OUTPUT_DIR),

    "--split",
    "val",

    "--max-scenes",
    "200",

    "--seed",
    "20260917",

    "--num-workers",
    "2",

    "--stride",
    "4",

    "--peak-kernel",
    "3",

    "--topk",
    "100",

    "--mask-threshold",
    "0.5",
]


print(
    "============================================================"
)

print(
    "PHASE 5C — RUNNING ORACLE COMPONENT DIAGNOSTIC"
)

print(
    "============================================================"
)

print(
    " ".join(cmd)
)

print()


subprocess.run(
    cmd,
    check=True,
)


summary_path = (
    OUTPUT_DIR
    / "oracle_component_diagnostic.json"
)

if not summary_path.exists():
    raise RuntimeError(
        "Oracle diagnostic JSON was not created."
    )


summary = json.loads(
    summary_path.read_text(
        encoding="utf-8"
    )
)


if (
    summary["scenes"]
    != 200
):
    raise RuntimeError(
        "Expected exactly 200 scenes."
    )


if (
    summary["particles"]
    <= 0
):
    raise RuntimeError(
        "No GT particles were evaluated."
    )


print()
print(
    "============================================================"
)

print(
    "ORACLE DIAGNOSTIC FILE CHECK: PASS"
)

print(
    "============================================================"
)

print(
    "Scenes:",
    summary["scenes"],
)

print(
    "Particles:",
    summary["particles"],
)

print(
    "Empty scenes:",
    summary["empty_scenes"],
)

print(
    "Summary:",
    summary_path,
)

PHASE 5C — RUNNING ORACLE COMPONENT DIAGNOSTIC
/usr/bin/python3 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/diagnose_phase5c_oracle.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --checkpoint /kaggle/working/dasonet_phase5b/dasonet_phase5c_pilot500_v1/best.pt --evaluator /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/evaluate_phase5c.py --output-dir /kaggle/working/dasonet_phase5c/oracle_component_diagnostic_v1 --split val --max-scenes 200 --seed 20260917 --num-workers 2 --stride 4 --peak-kernel 3 --topk 100 --mask-threshold 0.5

diagnostic 0001/0200 scene=369b96d7adde8c0026a7 gt=1 elapsed=0.1m
diagnostic 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 elapsed=0.3m
diagnostic 0041/0200 scene=11e8bd15bed065be3400 gt=2 elapsed=0.5m
diagnostic 0061/0200 scene=a3cb9b50dca442394cbe gt=2 elapsed=0.7m
diagnostic 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 elapsed=0.9m
diagnostic 0101/0200 scene=d3df4c3febde0d7548bc gt=0 elapsed=1.1m
diagno

In [34]:
# ============================================================
# PHASE 5C — SCHEDULE-CAPACITY PROBE v1
#
# Scientific question:
# Was the 500-step pilot simply too dominated by LR warm-up?
#
# SAME:
#   model architecture
#   instance/PCC/CRPA losses
#   renderer/data
#   T4x2
#   native 1024
#   batch/GPU = 1
#   accumulation = 4
#   effective batch = 8
#   AMP initial scale = 32
#
# ONLY optimization-schedule change:
#   50 optimizer updates warm-up
#   then constant LR = 3e-4
#
# Training:
#   1000 train steps/rank
#   250 optimizer updates
#   100 val-loss steps
#
# Engineering diagnostic only — NOT a paper result.
# ============================================================

import json
import shutil
import subprocess
import time
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

WRAPPER = Path(
    "/kaggle/working/"
    "launch_phase5b_amp32.py"
)

BASE_CONFIG = (
    CODE_ROOT
    / "configs"
    / "phase5b_kaggle_v1.json"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_scheduleprobe1000_v1.json"
)

RUN_NAME = (
    "dasonet_phase5c_scheduleprobe1000_v1"
)

OUT = (
    Path(
        "/kaggle/working/"
        "dasonet_phase5b"
    )
    / RUN_NAME
)


for required in (
    CODE_ROOT,
    PROJECT_ROOT,
    WRAPPER,
    BASE_CONFIG,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


# Preserve the DDP-safe zero-particle loss fix.
instance_file = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "losses"
    / "instance.py"
)

instance_text = instance_file.read_text(
    encoding="utf-8"
)

if (
    "DDP-safe zero-particle handling"
    not in instance_text
):
    raise RuntimeError(
        "DDP-safe zero-particle handling is missing."
    )


# Preserve the verified Phase-5C peak decoder.
decoder_file = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

decoder_text = decoder_file.read_text(
    encoding="utf-8"
)

if (
    "def local_peak_suppression("
    not in decoder_text
):
    raise RuntimeError(
        "Phase-5C local-peak decoder is missing."
    )


with BASE_CONFIG.open(
    "r",
    encoding="utf-8",
) as f:
    cfg = json.load(f)


cfg["experiment_name"] = RUN_NAME


# ------------------------------------------------------------
# Frozen practical hardware setup
# ------------------------------------------------------------

cfg["data"][
    "per_rank_synthetic_batch"
] = 1

cfg["data"][
    "per_rank_anchor_batch"
] = 2

cfg["data"][
    "synthetic_workers_per_rank"
] = 2

cfg["data"][
    "anchor_workers_per_rank"
] = 1

cfg["data"][
    "prefetch_factor"
] = 2


cfg["optimization"][
    "gradient_accumulation_steps"
] = 4

cfg["optimization"][
    "amp"
] = True


# ------------------------------------------------------------
# Controlled schedule probe
#
# 1000 train steps / accumulation 4
# = 250 optimizer updates.
#
# Warm up for 50 updates, then stay at base LR.
#
# Setting min_lr_ratio = 1.0 makes the post-warm-up
# LambdaLR multiplier exactly 1.0 throughout.
# ------------------------------------------------------------

cfg["optimization"][
    "learning_rate"
] = 3e-4

cfg["optimization"][
    "warmup_optimizer_steps"
] = 50

cfg["optimization"][
    "min_lr_ratio"
] = 1.0


with CONFIG.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        cfg,
        f,
        indent=2,
    )


if OUT.exists():
    shutil.rmtree(
        OUT
    )


cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(WRAPPER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "1",

    "--max-train-steps",
    "1000",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    "none",
]


print(
    "============================================================"
)

print(
    "PHASE 5C — 1000-STEP SCHEDULE-CAPACITY PROBE"
)

print(
    "============================================================"
)

print(
    "Expected optimizer updates: 250"
)

print(
    "Warm-up optimizer updates:  50"
)

print(
    "Post-warm-up LR:            3.0e-4 constant"
)

print()

print(
    " ".join(cmd)
)

print()


start = time.time()

subprocess.run(
    cmd,
    check=True,
)

elapsed = (
    time.time()
    - start
)


status_path = (
    OUT
    / "run_status.json"
)

if not status_path.exists():
    raise RuntimeError(
        "Schedule-probe run_status.json missing."
    )


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


print()
print(
    "============================================================"
)

print(
    "PHASE 5C SCHEDULE-CAPACITY TRAINING COMPLETE"
)

print(
    "============================================================"
)

print(
    f"Wall time: "
    f"{elapsed / 60.0:.2f} min"
)

print(
    "AMP skipped optimizer steps:",
    amp_skips,
)

print(
    "Best val total loss:",
    status.get(
        "best_val_total_loss"
    ),
)

print(
    "Best checkpoint:",
    status.get(
        "best_checkpoint"
    ),
)

print(
    "Last checkpoint:",
    status.get(
        "last_checkpoint"
    ),
)


if amp_skips != 0:
    raise RuntimeError(
        "Unexpected AMP skip(s). "
        "Do not evaluate this run."
    )


print()
print(
    "PHASE 5C SCHEDULE-CAPACITY PROBE TRAINING: PASS"
)

PHASE 5C — 1000-STEP SCHEDULE-CAPACITY PROBE
Expected optimizer updates: 250
Warm-up optimizer updates:  50
Post-warm-up LR:            3.0e-4 constant

torchrun --standalone --nproc_per_node=2 /kaggle/working/launch_phase5b_amp32.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_scheduleprobe1000_v1.json --output-root /kaggle/working --epochs 1 --max-train-steps 1000 --max-val-steps 100 --accum 4 --num-workers 2 --log-every 100 --resume none



W0913 10:01:41.723000 1190 torch/distributed/run.py:852] 
W0913 10:01:41.723000 1190 torch/distributed/run.py:852] *****************************************
W0913 10:01:41.723000 1190 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 10:01:41.723000 1190 torch/distributed/run.py:852] *****************************************
[W913 10:01:42.297885506 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5B AMP wrapper
Trainer: /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py
Forced AMP initial scale: 32.0
Trainer modification: NONE
DASO-Net Phase 5B AMP wrapper
Trainer: /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py
Forced AMP initial scale: 32.0
Trainer modification: NONE


[W913 10:01:44.092592669 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 10:01:44.135988767 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_scheduleprobe1000_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      250
Smoke mode:                 False
epoch=00 step=00000/01000 loss=27.7588 inst=27.1546 pcc=0.0000 crpa=0.6041 lr=6.00e-06 vram_alloc=8.84GB vram_res=9.11GB
epoch=00 step=00100/01000 loss=7.4273 inst=6.8452 pcc=0.0000 crpa=0.5821 lr=1.56e-04 vram_alloc=8.97GB vram_res=9.11GB
epoch=00 step=00200/01000 loss=0.4604 inst=0.1134 pcc=0.0000 crpa=0.

In [35]:
# ============================================================
# PHASE 5C — SCHEDULE-PROBE EVALUATION
#
# Compares:
#   previous 500-step pilot
#       vs
#   fresh 1000-step schedule-capacity probe
#
# Same deterministic 200 validation scenes.
# Current verified 3x3 peak decoder.
#
# Then reruns oracle-component diagnostic.
# ============================================================

import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

DIAGNOSTIC = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

NEW_CHECKPOINT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_scheduleprobe1000_v1/"
    "best.pt"
)

OLD_METRIC_JSON = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "pilot500_comparison_peaknms_v1/"
    "pilot_500_peaknms/"
    "phase5c_metric_summary.json"
)

OLD_ORACLE_JSON = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "oracle_component_diagnostic_v1/"
    "oracle_component_diagnostic.json"
)

ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "scheduleprobe1000_evaluation_v1"
)

METRIC_OUT = (
    ROOT
    / "metrics"
)

ORACLE_OUT = (
    ROOT
    / "oracle"
)


for required in (
    NEW_CHECKPOINT,
    OLD_METRIC_JSON,
    OLD_ORACLE_JSON,
    EVALUATOR,
    DIAGNOSTIC,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


if ROOT.exists():
    shutil.rmtree(
        ROOT
    )

ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 1. Same-200-scene quantitative evaluation
# ------------------------------------------------------------

metric_cmd = [
    sys.executable,
    str(EVALUATOR),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(NEW_CHECKPOINT),

    "--output-dir",
    str(METRIC_OUT),

    "--split",
    "val",

    "--max-scenes",
    "200",

    "--seed",
    "20260917",

    "--num-workers",
    "2",

    "--decode-min-score",
    "0.001",

    "--operating-score",
    "0.05",

    "--topk",
    "100",

    "--nms-iou",
    "0.5",

    "--mask-threshold",
    "0.5",

    "--match-iou",
    "0.5",
]


print(
    "============================================================"
)

print(
    "PHASE 5C — EVALUATING 1000-STEP SCHEDULE PROBE"
)

print(
    "============================================================"
)

subprocess.run(
    metric_cmd,
    check=True,
)


# ------------------------------------------------------------
# 2. Same-200-scene oracle component diagnostic
# ------------------------------------------------------------

oracle_cmd = [
    sys.executable,
    str(DIAGNOSTIC),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(NEW_CHECKPOINT),

    "--evaluator",
    str(EVALUATOR),

    "--output-dir",
    str(ORACLE_OUT),

    "--split",
    "val",

    "--max-scenes",
    "200",

    "--seed",
    "20260917",

    "--num-workers",
    "2",

    "--stride",
    "4",

    "--peak-kernel",
    "3",

    "--topk",
    "100",

    "--mask-threshold",
    "0.5",
]


print()
print(
    "============================================================"
)

print(
    "PHASE 5C — ORACLE DIAGNOSTIC FOR SCHEDULE PROBE"
)

print(
    "============================================================"
)

subprocess.run(
    oracle_cmd,
    check=True,
)


# ------------------------------------------------------------
# Load outputs
# ------------------------------------------------------------

NEW_METRIC_JSON = (
    METRIC_OUT
    / "phase5c_metric_summary.json"
)

NEW_ORACLE_JSON = (
    ORACLE_OUT
    / "oracle_component_diagnostic.json"
)


old_metric = json.loads(
    OLD_METRIC_JSON.read_text(
        encoding="utf-8"
    )
)

new_metric = json.loads(
    NEW_METRIC_JSON.read_text(
        encoding="utf-8"
    )
)

old_oracle = json.loads(
    OLD_ORACLE_JSON.read_text(
        encoding="utf-8"
    )
)

new_oracle = json.loads(
    NEW_ORACLE_JSON.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# Exact sample equivalence check
# ------------------------------------------------------------

if (
    old_metric[
        "sample_particle_count_distribution"
    ]
    !=
    new_metric[
        "sample_particle_count_distribution"
    ]
):
    raise RuntimeError(
        "Particle-count distribution mismatch."
    )


if (
    old_metric[
        "sample_fruit_distribution"
    ]
    !=
    new_metric[
        "sample_fruit_distribution"
    ]
):
    raise RuntimeError(
        "Fruit distribution mismatch."
    )


# ------------------------------------------------------------
# Metric helpers
# ------------------------------------------------------------

def metrics(summary):

    op = summary[
        "operating_point"
    ]

    ap = summary[
        "mask_ap"
    ]

    sweep = summary[
        "validation_threshold_sweep"
    ]

    return {
        "AP50":
            float(
                ap["mask_ap50"]
            ),

        "AP75":
            float(
                ap["mask_ap75"]
            ),

        "AP50:95":
            float(
                ap["mask_ap50_95"]
            ),

        "Precision@0.05":
            float(
                op["precision"]
            ),

        "Recall@0.05":
            float(
                op["recall"]
            ),

        "F1@0.05":
            float(
                op["f1"]
            ),

        "MatchedDice":
            float(
                op[
                    "matched_mask_dice_mean"
                ]
            ),

        "CountMAE":
            float(
                op["count_mae"]
            ),

        "EmptyFP/image":
            float(
                op[
                    "empty_scene_fp_per_image"
                ]
            ),

        "BestThreshold":
            float(
                sweep[
                    "best_threshold_by_f1"
                ]
            ),

        "BestF1":
            float(
                sweep["best_f1"]
            ),
    }


old_m = metrics(
    old_metric
)

new_m = metrics(
    new_metric
)


print()
print(
    "=================================================================="
)

print(
    "500-STEP -> 1000-STEP SCHEDULE-CAPACITY METRIC COMPARISON"
)

print(
    "=================================================================="
)

print(
    f"{'Metric':<22}"
    f"{'500-step':>14}"
    f"{'1000-step':>14}"
    f"{'Delta':>14}"
)

print(
    "-" * 64
)


for key in old_m:

    old = old_m[key]
    new = new_m[key]

    print(
        f"{key:<22}"
        f"{old:>14.4f}"
        f"{new:>14.4f}"
        f"{(new - old):>+14.4f}"
    )


# ------------------------------------------------------------
# Oracle helper
# ------------------------------------------------------------

def oracle_values(summary):

    center = summary[
        "center_head"
    ]

    geom = summary[
        "oracle_gt_cell_geometry"
    ]

    mask = summary[
        "oracle_gt_cell_mask"
    ]

    empty = summary[
        "empty_scene_center_behavior"
    ]

    return {
        "GTCenterScoreMean":
            float(
                center[
                    "gt_cell_score_stats"
                ][
                    "mean"
                ]
            ),

        "PeakRecall<=4px":
            float(
                center[
                    "top100_peak_recall_within_4px"
                ]
            ),

        "PeakRecall<=8px":
            float(
                center[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall<=16px":
            float(
                center[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "CenterErrorPx":
            float(
                geom[
                    "center_error_px_stats"
                ][
                    "mean"
                ]
            ),

        "BoxIoUMean":
            float(
                geom[
                    "box_iou_stats"
                ][
                    "mean"
                ]
            ),

        "BoxIoU>=0.50":
            float(
                geom[
                    "box_iou_ge_0_50"
                ]
            ),

        "WidthRatioMedian":
            float(
                geom[
                    "pred_to_gt_width_ratio_stats"
                ][
                    "median"
                ]
            ),

        "HeightRatioMedian":
            float(
                geom[
                    "pred_to_gt_height_ratio_stats"
                ][
                    "median"
                ]
            ),

        "GTBoxMaskIoUMean":
            float(
                mask[
                    "gt_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),

        "GTBoxMaskIoU>=0.50":
            float(
                mask[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),

        "EmptyMaxPeakMean":
            float(
                empty[
                    "max_peak_score_stats"
                ][
                    "mean"
                ]
            ),

        "EmptyPeaks>=0.05":
            float(
                empty[
                    "mean_local_peak_count_by_threshold"
                ][
                    "0.05"
                ]
            ),
    }


old_o = oracle_values(
    old_oracle
)

new_o = oracle_values(
    new_oracle
)


print()
print(
    "=================================================================="
)

print(
    "500-STEP -> 1000-STEP ORACLE-COMPONENT COMPARISON"
)

print(
    "=================================================================="
)

print(
    f"{'Diagnostic':<24}"
    f"{'500-step':>14}"
    f"{'1000-step':>14}"
    f"{'Delta':>14}"
)

print(
    "-" * 66
)


for key in old_o:

    old = old_o[key]
    new = new_o[key]

    print(
        f"{key:<24}"
        f"{old:>14.4f}"
        f"{new:>14.4f}"
        f"{(new - old):>+14.4f}"
    )


# ------------------------------------------------------------
# Save compact comparison record
# ------------------------------------------------------------

comparison = {
    "phase":
        "5C",

    "paper_result":
        False,

    "experiment":
        "schedule_capacity_probe_1000",

    "controlled_change": {
        "train_steps_per_rank":
            1000,

        "optimizer_updates":
            250,

        "warmup_optimizer_steps":
            50,

        "base_lr":
            3e-4,

        "post_warmup_schedule":
            "constant",

        "architecture_changed":
            False,

        "loss_weights_changed":
            False,

        "data_changed":
            False,
    },

    "metric_500":
        old_m,

    "metric_1000":
        new_m,

    "oracle_500":
        old_o,

    "oracle_1000":
        new_o,
}


comparison_path = (
    ROOT
    / "schedule_capacity_comparison.json"
)

comparison_path.write_text(
    json.dumps(
        comparison,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "Particle-count distribution:"
)

print(
    new_metric[
        "sample_particle_count_distribution"
    ]
)

print()
print(
    "Fruit distribution:"
)

print(
    new_metric[
        "sample_fruit_distribution"
    ]
)

print()
print(
    "Comparison JSON:"
)

print(
    comparison_path
)

print()
print(
    "============================================================"
)

print(
    "PHASE 5C SCHEDULE-CAPACITY EVALUATION: PASS"
)

print(
    "============================================================"
)

print(
    "Do not launch longer training until this comparison is reviewed."
)

PHASE 5C — EVALUATING 1000-STEP SCHEDULE PROBE
eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=99 op_pred=99 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=100 op_pred=100 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=99 op_pred=99 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=99 op_pred=99 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=100 op_pred=100 elapsed=0.5m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=100 op_pred=43 elapsed=0.6m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=98 op_pred=85 elapsed=0.7m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=100 op_pred=87 elapsed=0.8m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=99 op_pred=99 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=100 op_pred=82 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0d7548bc gt=0 decoded=100 op_pred=100 elapsed=1.1m
eval 0111/0200 scene=858baec99120971a3e47 

In [36]:
%%writefile /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/diagnose_phase5c_counterfactual_center.py
from __future__ import annotations

import argparse
import importlib.util
import json
import random
import sys
import time
from pathlib import Path

PACKAGE_ROOT = Path(__file__).resolve().parents[1]
SRC_ROOT = PACKAGE_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from dasonet.model.decoding import local_peak_suppression
from dasonet.training import (
    FruitMPSynTrainAdapter,
    collate_synthetic,
    seed_worker,
)


def parse_args():
    p = argparse.ArgumentParser(
        "DASO-Net Phase 5C counterfactual-center diagnostic"
    )

    p.add_argument("--project-root", required=True)
    p.add_argument("--checkpoint", required=True)
    p.add_argument("--evaluator", required=True)
    p.add_argument("--output-dir", required=True)

    p.add_argument(
        "--split",
        choices=["val", "test"],
        default="val",
    )

    p.add_argument(
        "--max-scenes",
        type=int,
        default=200,
    )

    p.add_argument(
        "--seed",
        type=int,
        default=20260917,
    )

    p.add_argument(
        "--num-workers",
        type=int,
        default=2,
    )

    p.add_argument(
        "--stride",
        type=int,
        default=4,
    )

    p.add_argument(
        "--peak-kernel",
        type=int,
        default=3,
    )

    p.add_argument(
        "--window-radius-feature",
        type=int,
        default=2,
        help=(
            "Radius around GT center in stride-4 feature cells. "
            "2 cells corresponds to roughly 8 px."
        ),
    )

    return p.parse_args()


def load_module(path: Path):
    spec = importlib.util.spec_from_file_location(
        "phase5c_counterfactual_eval",
        path,
    )

    if spec is None or spec.loader is None:
        raise RuntimeError(
            f"Unable to import:\n{path}"
        )

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module


def stats(values):
    x = np.asarray(
        values,
        dtype=np.float64,
    )

    x = x[
        np.isfinite(x)
    ]

    if x.size == 0:
        return {
            "n": 0,
            "mean": None,
            "median": None,
            "p10": None,
            "p25": None,
            "p75": None,
            "p90": None,
        }

    return {
        "n":
            int(x.size),

        "mean":
            float(np.mean(x)),

        "median":
            float(np.median(x)),

        "p10":
            float(np.quantile(x, 0.10)),

        "p25":
            float(np.quantile(x, 0.25)),

        "p75":
            float(np.quantile(x, 0.75)),

        "p90":
            float(np.quantile(x, 0.90)),
    }


def fraction(values, predicate):
    x = np.asarray(
        values,
        dtype=np.float64,
    )

    if x.size == 0:
        return 0.0

    return float(
        np.mean(
            predicate(x)
        )
    )


def local_window_max(
    x: torch.Tensor,
    cx: int,
    cy: int,
    radius: int,
) -> float:
    """
    x: [H,W]
    """

    h, w = x.shape

    x1 = max(
        0,
        cx - radius,
    )

    y1 = max(
        0,
        cy - radius,
    )

    x2 = min(
        w,
        cx + radius + 1,
    )

    y2 = min(
        h,
        cy + radius + 1,
    )

    if (
        x2 <= x1
        or y2 <= y1
    ):
        return 0.0

    return float(
        x[
            y1:y2,
            x1:x2,
        ]
        .max()
        .item()
    )


def positive_peak_count(
    peak_map: torch.Tensor,
    threshold: float,
    valid_mask: torch.Tensor | None = None,
) -> int:
    """
    peak_map: [H,W]
    valid_mask: optional [H,W] bool
    """

    selected = (
        peak_map
        >= threshold
    )

    if valid_mask is not None:
        selected = (
            selected
            & valid_mask
        )

    return int(
        selected.sum().item()
    )


def main():
    args = parse_args()

    random.seed(
        args.seed
    )

    np.random.seed(
        args.seed
    )

    torch.manual_seed(
        args.seed
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required."
        )

    device = torch.device(
        "cuda:0"
    )

    torch.cuda.set_device(
        device
    )

    project_root = Path(
        args.project_root
    ).resolve()

    checkpoint_path = Path(
        args.checkpoint
    ).resolve()

    evaluator_path = Path(
        args.evaluator
    ).resolve()

    output_dir = Path(
        args.output_dir
    ).resolve()

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    evaluator = load_module(
        evaluator_path
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    model = evaluator.build_model(
        checkpoint,
        device,
    )

    dataset = FruitMPSynTrainAdapter(
        project_root,
        args.split,
        validate_frozen_hashes=True,
    )

    indices = evaluator.select_indices(
        dataset,
        args.max_scenes,
        args.seed,
    )

    loader_kwargs = {
        "dataset":
            Subset(
                dataset,
                indices,
            ),

        "batch_size":
            1,

        "shuffle":
            False,

        "num_workers":
            max(
                0,
                args.num_workers,
            ),

        "pin_memory":
            True,

        "collate_fn":
            collate_synthetic,

        "worker_init_fn":
            seed_worker,
    }

    if args.num_workers > 0:
        loader_kwargs[
            "prefetch_factor"
        ] = 2

        loader_kwargs[
            "persistent_workers"
        ] = True

    loader = DataLoader(
        **loader_kwargs
    )

    thresholds = [
        0.05,
        0.10,
        0.15,
        0.20,
        0.30,
    ]

    # --------------------------------------------------------
    # Particle-level counterfactual measurements
    # --------------------------------------------------------

    clean_gt_scores = []
    composite_gt_scores = []
    exact_score_lifts = []

    clean_window_peaks = []
    composite_window_peaks = []
    window_peak_lifts = []

    # --------------------------------------------------------
    # Scene-level background/confounder measurements
    # --------------------------------------------------------

    clean_max_peak = []
    composite_outside_max_peak = []
    composite_effect_max_peak = []

    clean_peak_counts = {
        str(t): []
        for t in thresholds
    }

    composite_outside_peak_counts = {
        str(t): []
        for t in thresholds
    }

    composite_effect_peak_counts = {
        str(t): []
        for t in thresholds
    }

    scene_count = 0
    particle_count = 0

    start = time.time()

    torch.cuda.reset_peak_memory_stats(
        device
    )

    with torch.inference_mode():

        for step, batch in enumerate(
            loader
        ):

            clean = (
                batch[
                    "clean"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
            )

            composite = (
                batch[
                    "composite"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
            )

            effect = (
                batch[
                    "effect_mask"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
                .float()
            )

            target = (
                batch[
                    "targets"
                ][0]
            )

            gt_boxes = (
                target[
                    "boxes"
                ]
                .to(
                    device,
                    non_blocking=True,
                )
                .float()
            )

            with torch.amp.autocast(
                "cuda",
                enabled=True,
                dtype=torch.float16,
            ):
                composite_out = model(
                    composite
                )

                clean_out = model(
                    clean
                )

            composite_prob = torch.sigmoid(
                composite_out[
                    "center_logits"
                ]
            ).float()

            clean_prob = torch.sigmoid(
                clean_out[
                    "center_logits"
                ]
            ).float()

            composite_peaks = (
                local_peak_suppression(
                    composite_prob,
                    kernel=
                        args.peak_kernel,
                )
            )

            clean_peaks = (
                local_peak_suppression(
                    clean_prob,
                    kernel=
                        args.peak_kernel,
                )
            )

            fh = int(
                composite_prob.shape[-2]
            )

            fw = int(
                composite_prob.shape[-1]
            )

            # Counterfactual effect support at stride 4.
            # Max pooling ensures a feature cell is declared affected
            # if ANY native-resolution effect pixel falls inside it.
            if (
                effect.shape[-2]
                % args.stride
                == 0
                and
                effect.shape[-1]
                % args.stride
                == 0
            ):
                effect_small = F.max_pool2d(
                    effect,
                    kernel_size=
                        args.stride,
                    stride=
                        args.stride,
                )
            else:
                effect_small = F.adaptive_max_pool2d(
                    effect,
                    output_size=(
                        fh,
                        fw,
                    ),
                )

            effect_small = (
                effect_small[
                    0,
                    0,
                ]
                > 0
            )

            outside_small = (
                ~effect_small
            )

            clean_peak_map = (
                clean_peaks[
                    0,
                    0,
                ]
            )

            composite_peak_map = (
                composite_peaks[
                    0,
                    0,
                ]
            )

            # ------------------------------------------------
            # Whole-scene counterfactual confounder behavior
            # ------------------------------------------------

            clean_max_peak.append(
                float(
                    clean_peak_map
                    .max()
                    .item()
                )
            )

            if outside_small.any():
                composite_outside_max_peak.append(
                    float(
                        composite_peak_map[
                            outside_small
                        ]
                        .max()
                        .item()
                    )
                )
            else:
                composite_outside_max_peak.append(
                    0.0
                )

            if effect_small.any():
                composite_effect_max_peak.append(
                    float(
                        composite_peak_map[
                            effect_small
                        ]
                        .max()
                        .item()
                    )
                )
            else:
                composite_effect_max_peak.append(
                    0.0
                )

            for threshold in thresholds:
                key = str(
                    threshold
                )

                clean_peak_counts[
                    key
                ].append(
                    positive_peak_count(
                        clean_peak_map,
                        threshold,
                    )
                )

                composite_outside_peak_counts[
                    key
                ].append(
                    positive_peak_count(
                        composite_peak_map,
                        threshold,
                        outside_small,
                    )
                )

                composite_effect_peak_counts[
                    key
                ].append(
                    positive_peak_count(
                        composite_peak_map,
                        threshold,
                        effect_small,
                    )
                )

            # ------------------------------------------------
            # Particle-level clean-vs-composite response
            # ------------------------------------------------

            gt_count = int(
                gt_boxes.shape[0]
            )

            scene_count += 1
            particle_count += gt_count

            for j in range(
                gt_count
            ):

                x1, y1, x2, y2 = (
                    gt_boxes[j]
                )

                cx = (
                    0.5
                    * (
                        x1
                        + x2
                    )
                    / float(
                        args.stride
                    )
                )

                cy = (
                    0.5
                    * (
                        y1
                        + y2
                    )
                    / float(
                        args.stride
                    )
                )

                ix = int(
                    torch.round(
                        cx
                    ).item()
                )

                iy = int(
                    torch.round(
                        cy
                    ).item()
                )

                ix = max(
                    0,
                    min(
                        fw - 1,
                        ix,
                    ),
                )

                iy = max(
                    0,
                    min(
                        fh - 1,
                        iy,
                    ),
                )

                c_clean = float(
                    clean_prob[
                        0,
                        0,
                        iy,
                        ix,
                    ].item()
                )

                c_comp = float(
                    composite_prob[
                        0,
                        0,
                        iy,
                        ix,
                    ].item()
                )

                clean_gt_scores.append(
                    c_clean
                )

                composite_gt_scores.append(
                    c_comp
                )

                exact_score_lifts.append(
                    c_comp
                    - c_clean
                )

                clean_local = local_window_max(
                    clean_peak_map,
                    ix,
                    iy,
                    args.window_radius_feature,
                )

                composite_local = local_window_max(
                    composite_peak_map,
                    ix,
                    iy,
                    args.window_radius_feature,
                )

                clean_window_peaks.append(
                    clean_local
                )

                composite_window_peaks.append(
                    composite_local
                )

                window_peak_lifts.append(
                    composite_local
                    - clean_local
                )

            if (
                step % 20 == 0
                or step + 1
                == len(loader)
            ):
                print(
                    f"counterfactual "
                    f"{step + 1:04d}/"
                    f"{len(loader):04d} "
                    f"scene="
                    f"{batch['scene_id'][0]} "
                    f"gt={gt_count} "
                    f"elapsed="
                    f"{(time.time() - start) / 60.0:.1f}m"
                )

    elapsed = (
        time.time()
        - start
    )

    summary = {
        "phase":
            "5C",

        "diagnostic":
            "counterfactual_center_response",

        "paper_result":
            False,

        "checkpoint":
            str(
                checkpoint_path
            ),

        "checkpoint_optimizer_step":
            int(
                checkpoint.get(
                    "optimizer_step",
                    -1,
                )
            ),

        "scenes":
            scene_count,

        "particles":
            particle_count,

        "exact_gt_cell": {
            "clean_score_stats":
                stats(
                    clean_gt_scores
                ),

            "composite_score_stats":
                stats(
                    composite_gt_scores
                ),

            "composite_minus_clean_stats":
                stats(
                    exact_score_lifts
                ),

            "fraction_composite_gt_clean":
                fraction(
                    exact_score_lifts,
                    lambda x:
                        x > 0.0,
                ),

            "fraction_lift_ge_0_01":
                fraction(
                    exact_score_lifts,
                    lambda x:
                        x >= 0.01,
                ),

            "fraction_lift_ge_0_05":
                fraction(
                    exact_score_lifts,
                    lambda x:
                        x >= 0.05,
                ),
        },

        "local_window_peak": {
            "window_radius_feature":
                args.window_radius_feature,

            "approx_window_radius_px":
                (
                    args.window_radius_feature
                    * args.stride
                ),

            "clean_peak_stats":
                stats(
                    clean_window_peaks
                ),

            "composite_peak_stats":
                stats(
                    composite_window_peaks
                ),

            "composite_minus_clean_stats":
                stats(
                    window_peak_lifts
                ),

            "fraction_composite_gt_clean":
                fraction(
                    window_peak_lifts,
                    lambda x:
                        x > 0.0,
                ),

            "fraction_lift_ge_0_01":
                fraction(
                    window_peak_lifts,
                    lambda x:
                        x >= 0.01,
                ),

            "fraction_lift_ge_0_05":
                fraction(
                    window_peak_lifts,
                    lambda x:
                        x >= 0.05,
                ),
        },

        "scene_level": {
            "clean_max_peak_stats":
                stats(
                    clean_max_peak
                ),

            "composite_outside_effect_max_peak_stats":
                stats(
                    composite_outside_max_peak
                ),

            "composite_effect_max_peak_stats":
                stats(
                    composite_effect_max_peak
                ),

            "clean_mean_peak_count_by_threshold": {
                key:
                    float(
                        np.mean(
                            values
                        )
                    )
                for key, values
                in clean_peak_counts.items()
            },

            "composite_outside_effect_mean_peak_count_by_threshold": {
                key:
                    float(
                        np.mean(
                            values
                        )
                    )
                for key, values
                in composite_outside_peak_counts.items()
            },

            "composite_effect_mean_peak_count_by_threshold": {
                key:
                    float(
                        np.mean(
                            values
                        )
                    )
                for key, values
                in composite_effect_peak_counts.items()
            },
        },

        "runtime": {
            "minutes":
                elapsed / 60.0,

            "peak_reserved_vram_gb":
                float(
                    torch.cuda
                    .max_memory_reserved(
                        device
                    )
                    / (1024 ** 3)
                ),
        },
    }

    output_json = (
        output_dir
        / "counterfactual_center_diagnostic.json"
    )

    output_json.write_text(
        json.dumps(
            summary,
            indent=2,
        ),
        encoding="utf-8",
    )

    exact = summary[
        "exact_gt_cell"
    ]

    local = summary[
        "local_window_peak"
    ]

    scene = summary[
        "scene_level"
    ]

    print()
    print(
        "============================================================"
    )
    print(
        "PHASE 5C — COUNTERFACTUAL CENTER DIAGNOSTIC"
    )
    print(
        "============================================================"
    )

    print(
        "Scenes:",
        scene_count,
    )

    print(
        "GT particles:",
        particle_count,
    )

    print()
    print(
        "--- EXACT GT CELL: CLEAN vs COMPOSITE ---"
    )

    print(
        "Clean score mean:",
        f"{exact['clean_score_stats']['mean']:.4f}",
    )

    print(
        "Composite score mean:",
        f"{exact['composite_score_stats']['mean']:.4f}",
    )

    print(
        "Score lift mean:",
        f"{exact['composite_minus_clean_stats']['mean']:+.4f}",
    )

    print(
        "Score lift median:",
        f"{exact['composite_minus_clean_stats']['median']:+.4f}",
    )

    print(
        "Composite > clean:",
        f"{exact['fraction_composite_gt_clean']:.4f}",
    )

    print(
        "Lift >= 0.01:",
        f"{exact['fraction_lift_ge_0_01']:.4f}",
    )

    print(
        "Lift >= 0.05:",
        f"{exact['fraction_lift_ge_0_05']:.4f}",
    )

    print()
    print(
        "--- +/- 8 PX LOCAL PEAK WINDOW ---"
    )

    print(
        "Clean local peak mean:",
        f"{local['clean_peak_stats']['mean']:.4f}",
    )

    print(
        "Composite local peak mean:",
        f"{local['composite_peak_stats']['mean']:.4f}",
    )

    print(
        "Local peak lift mean:",
        f"{local['composite_minus_clean_stats']['mean']:+.4f}",
    )

    print(
        "Local peak lift median:",
        f"{local['composite_minus_clean_stats']['median']:+.4f}",
    )

    print(
        "Composite > clean:",
        f"{local['fraction_composite_gt_clean']:.4f}",
    )

    print(
        "Lift >= 0.01:",
        f"{local['fraction_lift_ge_0_01']:.4f}",
    )

    print(
        "Lift >= 0.05:",
        f"{local['fraction_lift_ge_0_05']:.4f}",
    )

    print()
    print(
        "--- SCENE-LEVEL CONFOUNDER RESPONSE ---"
    )

    print(
        "Clean max peak mean:",
        f"{scene['clean_max_peak_stats']['mean']:.4f}",
    )

    print(
        "Composite outside-effect max peak mean:",
        f"{scene['composite_outside_effect_max_peak_stats']['mean']:.4f}",
    )

    print(
        "Composite effect-region max peak mean:",
        f"{scene['composite_effect_max_peak_stats']['mean']:.4f}",
    )

    print()

    print(
        f"{'Threshold':<12}"
        f"{'Clean peaks/img':>18}"
        f"{'Comp outside/img':>20}"
        f"{'Comp effect/img':>18}"
    )

    print(
        "-" * 68
    )

    for threshold in thresholds:
        key = str(
            threshold
        )

        print(
            f"{threshold:<12.2f}"
            f"{scene['clean_mean_peak_count_by_threshold'][key]:>18.2f}"
            f"{scene['composite_outside_effect_mean_peak_count_by_threshold'][key]:>20.2f}"
            f"{scene['composite_effect_mean_peak_count_by_threshold'][key]:>18.2f}"
        )

    print()
    print(
        "Runtime:",
        f"{elapsed / 60.0:.2f} min",
    )

    print(
        "Peak reserved VRAM:",
        f"{summary['runtime']['peak_reserved_vram_gb']:.2f} GB",
    )

    print(
        "JSON:",
        output_json,
    )

    print()
    print(
        "PHASE 5C COUNTERFACTUAL CENTER DIAGNOSTIC: PASS"
    )


if __name__ == "__main__":
    main()

Writing /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/diagnose_phase5c_counterfactual_center.py


In [37]:
# ============================================================
# PHASE 5C — RUN COUNTERFACTUAL CENTER DIAGNOSTIC
#
# Scientific question:
#
# At true synthetic-particle locations, does inserting the
# particle actually increase the center-head response relative
# to the exact clean counterfactual?
#
# No training.
# Same 200 validation scenes.
# Uses the 1000-step schedule-probe checkpoint.
# ============================================================

import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CHECKPOINT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_scheduleprobe1000_v1/"
    "best.pt"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

DIAGNOSTIC = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "counterfactual_center_diagnostic_v1"
)


for required in (
    PROJECT_ROOT,
    CHECKPOINT,
    EVALUATOR,
    DIAGNOSTIC,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


if OUTPUT_DIR.exists():
    shutil.rmtree(
        OUTPUT_DIR
    )


cmd = [
    sys.executable,
    str(DIAGNOSTIC),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(CHECKPOINT),

    "--evaluator",
    str(EVALUATOR),

    "--output-dir",
    str(OUTPUT_DIR),

    "--split",
    "val",

    "--max-scenes",
    "200",

    "--seed",
    "20260917",

    "--num-workers",
    "2",

    "--stride",
    "4",

    "--peak-kernel",
    "3",

    "--window-radius-feature",
    "2",
]


print(
    "============================================================"
)

print(
    "PHASE 5C — RUNNING COUNTERFACTUAL CENTER DIAGNOSTIC"
)

print(
    "============================================================"
)

print(
    " ".join(cmd)
)

print()


subprocess.run(
    cmd,
    check=True,
)


summary_path = (
    OUTPUT_DIR
    / "counterfactual_center_diagnostic.json"
)


if not summary_path.exists():
    raise RuntimeError(
        "Counterfactual diagnostic JSON missing."
    )


summary = json.loads(
    summary_path.read_text(
        encoding="utf-8"
    )
)


if summary["scenes"] != 200:
    raise RuntimeError(
        "Expected exactly 200 scenes."
    )


if summary["particles"] != 400:
    raise RuntimeError(
        "Expected exactly 400 particles "
        "for the frozen 200-scene diagnostic sample."
    )


print()
print(
    "============================================================"
)

print(
    "COUNTERFACTUAL CENTER FILE CHECK: PASS"
)

print(
    "============================================================"
)

print(
    "Scenes:",
    summary["scenes"],
)

print(
    "Particles:",
    summary["particles"],
)

print(
    "Summary:",
    summary_path,
)

PHASE 5C — RUNNING COUNTERFACTUAL CENTER DIAGNOSTIC
/usr/bin/python3 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/diagnose_phase5c_counterfactual_center.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --checkpoint /kaggle/working/dasonet_phase5b/dasonet_phase5c_scheduleprobe1000_v1/best.pt --evaluator /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/evaluate_phase5c.py --output-dir /kaggle/working/dasonet_phase5c/counterfactual_center_diagnostic_v1 --split val --max-scenes 200 --seed 20260917 --num-workers 2 --stride 4 --peak-kernel 3 --window-radius-feature 2

counterfactual 0001/0200 scene=369b96d7adde8c0026a7 gt=1 elapsed=0.1m
counterfactual 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 elapsed=0.3m
counterfactual 0041/0200 scene=11e8bd15bed065be3400 gt=2 elapsed=0.5m
counterfactual 0061/0200 scene=a3cb9b50dca442394cbe gt=2 elapsed=0.8m
counterfactual 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 elapsed=0.9m
counterfactual 0101/0200

In [38]:
# ============================================================
# PHASE 5C — LOSS-WIRING AUDIT
#
# READ ONLY.
# Does not modify any project file.
#
# Purpose:
#   Inspect the exact CURRENT hardened trainer/loss APIs so the
#   PCC-center experiment can be implemented without guessing.
# ============================================================

from pathlib import Path
import inspect
import re
import sys


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

SRC_ROOT = (
    CODE_ROOT
    / "src"
)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_ROOT),
    )


TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

PCC_FILE = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "losses"
    / "pcc.py"
)

INSTANCE_FILE = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "losses"
    / "instance.py"
)

CRPA_FILE = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "losses"
    / "crpa.py"
)


for path in [
    TRAINER,
    PCC_FILE,
    INSTANCE_FILE,
    CRPA_FILE,
]:
    if not path.exists():
        raise RuntimeError(
            f"Missing file:\n{path}"
        )


print(
    "============================================================"
)
print(
    "PHASE 5C — CURRENT LOSS-WIRING AUDIT"
)
print(
    "============================================================"
)

print()
print("Trainer:")
print(TRAINER)

print()
print("PCC:")
print(PCC_FILE)

print()
print("Instance:")
print(INSTANCE_FILE)

print()
print("CRPA:")
print(CRPA_FILE)


# ------------------------------------------------------------
# Import loss modules and print their public call signatures.
# ------------------------------------------------------------

import dasonet.losses.pcc as pcc_mod
import dasonet.losses.instance as instance_mod
import dasonet.losses.crpa as crpa_mod


def print_callables(title, module):
    print()
    print(
        "------------------------------------------------------------"
    )
    print(title)
    print(
        "------------------------------------------------------------"
    )

    found = False

    for name, obj in inspect.getmembers(module):
        if name.startswith("_"):
            continue

        if inspect.isfunction(obj):
            try:
                sig = inspect.signature(obj)
            except Exception:
                continue

            if (
                "loss" in name.lower()
                or "pcc" in name.lower()
                or "crpa" in name.lower()
                or "instance" in name.lower()
            ):
                print(
                    f"{name}{sig}"
                )
                found = True

        elif inspect.isclass(obj):
            if (
                "loss" in name.lower()
                or "pcc" in name.lower()
                or "crpa" in name.lower()
                or "instance" in name.lower()
            ):
                try:
                    sig = inspect.signature(obj)
                except Exception:
                    sig = "(signature unavailable)"

                print(
                    f"{name}{sig}"
                )
                found = True

    if not found:
        print(
            "(No matching public callables found)"
        )


print_callables(
    "PCC MODULE CALLABLES",
    pcc_mod,
)

print_callables(
    "INSTANCE MODULE CALLABLES",
    instance_mod,
)

print_callables(
    "CRPA MODULE CALLABLES",
    crpa_mod,
)


# ------------------------------------------------------------
# Print COMPLETE pcc.py because this is the main file that may
# receive a new auxiliary loss.
# ------------------------------------------------------------

print()
print()
print(
    "============================================================"
)
print(
    "COMPLETE CURRENT pcc.py"
)
print(
    "============================================================"
)

pcc_text = PCC_FILE.read_text(
    encoding="utf-8"
)

print(pcc_text)


# ------------------------------------------------------------
# Print only the relevant TRAINER regions surrounding the
# model forwards and all loss calculations.
#
# Large context windows are intentionally used so I can later
# provide a COMPLETE replacement safely.
# ------------------------------------------------------------

trainer_lines = TRAINER.read_text(
    encoding="utf-8"
).splitlines()


search_patterns = [
    r"clean",
    r"composite",
    r"instance",
    r"pcc",
    r"crpa",
    r"total_loss",
    r"loss_total",
    r"backward",
]


hits = []

for i, line in enumerate(
    trainer_lines
):
    lower = line.lower()

    if any(
        re.search(
            pattern,
            lower,
        )
        for pattern
        in search_patterns
    ):
        hits.append(i)


# Merge hit neighborhoods so output is readable.
ranges = []

for hit in hits:
    start = max(
        0,
        hit - 18,
    )

    end = min(
        len(trainer_lines),
        hit + 30,
    )

    if (
        not ranges
        or start
        > ranges[-1][1] + 1
    ):
        ranges.append(
            [start, end]
        )
    else:
        ranges[-1][1] = max(
            ranges[-1][1],
            end,
        )


print()
print()
print(
    "============================================================"
)
print(
    "CURRENT TRAINER LOSS/FORWARD REGIONS"
)
print(
    "============================================================"
)


for start, end in ranges:

    print()
    print(
        "------------------------------------------------------------"
    )
    print(
        f"LINES {start + 1}–{end}"
    )
    print(
        "------------------------------------------------------------"
    )

    for idx in range(
        start,
        end,
    ):
        print(
            f"{idx + 1:04d}: "
            f"{trainer_lines[idx]}"
        )


# ------------------------------------------------------------
# Also show the model forward API and output keys from the
# checkpoint/model package definitions when available.
# ------------------------------------------------------------

from dasonet.model import DASONet


print()
print()
print(
    "============================================================"
)
print(
    "MODEL FORWARD SIGNATURE"
)
print(
    "============================================================"
)

print(
    inspect.signature(
        DASONet.forward
    )
)


print()
print(
    "============================================================"
)
print(
    "LOSS-WIRING AUDIT: PASS"
)
print(
    "============================================================"
)

print(
    "No files were modified."
)

PHASE 5C — CURRENT LOSS-WIRING AUDIT

Trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/src/dasonet/losses/pcc.py

Instance:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/src/dasonet/losses/instance.py

CRPA:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/src/dasonet/losses/crpa.py

------------------------------------------------------------
PCC MODULE CALLABLES
------------------------------------------------------------
PCCLoss(strides: 'dict[str, int] | None' = None, guard_feature_pixels: 'dict[str, int] | None' = None, outside_weight: 'float' = 0.5, particle_weight: 'float' = 0.25) -> 'None'

------------------------------------------------------------
INSTANCE MODULE CALLABLES
------------------------------------------------------------
InstanceSegLoss(stride: 'int' = 4, center_weight: 'float' = 1.0, size_weight: 'float' = 0.1, offset_weight: 'float' = 1.0, mask_bce_weight: 'float' = 1.0, mas

In [41]:
# ============================================================
# PHASE 5C — TRAIN BUNDLE FORWARD AUDIT
#
# READ ONLY. NO FILES ARE MODIFIED.
#
# Purpose:
#   Determine exactly how DASONetTrainBundle performs:
#     - clean forward
#     - composite forward
#     - anchor forwards
#     - variant forward
#
# Then we can implement PCC-center without guessing.
# ============================================================

from pathlib import Path
import inspect
import sys


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

SRC_ROOT = (
    CODE_ROOT
    / "src"
)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_ROOT),
    )


from dasonet.training import DASONetTrainBundle
from dasonet.model import DASONet


print(
    "============================================================"
)

print(
    "PHASE 5C — DASONetTrainBundle FORWARD AUDIT"
)

print(
    "============================================================"
)


print()
print(
    "DASONetTrainBundle class:"
)

print(
    inspect.getsource(
        DASONetTrainBundle
    )
)


print()
print(
    "============================================================"
)

print(
    "DASONet PUBLIC/RELEVANT METHODS"
)

print(
    "============================================================"
)


for name in [
    "forward",
    "forward_features",
    "extract_features",
    "encode",
]:

    if hasattr(
        DASONet,
        name,
    ):

        obj = getattr(
            DASONet,
            name,
        )

        print()
        print(
            "------------------------------------------------------------"
        )

        print(
            f"DASONet.{name}"
        )

        print(
            "------------------------------------------------------------"
        )

        try:
            print(
                inspect.signature(
                    obj
                )
            )
        except Exception as exc:
            print(
                "Signature unavailable:",
                exc,
            )

        try:
            print(
                inspect.getsource(
                    obj
                )
            )
        except Exception as exc:
            print(
                "Source unavailable:",
                exc,
            )


print()
print(
    "============================================================"
)

print(
    "DASONet METHOD NAMES"
)

print(
    "============================================================"
)


methods = []

for name, obj in inspect.getmembers(
    DASONet
):
    if (
        callable(obj)
        and not name.startswith(
            "__"
        )
    ):
        methods.append(
            name
        )


print(
    methods
)


print()
print(
    "============================================================"
)

print(
    "TRAIN BUNDLE FORWARD AUDIT: PASS"
)

print(
    "============================================================"
)

print(
    "No files were modified."
)

PHASE 5C — DASONetTrainBundle FORWARD AUDIT

DASONetTrainBundle class:
class DASONetTrainBundle(nn.Module):
    """
    One DDP-visible forward that covers all trainable parameter families.

    This avoids fragile multi-forward/unused-parameter interactions in DDP.
    """

    def __init__(self, core: DASONet) -> None:
        super().__init__()
        self.core = core

    def forward(
        self,
        clean: torch.Tensor,
        composite: torch.Tensor,
        real_positive: torch.Tensor,
        real_negative: torch.Tensor,
        composite_variant: torch.Tensor | None = None,
    ) -> dict:
        composite_out = self.core(composite)
        clean_features = self.core.extract_features(clean)
        pos_out = self.core.encode_anchor(real_positive)
        neg_out = self.core.encode_anchor(real_negative)

        variant_embedding_map = None
        if composite_variant is not None:
            variant_features = self.core.extract_features(composite_variant)
            

In [42]:
%%writefile /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5c_pcc_center.py
from __future__ import annotations

import argparse
import importlib.util
import json
import sys
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F


# ============================================================
# DASO-Net Phase 5C PCC-Center trainer v2
#
# COMPLETE replacement.
#
# Key correction relative to failed v1:
#   DASONetTrainBundle obtains clean features via:
#
#       self.core.extract_features(clean)
#
#   rather than:
#
#       self.core(clean)
#
# Therefore a forward hook on the whole core could not capture
# clean center logits.
#
# v2 reuses the already-computed clean features and executes:
#
#       clean_pred = self.core.pred_tower(clean_features["s4"])
#       clean_center_logits = self.core.center_head(clean_pred)
#
# This adds NO second clean backbone/FPN pass.
#
# Architecture parameters: unchanged.
# Renderer/data: unchanged.
# Instance loss: unchanged.
# Original PCC: unchanged.
# CRPA: unchanged.
#
# Engineering experiment only; not manuscript performance.
# ============================================================


THIS_FILE = Path(__file__).resolve()
PACKAGE_ROOT = THIS_FILE.parents[1]
SRC_ROOT = PACKAGE_ROOT / "src"
BASE_TRAINER = THIS_FILE.parent / "train_phase5b.py"


if str(SRC_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_ROOT),
    )


if not BASE_TRAINER.exists():
    raise FileNotFoundError(
        f"Base trainer not found:\n{BASE_TRAINER}"
    )


# ============================================================
# Import current hardened Phase-5B trainer as a module.
# ============================================================

def load_base_trainer():
    spec = importlib.util.spec_from_file_location(
        "dasonet_phase5b_base_trainer",
        BASE_TRAINER,
    )

    if (
        spec is None
        or spec.loader is None
    ):
        raise RuntimeError(
            "Could not import hardened Phase-5B trainer."
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


base = load_base_trainer()


# ============================================================
# Read PCC-center configuration.
# ============================================================

def get_config_path_from_argv() -> Path:
    parser = argparse.ArgumentParser(
        add_help=False
    )

    parser.add_argument(
        "--config",
        required=True,
    )

    known, _ = parser.parse_known_args()

    path = Path(
        known.config
    ).resolve()

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    return path


CONFIG_PATH = get_config_path_from_argv()


FULL_CONFIG = json.loads(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)


PCC_CENTER_CFG = (
    FULL_CONFIG
    .get(
        "loss",
        {},
    )
    .get(
        "pcc_center",
        {},
    )
)


if not PCC_CENTER_CFG:
    raise RuntimeError(
        "Configuration is missing loss.pcc_center."
    )


if not bool(
    PCC_CENTER_CFG.get(
        "enabled",
        False,
    )
):
    raise RuntimeError(
        "loss.pcc_center.enabled must be true."
    )


# ============================================================
# PCC-CENTER AUXILIARY LOSS
# ============================================================

class PCCCenterAuxLoss(nn.Module):
    """
    Detection-level counterfactual PCC auxiliary.

    The Phase-5C diagnostic established that synthetic particles usually
    increase the center-head response relative to their exact clean
    counterfactual, but natural fruit texture still produces stronger
    competing center peaks.

    This auxiliary therefore optimizes four related objectives:

    1. Counterfactual particle lift
       Composite center logits at GT particle centers should exceed the
       corresponding clean-image logits.

    2. Particle/background ranking
       Composite GT center logits should outrank hard composite
       outside-effect center responses.

    3. Clean hard-negative suppression
       The strongest center responses in the exact clean fruit image
       should be suppressed.

    4. Composite outside-effect hard-negative suppression
       Strong responses outside the renderer-declared effect
       neighborhood should be suppressed.

    All calculations are performed in FP32 even under AMP.
    """

    def __init__(
        self,
        stride: int = 4,
        guard_feature_pixels: int = 2,
        hard_negative_topk: int = 64,
        lift_margin_logit: float = 0.50,
        rank_margin_logit: float = 1.00,
        lift_weight: float = 0.25,
        rank_weight: float = 0.25,
        clean_hard_negative_weight: float = 0.10,
        composite_outside_hard_negative_weight: float = 0.10,
    ) -> None:
        super().__init__()

        self.stride = int(
            stride
        )

        self.guard_feature_pixels = int(
            guard_feature_pixels
        )

        self.hard_negative_topk = int(
            hard_negative_topk
        )

        self.lift_margin_logit = float(
            lift_margin_logit
        )

        self.rank_margin_logit = float(
            rank_margin_logit
        )

        self.lift_weight = float(
            lift_weight
        )

        self.rank_weight = float(
            rank_weight
        )

        self.clean_hard_negative_weight = float(
            clean_hard_negative_weight
        )

        self.composite_outside_hard_negative_weight = float(
            composite_outside_hard_negative_weight
        )


    @staticmethod
    def _hard_topk_mean(
        values: torch.Tensor,
        topk: int,
    ) -> torch.Tensor:

        values = (
            values
            .float()
            .reshape(-1)
        )

        if values.numel() == 0:
            return (
                values.sum()
                * 0.0
            )

        k = min(
            int(topk),
            int(
                values.numel()
            ),
        )

        hard = torch.topk(
            values,
            k=k,
            largest=True,
            sorted=False,
        ).values

        return hard.mean()


    @staticmethod
    def _hard_negative_bce(
        values: torch.Tensor,
        topk: int,
    ) -> torch.Tensor:
        """
        BCEWithLogits(logit, target=0) for the hardest negatives.

        softplus(logit) == BCEWithLogits(logit, zero).
        """

        values = (
            values
            .float()
            .reshape(-1)
        )

        if values.numel() == 0:
            return (
                values.sum()
                * 0.0
            )

        k = min(
            int(topk),
            int(
                values.numel()
            ),
        )

        hard = torch.topk(
            values,
            k=k,
            largest=True,
            sorted=False,
        ).values

        return (
            F.softplus(
                hard
            )
            .mean()
        )


    def _safe_outside_mask(
        self,
        effect_mask: torch.Tensor,
        spatial_size: tuple[int, int],
    ) -> torch.Tensor:

        if effect_mask.ndim == 3:
            effect_mask = (
                effect_mask
                .unsqueeze(1)
            )

        if effect_mask.ndim != 4:
            raise ValueError(
                "Expected effect mask [B,1,H,W] "
                f"or [B,H,W], got {effect_mask.shape}"
            )

        # Mark a feature location as affected whenever any native
        # effect pixel contributes to that feature cell.
        effect = F.adaptive_max_pool2d(
            effect_mask.float(),
            output_size=spatial_size,
        )

        guard = int(
            self.guard_feature_pixels
        )

        if guard > 0:
            kernel = (
                2 * guard
                + 1
            )

            effect = F.max_pool2d(
                effect,
                kernel_size=kernel,
                stride=1,
                padding=guard,
            )

        safe = (
            effect
            <= 0.0
        )

        return safe.detach()


    def forward(
        self,
        clean_center_logits: torch.Tensor,
        composite_center_logits: torch.Tensor,
        targets: list[dict[str, torch.Tensor]],
        effect_mask: torch.Tensor,
    ) -> dict[str, torch.Tensor]:

        if (
            clean_center_logits.shape
            != composite_center_logits.shape
        ):
            raise ValueError(
                "PCC-center clean/composite shape mismatch: "
                f"{clean_center_logits.shape} vs "
                f"{composite_center_logits.shape}"
            )

        if (
            composite_center_logits.ndim != 4
            or
            composite_center_logits.shape[1] != 1
        ):
            raise ValueError(
                "PCC-center expected logits [B,1,H,W], got "
                f"{composite_center_logits.shape}"
            )

        batch_size = int(
            composite_center_logits.shape[0]
        )

        if len(targets) != batch_size:
            raise ValueError(
                "PCC-center target batch mismatch: "
                f"{len(targets)} targets for batch {batch_size}"
            )

        device = (
            composite_center_logits.device
        )

        feature_h = int(
            composite_center_logits.shape[-2]
        )

        feature_w = int(
            composite_center_logits.shape[-1]
        )

        safe_outside = (
            self._safe_outside_mask(
                effect_mask,
                (
                    feature_h,
                    feature_w,
                ),
            )
            .to(
                device=device
            )
        )

        lift_losses = []
        ranking_losses = []

        clean_negative_losses = []
        outside_negative_losses = []

        positive_count = 0


        for b in range(
            batch_size
        ):

            clean_logits = (
                clean_center_logits[
                    b,
                    0,
                ]
                .float()
            )

            composite_logits = (
                composite_center_logits[
                    b,
                    0,
                ]
                .float()
            )

            # ------------------------------------------------
            # Clean counterfactual has zero synthetic particles.
            # Suppress its hardest center responses.
            # ------------------------------------------------

            clean_negative_losses.append(
                self._hard_negative_bce(
                    clean_logits,
                    self.hard_negative_topk,
                )
            )


            # ------------------------------------------------
            # Composite hard negatives only from safely outside
            # the declared renderer effect neighborhood.
            # ------------------------------------------------

            safe_b = (
                safe_outside[
                    b,
                    0,
                ]
            )

            outside_values = (
                composite_logits[
                    safe_b
                ]
            )

            outside_negative_losses.append(
                self._hard_negative_bce(
                    outside_values,
                    self.hard_negative_topk,
                )
            )

            hard_background_logit = (
                self._hard_topk_mean(
                    outside_values,
                    self.hard_negative_topk,
                )
            )


            # ------------------------------------------------
            # GT particle-center locations.
            #
            # Must follow frozen Phase-5B convention:
            #
            #   grid cell = round(center / stride)
            #   offset    = center/stride - round(...)
            # ------------------------------------------------

            boxes = (
                targets[b][
                    "boxes"
                ]
                .to(
                    device=device,
                    dtype=torch.float32,
                )
            )

            if boxes.numel() == 0:
                continue

            if (
                boxes.ndim != 2
                or
                boxes.shape[1] != 4
            ):
                raise ValueError(
                    "Expected target boxes [N,4], got "
                    f"{boxes.shape}"
                )

            center_x = (
                (
                    boxes[:, 0]
                    + boxes[:, 2]
                )
                * 0.5
                / float(
                    self.stride
                )
            )

            center_y = (
                (
                    boxes[:, 1]
                    + boxes[:, 3]
                )
                * 0.5
                / float(
                    self.stride
                )
            )

            ix = (
                torch.round(
                    center_x
                )
                .long()
                .clamp(
                    min=0,
                    max=feature_w - 1,
                )
            )

            iy = (
                torch.round(
                    center_y
                )
                .long()
                .clamp(
                    min=0,
                    max=feature_h - 1,
                )
            )

            composite_positive = (
                composite_logits[
                    iy,
                    ix,
                ]
            )

            clean_positive = (
                clean_logits[
                    iy,
                    ix,
                ]
            )

            positive_count += int(
                composite_positive.numel()
            )


            # ------------------------------------------------
            # Counterfactual logit lift.
            #
            # Encourage:
            #
            # composite_logit - clean_logit >= lift_margin
            # ------------------------------------------------

            logit_lift = (
                composite_positive
                - clean_positive
            )

            lift_losses.append(
                F.softplus(
                    self.lift_margin_logit
                    - logit_lift
                ).mean()
            )


            # ------------------------------------------------
            # Particle-vs-hard-background ranking.
            #
            # Encourage:
            #
            # composite_positive
            #   >= hard_background + rank_margin
            # ------------------------------------------------

            ranking_losses.append(
                F.softplus(
                    self.rank_margin_logit
                    + hard_background_logit
                    - composite_positive
                ).mean()
            )


        reference = (
            composite_center_logits
            .float()
            .sum()
            * 0.0
        )


        if lift_losses:
            lift = (
                torch.stack(
                    lift_losses
                )
                .mean()
            )
        else:
            lift = reference


        if ranking_losses:
            ranking = (
                torch.stack(
                    ranking_losses
                )
                .mean()
            )
        else:
            ranking = reference


        if clean_negative_losses:
            clean_hard_negative = (
                torch.stack(
                    clean_negative_losses
                )
                .mean()
            )
        else:
            clean_hard_negative = reference


        if outside_negative_losses:
            outside_hard_negative = (
                torch.stack(
                    outside_negative_losses
                )
                .mean()
            )
        else:
            outside_hard_negative = reference


        total = (
            self.lift_weight
            * lift

            + self.rank_weight
            * ranking

            + self.clean_hard_negative_weight
            * clean_hard_negative

            + self.composite_outside_hard_negative_weight
            * outside_hard_negative
        )


        return {
            "total":
                total,

            "lift":
                lift,

            "ranking":
                ranking,

            "clean_hard_negative":
                clean_hard_negative,

            "composite_outside_hard_negative":
                outside_hard_negative,

            "positive_count":
                torch.tensor(
                    float(
                        positive_count
                    ),
                    dtype=torch.float32,
                    device=device,
                ),
        }


# ============================================================
# Instantiate stateless auxiliary loss.
# ============================================================

PCC_CENTER_LOSS = PCCCenterAuxLoss(
    stride=int(
        PCC_CENTER_CFG.get(
            "stride",
            4,
        )
    ),

    guard_feature_pixels=int(
        PCC_CENTER_CFG.get(
            "guard_feature_pixels",
            2,
        )
    ),

    hard_negative_topk=int(
        PCC_CENTER_CFG.get(
            "hard_negative_topk",
            64,
        )
    ),

    lift_margin_logit=float(
        PCC_CENTER_CFG.get(
            "lift_margin_logit",
            0.50,
        )
    ),

    rank_margin_logit=float(
        PCC_CENTER_CFG.get(
            "rank_margin_logit",
            1.00,
        )
    ),

    lift_weight=float(
        PCC_CENTER_CFG.get(
            "lift_weight",
            0.25,
        )
    ),

    rank_weight=float(
        PCC_CENTER_CFG.get(
            "rank_weight",
            0.25,
        )
    ),

    clean_hard_negative_weight=float(
        PCC_CENTER_CFG.get(
            "clean_hard_negative_weight",
            0.10,
        )
    ),

    composite_outside_hard_negative_weight=float(
        PCC_CENTER_CFG.get(
            "composite_outside_hard_negative_weight",
            0.10,
        )
    ),
)


# ============================================================
# COMPLETE replacement of DASONetTrainBundle.forward behavior.
#
# Derived directly from the audited current implementation.
#
# Important:
#   clean_features are computed ONCE.
#
# The additional center branch reuses clean_features["s4"].
# There is no second backbone/FPN extraction.
# ============================================================

def phase5c_bundle_forward(
    self,
    clean: torch.Tensor,
    composite: torch.Tensor,
    real_positive: torch.Tensor,
    real_negative: torch.Tensor,
    composite_variant: torch.Tensor | None = None,
) -> dict:

    # --------------------------------------------------------
    # Original composite full prediction forward.
    # --------------------------------------------------------

    composite_out = (
        self.core(
            composite
        )
    )


    # --------------------------------------------------------
    # Original PCC clean feature forward.
    # --------------------------------------------------------

    clean_features = (
        self.core.extract_features(
            clean
        )
    )


    # --------------------------------------------------------
    # Phase-5C addition:
    #
    # Produce clean center logits from the SAME clean s4 feature.
    #
    # This repeats only the lightweight prediction tower and center
    # head. It does NOT run stem/s4/s8/s16/FPN again.
    # --------------------------------------------------------

    clean_pred = (
        self.core.pred_tower(
            clean_features[
                "s4"
            ]
        )
    )

    clean_center_logits = (
        self.core.center_head(
            clean_pred
        )
    )


    # --------------------------------------------------------
    # Original real-anchor forwards.
    # --------------------------------------------------------

    pos_out = (
        self.core.encode_anchor(
            real_positive
        )
    )

    neg_out = (
        self.core.encode_anchor(
            real_negative
        )
    )


    # --------------------------------------------------------
    # Original optional appearance-consistency path.
    # --------------------------------------------------------

    variant_embedding_map = None

    if composite_variant is not None:

        variant_features = (
            self.core.extract_features(
                composite_variant
            )
        )

        variant_embedding_map = (
            self.core.embedding_from_features(
                variant_features
            )
        )


    return {
        "composite":
            composite_out,

        "clean_features":
            clean_features,

        "clean_center_logits":
            clean_center_logits,

        "positive_anchor":
            pos_out,

        "negative_anchor":
            neg_out,

        "variant_embedding_map":
            variant_embedding_map,
    }


# Install corrected Phase-5C bundle forward.
base.DASONetTrainBundle.forward = (
    phase5c_bundle_forward
)


# ============================================================
# LOSS COMPOSITION
#
# Preserve original Instance + PCC + CRPA exactly, then add
# PCC-center auxiliary.
# ============================================================

_ORIGINAL_COMPUTE_LOSSES = (
    base.compute_losses
)


def phase5c_compute_losses(
    bundle_out,
    syn_batch,
    inst_loss_fn,
    pcc_loss_fn,
    crpa_loss_fn,
    particle_consistency,
):

    original_total, original_metrics = (
        _ORIGINAL_COMPUTE_LOSSES(
            bundle_out,
            syn_batch,
            inst_loss_fn,
            pcc_loss_fn,
            crpa_loss_fn,
            particle_consistency,
        )
    )


    if (
        "clean_center_logits"
        not in bundle_out
    ):
        raise RuntimeError(
            "Phase-5C bundle output is missing "
            "clean_center_logits."
        )


    center_aux = (
        PCC_CENTER_LOSS(
            clean_center_logits=
                bundle_out[
                    "clean_center_logits"
                ],

            composite_center_logits=
                bundle_out[
                    "composite"
                ][
                    "center_logits"
                ],

            targets=
                syn_batch[
                    "targets"
                ],

            effect_mask=
                syn_batch[
                    "effect_mask"
                ],
        )
    )


    total = (
        original_total
        + center_aux[
            "total"
        ]
    )


    metrics = dict(
        original_metrics
    )


    # Preserve original feature/embedding PCC explicitly.
    metrics[
        "pcc_feature"
    ] = (
        original_metrics[
            "pcc"
        ]
    )


    metrics[
        "pcc_center"
    ] = (
        center_aux[
            "total"
        ]
    )

    metrics[
        "pcc_center_lift"
    ] = (
        center_aux[
            "lift"
        ]
    )

    metrics[
        "pcc_center_ranking"
    ] = (
        center_aux[
            "ranking"
        ]
    )

    metrics[
        "pcc_center_clean_hard_negative"
    ] = (
        center_aux[
            "clean_hard_negative"
        ]
    )

    metrics[
        "pcc_center_outside_hard_negative"
    ] = (
        center_aux[
            "composite_outside_hard_negative"
        ]
    )

    metrics[
        "pcc_center_positive_count"
    ] = (
        center_aux[
            "positive_count"
        ]
    )


    # The hardened Phase-5B trainer already logs and validates
    # metrics["pcc"]. Make that diagnostic include both PCC pieces.
    metrics[
        "pcc"
    ] = (
        original_metrics[
            "pcc"
        ]
        + center_aux[
            "total"
        ]
    )


    metrics[
        "total"
    ] = total


    return (
        total,
        metrics,
    )


base.compute_losses = (
    phase5c_compute_losses
)


# ============================================================
# Preserve Phase-5B AMP engineering decision:
# T4 FP16 initial loss scale = 32.
# ============================================================

_ORIGINAL_GRAD_SCALER = (
    torch.amp.GradScaler
)


class T4StableGradScaler(
    _ORIGINAL_GRAD_SCALER
):

    def __init__(
        self,
        *args,
        **kwargs,
    ):
        kwargs.setdefault(
            "init_scale",
            32.0,
        )

        super().__init__(
            *args,
            **kwargs,
        )


torch.amp.GradScaler = (
    T4StableGradScaler
)


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":

    print(
        "============================================================"
    )

    print(
        "DASO-Net Phase 5C — PCC-CENTER TRAINER v2"
    )

    print(
        "============================================================"
    )

    print(
        "Base trainer:"
    )

    print(
        BASE_TRAINER
    )

    print()

    print(
        "PCC-center configuration:"
    )

    print(
        json.dumps(
            PCC_CENTER_CFG,
            indent=2,
        )
    )

    print()

    print(
        "Clean backbone/FPN extraction count: 1"
    )

    print(
        "Clean pred_tower + center_head:      enabled"
    )

    print(
        "Second clean backbone pass:          NO"
    )

    print(
        "Architecture parameters changed:     NO"
    )

    print(
        "Instance loss changed:               NO"
    )

    print(
        "Original feature PCC changed:        NO"
    )

    print(
        "CRPA changed:                        NO"
    )

    print(
        "Renderer/data changed:               NO"
    )

    print(
        "AMP initial scale:                   32"
    )

    print(
        "============================================================"
    )

    base.main()

Overwriting /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5c_pcc_center.py


In [43]:
# ============================================================
# PHASE 5C PCC-CENTER v2 — T4x2 DDP SMOKE
#
# Verifies:
#   - corrected clean-center path
#   - DDP graph safety
#   - zero-particle scenes
#   - backward
#   - validation
#   - AMP scale 32 / zero skips
#
# No long training here.
# ============================================================

import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5c_pcc_center.py"
)

BASE_CONFIG = Path(
    "/kaggle/working/"
    "phase5c_scheduleprobe1000_v1.json"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_pcccenter_smoke_v2.json"
)

RUN_NAME = (
    "dasonet_phase5c_pcccenter_smoke_v2"
)

OUT = (
    Path(
        "/kaggle/working/"
        "dasonet_phase5b"
    )
    / RUN_NAME
)


for required in (
    TRAINER,
    BASE_CONFIG,
    PROJECT_ROOT,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


# ------------------------------------------------------------
# Verify complete replacement is the corrected v2.
# ------------------------------------------------------------

trainer_text = TRAINER.read_text(
    encoding="utf-8"
)


required_markers = [
    "DASO-Net Phase 5C PCC-Center trainer v2",
    "clean_features =",
    "self.core.extract_features(",
    "clean_pred =",
    "self.core.pred_tower(",
    'clean_features[',
    '"s4"',
    "clean_center_logits =",
    "self.core.center_head(",
    "Second clean backbone pass:",
]


missing = [
    marker
    for marker in required_markers
    if marker not in trainer_text
]


if missing:
    raise RuntimeError(
        "Corrected PCC-center trainer verification failed.\n"
        f"Missing markers: {missing}"
    )


# The failed hook-based implementation must be gone.
forbidden_markers = [
    "register_forward_hook",
    "clean_ptr = int(",
    "PCC-center wrapper failed to capture clean center logits",
]


found_forbidden = [
    marker
    for marker in forbidden_markers
    if marker in trainer_text
]


if found_forbidden:
    raise RuntimeError(
        "Old hook-based PCC-center implementation is still present.\n"
        f"Found: {found_forbidden}"
    )


# Syntax check before launching torchrun.
subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(TRAINER),
    ],
    check=True,
)


# ------------------------------------------------------------
# Build smoke configuration.
# ------------------------------------------------------------

cfg = json.loads(
    BASE_CONFIG.read_text(
        encoding="utf-8"
    )
)


cfg[
    "experiment_name"
] = RUN_NAME


cfg.setdefault(
    "loss",
    {}
)


cfg[
    "loss"
][
    "pcc_center"
] = {
    "enabled":
        True,

    "stride":
        4,

    "guard_feature_pixels":
        2,

    "hard_negative_topk":
        64,

    "lift_margin_logit":
        0.50,

    "rank_margin_logit":
        1.00,

    "lift_weight":
        0.25,

    "rank_weight":
        0.25,

    "clean_hard_negative_weight":
        0.10,

    "composite_outside_hard_negative_weight":
        0.10,
}


CONFIG.write_text(
    json.dumps(
        cfg,
        indent=2,
    ),
    encoding="utf-8",
)


if OUT.exists():
    shutil.rmtree(
        OUT
    )


cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(TRAINER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--smoke",

    "--max-train-steps",
    "12",

    "--max-val-steps",
    "4",

    "--log-every",
    "1",

    "--resume",
    "none",
]


print(
    "============================================================"
)

print(
    "PHASE 5C PCC-CENTER v2 — DDP SMOKE"
)

print(
    "============================================================"
)

print(
    "Corrected clean-feature reuse path: PASS"
)

print(
    "Old hook implementation absent:     PASS"
)

print(
    "Python syntax:                       PASS"
)

print()

print(
    " ".join(cmd)
)

print()


subprocess.run(
    cmd,
    check=True,
)


# ------------------------------------------------------------
# Verify smoke artifacts.
# ------------------------------------------------------------

status_path = (
    OUT
    / "run_status.json"
)

history_path = (
    OUT
    / "history.csv"
)

best_path = (
    OUT
    / "best.pt"
)

last_path = (
    OUT
    / "last.pt"
)


for path in (
    status_path,
    history_path,
    best_path,
    last_path,
):
    if not path.exists():
        raise RuntimeError(
            f"Smoke artifact missing:\n{path}"
        )


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:
    raise RuntimeError(
        f"Unexpected AMP skips: {amp_skips}"
    )


if status.get(
    "status"
) != "smoke_passed":
    raise RuntimeError(
        "Trainer did not report smoke_passed."
    )


print()
print(
    "============================================================"
)

print(
    "PHASE 5C PCC-CENTER v2 DDP SMOKE: PASS"
)

print(
    "============================================================"
)

print(
    "AMP skipped optimizer steps:",
    amp_skips,
)

print(
    "Best validation total loss:",
    status.get(
        "best_val_total_loss"
    ),
)

print(
    "Best checkpoint:",
    best_path,
)

print()
print(
    "Do not launch the 1000-step ablation yet."
)

PHASE 5C PCC-CENTER v2 — DDP SMOKE
Corrected clean-feature reuse path: PASS
Old hook implementation absent:     PASS
Python syntax:                       PASS

torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5c_pcc_center.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_pcccenter_smoke_v2.json --output-root /kaggle/working --smoke --max-train-steps 12 --max-val-steps 4 --log-every 1 --resume none



W0913 10:44:58.147000 1475 torch/distributed/run.py:852] 
W0913 10:44:58.147000 1475 torch/distributed/run.py:852] *****************************************
W0913 10:44:58.147000 1475 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 10:44:58.147000 1475 torch/distributed/run.py:852] *****************************************
[W913 10:44:58.614666411 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — PCC-CENTER TRAINER v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC-center configuration:
{
  "enabled": true,
  "stride": 4,
  "guard_feature_pixels": 2,
  "hard_negative_topk": 64,
  "lift_margin_logit": 0.5,
  "rank_margin_logit": 1.0,
  "lift_weight": 0.25,
  "rank_weight": 0.25,
  "clean_hard_negative_weight": 0.1,
  "composite_outside_hard_negative_weight": 0.1
}

Clean backbone/FPN extraction count: 1
Clean pred_tower + center_head:      enabled
Second clean backbone pass:          NO
Architecture parameters changed:     NO
Instance loss changed:               NO
Original feature PCC changed:        NO
CRPA changed:                        NO
Renderer/data changed:               NO
AMP initial scale:                   32
DASO-Net Phase 5C — PCC-CENTER TRAINER v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC-center configuration:
{
  "enabled": true,
  "stride": 4,
  "gu

[W913 10:45:01.393719717 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 10:45:01.401188237 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_pcccenter_smoke_v2
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     0
Anchor workers/rank:        0
Gradient accumulation:      2
Effective synthetic batch:  4
AMP fp16:                   True
Train steps/epoch/rank:     12
Optimizer steps total:      6
Smoke mode:                 True
epoch=00 step=00000/00012 loss=29.9623 inst=27.1531 pcc=2.2080 crpa=0.6012 lr=6.00e-05 vram_alloc=9.48GB vram_res=9.75GB
epoch=00 step=00001/00012 loss=143.6663 inst=140.7300 pcc=2.2791 crpa=0.6572 lr=6.00e-05 vram_alloc=9.48GB vram_res=9.75GB
epoch=00 step=00002/00012 loss=9.8755 inst=7.7390 pcc=1.3377 crpa=0.798

RuntimeError: Unexpected AMP skips: 1

In [44]:
# ============================================================
# PHASE 5C PCC-CENTER v2
# PRODUCTION-SHAPE AMP/DDP SANITY
#
# Unlike --smoke, this preserves the intended:
#   accumulation = 4
#   workers/GPU = 2
#   anchor workers/GPU = 1
#   effective synthetic batch = 8
#
# Only 40 train steps/rank = 10 intended optimizer updates.
#
# No code modifications.
# ============================================================

import json
import shutil
import subprocess
import time
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5c_pcc_center.py"
)

BASE_CONFIG = Path(
    "/kaggle/working/"
    "phase5c_scheduleprobe1000_v1.json"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_pcccenter_prodshape_sanity_v1.json"
)

RUN_NAME = (
    "dasonet_phase5c_pcccenter_prodshape_sanity_v1"
)

OUT = (
    Path(
        "/kaggle/working/"
        "dasonet_phase5b"
    )
    / RUN_NAME
)


for required in (
    TRAINER,
    BASE_CONFIG,
    PROJECT_ROOT,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


cfg = json.loads(
    BASE_CONFIG.read_text(
        encoding="utf-8"
    )
)

cfg[
    "experiment_name"
] = RUN_NAME


# ------------------------------------------------------------
# Exact intended production-shape hardware/runtime settings.
# ------------------------------------------------------------

cfg["data"][
    "per_rank_synthetic_batch"
] = 1

cfg["data"][
    "per_rank_anchor_batch"
] = 2

cfg["data"][
    "synthetic_workers_per_rank"
] = 2

cfg["data"][
    "anchor_workers_per_rank"
] = 1

cfg["data"][
    "prefetch_factor"
] = 2


cfg["optimization"][
    "gradient_accumulation_steps"
] = 4

cfg["optimization"][
    "amp"
] = True

cfg["optimization"][
    "learning_rate"
] = 3e-4

cfg["optimization"][
    "warmup_optimizer_steps"
] = 50

cfg["optimization"][
    "min_lr_ratio"
] = 1.0


cfg.setdefault(
    "loss",
    {}
)


cfg[
    "loss"
][
    "pcc_center"
] = {
    "enabled":
        True,

    "stride":
        4,

    "guard_feature_pixels":
        2,

    "hard_negative_topk":
        64,

    "lift_margin_logit":
        0.50,

    "rank_margin_logit":
        1.00,

    "lift_weight":
        0.25,

    "rank_weight":
        0.25,

    "clean_hard_negative_weight":
        0.10,

    "composite_outside_hard_negative_weight":
        0.10,
}


CONFIG.write_text(
    json.dumps(
        cfg,
        indent=2,
    ),
    encoding="utf-8",
)


if OUT.exists():
    shutil.rmtree(
        OUT
    )


cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(TRAINER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "1",

    "--max-train-steps",
    "40",

    "--max-val-steps",
    "4",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "5",

    "--resume",
    "none",
]


print(
    "============================================================"
)

print(
    "PHASE 5C PCC-CENTER v2 — PRODUCTION-SHAPE SANITY"
)

print(
    "============================================================"
)

print(
    "Train steps/rank:         40"
)

print(
    "Expected optimizer steps: 10"
)

print(
    "Accumulation:             4"
)

print(
    "Effective batch:          8"
)

print(
    "AMP initial scale:        32"
)

print()

print(
    " ".join(cmd)
)

print()


start = time.time()


subprocess.run(
    cmd,
    check=True,
)


elapsed = (
    time.time()
    - start
)


status_path = (
    OUT
    / "run_status.json"
)

last_path = (
    OUT
    / "last.pt"
)


if not status_path.exists():
    raise RuntimeError(
        "run_status.json was not created."
    )


if not last_path.exists():
    raise RuntimeError(
        "last.pt was not created."
    )


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


checkpoint = __import__(
    "torch"
).load(
    last_path,
    map_location="cpu",
    weights_only=False,
)


scaler_state = checkpoint.get(
    "scaler_state_dict",
    {}
)


print()
print(
    "============================================================"
)

print(
    "PRODUCTION-SHAPE SANITY RESULT"
)

print(
    "============================================================"
)

print(
    f"Wall time:              "
    f"{elapsed / 60.0:.2f} min"
)

print(
    "AMP skipped updates:    ",
    amp_skips,
)

print(
    "Final scaler scale:      ",
    scaler_state.get(
        "scale"
    ),
)

print(
    "Scaler growth tracker:   ",
    scaler_state.get(
        "_growth_tracker"
    ),
)

print(
    "Best validation loss:    ",
    status.get(
        "best_val_total_loss"
    ),
)


if amp_skips != 0:
    print()
    print(
        "RESULT: AMP STABILITY NOT YET ACCEPTED"
    )

    raise RuntimeError(
        "PCC-center still produced an AMP skip "
        "under the actual accumulation-4 regime. "
        "Do not start the 1000-step ablation."
    )


print()
print(
    "========================================"
)

print(
    "PHASE 5C PCC-CENTER PRODUCTION-SHAPE SANITY: PASS"
)

print(
    "========================================"
)

PHASE 5C PCC-CENTER v2 — PRODUCTION-SHAPE SANITY
Train steps/rank:         40
Expected optimizer steps: 10
Accumulation:             4
Effective batch:          8
AMP initial scale:        32

torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5c_pcc_center.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_pcccenter_prodshape_sanity_v1.json --output-root /kaggle/working --epochs 1 --max-train-steps 40 --max-val-steps 4 --accum 4 --num-workers 2 --log-every 5 --resume none



W0913 10:48:24.508000 1518 torch/distributed/run.py:852] 
W0913 10:48:24.508000 1518 torch/distributed/run.py:852] *****************************************
W0913 10:48:24.508000 1518 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 10:48:24.508000 1518 torch/distributed/run.py:852] *****************************************
[W913 10:48:24.961068050 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — PCC-CENTER TRAINER v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC-center configuration:
{
  "enabled": true,
  "stride": 4,
  "guard_feature_pixels": 2,
  "hard_negative_topk": 64,
  "lift_margin_logit": 0.5,
  "rank_margin_logit": 1.0,
  "lift_weight": 0.25,
  "rank_weight": 0.25,
  "clean_hard_negative_weight": 0.1,
  "composite_outside_hard_negative_weight": 0.1
}

Clean backbone/FPN extraction count: 1
Clean pred_tower + center_head:      enabled
Second clean backbone pass:          NO
Architecture parameters changed:     NO
Instance loss changed:               NO
Original feature PCC changed:        NO
CRPA changed:                        NO
Renderer/data changed:               NO
AMP initial scale:                   32
DASO-Net Phase 5C — PCC-CENTER TRAINER v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC-center configuration:
{
  "enabled": true,
  "stride": 4,
  "gu

[W913 10:48:27.684863541 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 10:48:27.707328352 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_pcccenter_prodshape_sanity_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     40
Optimizer steps total:      10
Smoke mode:                 False
epoch=00 step=00000/00040 loss=29.9726 inst=27.1603 pcc=2.2081 crpa=0.6041 lr=3.33e-05 vram_alloc=9.48GB vram_res=9.75GB
epoch=00 step=00005/00040 loss=17.6372 inst=15.3056 pcc=1.7120 crpa=0.6195 lr=6.67e-05 vram_alloc=9.55GB vram_res=9.75GB
epoch=00 step=00010/00040 loss=8.6362 inst=6.9753 pcc=0.8976

In [45]:
# ============================================================
# PHASE 5C — PCC-CENTER 1000-STEP CONTROLLED ABLATION
#
# BASELINE MATCH:
#   fresh initialization
#   same random seed
#   same architecture
#   same renderer/data
#   same InstanceSegLoss
#   same original PCC
#   same CRPA
#   same T4 x2
#   same native 1024
#   same batch/GPU = 1
#   same accumulation = 4
#   same effective synthetic batch = 8
#   same AMP FP16 init scale = 32
#   same 1000 train steps/rank
#   same 250 optimizer updates
#   same 50-update warm-up
#   same constant 3e-4 post-warm-up LR
#
# ONLY SCIENTIFIC CHANGE:
#   + PCC-center auxiliary objective
#
# Engineering ablation only.
# NOT a manuscript performance result.
# ============================================================

import hashlib
import json
import shutil
import subprocess
import time
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5c_pcc_center.py"
)

BASE_CONFIG = Path(
    "/kaggle/working/"
    "phase5c_scheduleprobe1000_v1.json"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_pcccenter1000_v2.json"
)

RUN_NAME = (
    "dasonet_phase5c_pcccenter1000_v2"
)

OUT = (
    Path(
        "/kaggle/working/"
        "dasonet_phase5b"
    )
    / RUN_NAME
)


def sha256(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


for required in (
    TRAINER,
    BASE_CONFIG,
    PROJECT_ROOT,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


# ------------------------------------------------------------
# Verify the proven v2 trainer is still active.
# ------------------------------------------------------------

trainer_text = TRAINER.read_text(
    encoding="utf-8"
)

required_markers = [
    "DASO-Net Phase 5C PCC-Center trainer v2",
    "clean_features =",
    "self.core.extract_features(",
    "clean_pred =",
    "self.core.pred_tower(",
    "clean_center_logits =",
    "self.core.center_head(",
]

missing = [
    marker
    for marker in required_markers
    if marker not in trainer_text
]

if missing:
    raise RuntimeError(
        "Verified PCC-center v2 trainer is not active.\n"
        f"Missing: {missing}"
    )


forbidden = [
    "register_forward_hook",
    "PCC-center wrapper failed to capture clean center logits",
]

bad = [
    marker
    for marker in forbidden
    if marker in trainer_text
]

if bad:
    raise RuntimeError(
        "Old hook-based implementation is present.\n"
        f"Found: {bad}"
    )


TRAINER_SHA256 = sha256(
    TRAINER
)


# ------------------------------------------------------------
# Start from the exact matched 1000-step baseline config.
# ------------------------------------------------------------

cfg = json.loads(
    BASE_CONFIG.read_text(
        encoding="utf-8"
    )
)


cfg[
    "experiment_name"
] = RUN_NAME


# ------------------------------------------------------------
# Frozen practical Phase-5B/5C runtime.
# ------------------------------------------------------------

cfg["data"][
    "per_rank_synthetic_batch"
] = 1

cfg["data"][
    "per_rank_anchor_batch"
] = 2

cfg["data"][
    "synthetic_workers_per_rank"
] = 2

cfg["data"][
    "anchor_workers_per_rank"
] = 1

cfg["data"][
    "prefetch_factor"
] = 2


cfg["optimization"][
    "gradient_accumulation_steps"
] = 4

cfg["optimization"][
    "amp"
] = True

cfg["optimization"][
    "learning_rate"
] = 3e-4

cfg["optimization"][
    "warmup_optimizer_steps"
] = 50

cfg["optimization"][
    "min_lr_ratio"
] = 1.0


# ------------------------------------------------------------
# ONLY controlled scientific addition.
# ------------------------------------------------------------

cfg.setdefault(
    "loss",
    {}
)

cfg[
    "loss"
][
    "pcc_center"
] = {
    "enabled":
        True,

    "stride":
        4,

    "guard_feature_pixels":
        2,

    "hard_negative_topk":
        64,

    "lift_margin_logit":
        0.50,

    "rank_margin_logit":
        1.00,

    "lift_weight":
        0.25,

    "rank_weight":
        0.25,

    "clean_hard_negative_weight":
        0.10,

    "composite_outside_hard_negative_weight":
        0.10,
}


CONFIG.write_text(
    json.dumps(
        cfg,
        indent=2,
    ),
    encoding="utf-8",
)


CONFIG_SHA256 = sha256(
    CONFIG
)


if OUT.exists():
    shutil.rmtree(
        OUT
    )


cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(TRAINER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "1",

    "--max-train-steps",
    "1000",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    "none",
]


print(
    "============================================================"
)

print(
    "PHASE 5C — PCC-CENTER 1000-STEP CONTROLLED ABLATION"
)

print(
    "============================================================"
)

print(
    "Trainer SHA256:"
)

print(
    TRAINER_SHA256
)

print()

print(
    "Config SHA256:"
)

print(
    CONFIG_SHA256
)

print()

print(
    "Train steps/rank:         1000"
)

print(
    "Optimizer updates:        250"
)

print(
    "Warm-up updates:          50"
)

print(
    "Post-warm-up LR:          3.0e-4 constant"
)

print(
    "Accumulation:             4"
)

print(
    "Effective batch:          8"
)

print(
    "AMP initial scale:        32"
)

print(
    "Controlled change:        PCC-center auxiliary ONLY"
)

print()

print(
    " ".join(cmd)
)

print()


start = time.time()


subprocess.run(
    cmd,
    check=True,
)


elapsed = (
    time.time()
    - start
)


status_path = (
    OUT
    / "run_status.json"
)

best_path = (
    OUT
    / "best.pt"
)

last_path = (
    OUT
    / "last.pt"
)


for path in (
    status_path,
    best_path,
    last_path,
):
    if not path.exists():
        raise RuntimeError(
            f"Missing training artifact:\n{path}"
        )


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:
    raise RuntimeError(
        "PCC-center 1000-step ablation had "
        f"{amp_skips} AMP-skipped optimizer updates. "
        "Do not evaluate this run."
    )


# Check final scaler state.
import torch

last_ckpt = torch.load(
    last_path,
    map_location="cpu",
    weights_only=False,
)

scaler_state = (
    last_ckpt.get(
        "scaler_state_dict",
        {},
    )
)


print()
print(
    "============================================================"
)

print(
    "PHASE 5C PCC-CENTER TRAINING COMPLETE"
)

print(
    "============================================================"
)

print(
    f"Wall time:                 "
    f"{elapsed / 60.0:.2f} min"
)

print(
    "AMP skipped updates:       ",
    amp_skips,
)

print(
    "Final scaler scale:         ",
    scaler_state.get(
        "scale"
    ),
)

print(
    "Best validation total loss: ",
    status.get(
        "best_val_total_loss"
    ),
)

print(
    "Best checkpoint:"
)

print(
    best_path
)

print()

print(
    "========================================"
)

print(
    "PHASE 5C PCC-CENTER 1000-STEP ABLATION: PASS"
)

print(
    "========================================"
)

PHASE 5C — PCC-CENTER 1000-STEP CONTROLLED ABLATION
Trainer SHA256:
0ce021074170332be6fc682b076507b63c6dafe40af7c50fa5909f8232591787

Config SHA256:
1ddc9978ddc016c6c19ccd8892885af4afa7959d70ad24ccab44a7fe80ff6b3d

Train steps/rank:         1000
Optimizer updates:        250
Warm-up updates:          50
Post-warm-up LR:          3.0e-4 constant
Accumulation:             4
Effective batch:          8
AMP initial scale:        32
Controlled change:        PCC-center auxiliary ONLY

torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5c_pcc_center.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_pcccenter1000_v2.json --output-root /kaggle/working --epochs 1 --max-train-steps 1000 --max-val-steps 100 --accum 4 --num-workers 2 --log-every 100 --resume none



W0913 10:52:24.895000 1655 torch/distributed/run.py:852] 
W0913 10:52:24.895000 1655 torch/distributed/run.py:852] *****************************************
W0913 10:52:24.895000 1655 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 10:52:24.895000 1655 torch/distributed/run.py:852] *****************************************
[W913 10:52:25.417560364 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — PCC-CENTER TRAINER v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC-center configuration:
{
  "enabled": true,
  "stride": 4,
  "guard_feature_pixels": 2,
  "hard_negative_topk": 64,
  "lift_margin_logit": 0.5,
  "rank_margin_logit": 1.0,
  "lift_weight": 0.25,
  "rank_weight": 0.25,
  "clean_hard_negative_weight": 0.1,
  "composite_outside_hard_negative_weight": 0.1
}

Clean backbone/FPN extraction count: 1
Clean pred_tower + center_head:      enabled
Second clean backbone pass:          NO
Architecture parameters changed:     NO
Instance loss changed:               NO
Original feature PCC changed:        NO
CRPA changed:                        NO
Renderer/data changed:               NO
AMP initial scale:                   32
DASO-Net Phase 5C — PCC-CENTER TRAINER v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC-center configuration:
{
  "enabled": true,
  "stride": 4,
  "gu

[W913 10:52:27.120772337 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 10:52:27.148339509 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_pcccenter1000_v2
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      250
Smoke mode:                 False
epoch=00 step=00000/01000 loss=29.9725 inst=27.1603 pcc=2.2081 crpa=0.6041 lr=6.00e-06 vram_alloc=9.48GB vram_res=9.75GB
epoch=00 step=00100/01000 loss=8.1298 inst=6.8498 pcc=0.6988 crpa=0.5811 lr=1.56e-04 vram_alloc=9.64GB vram_res=9.75GB
epoch=00 step=00200/01000 loss=0.4481 inst=0.0983 pcc=0.0025 crpa=0.3473

## Controlled evaluation against matched baseline

In [46]:
# ============================================================
# PHASE 5C — PCC-CENTER CONTROLLED ABLATION EVALUATION
#
# MATCHED BASELINE:
#   dasonet_phase5c_scheduleprobe1000_v1
#
# PCC-CENTER:
#   dasonet_phase5c_pcccenter1000_v2
#
# SAME deterministic 200 validation scenes.
#
# Evaluates:
#   1. Instance metrics
#   2. Oracle component behavior
#   3. Clean/composite counterfactual-center behavior
#
# Engineering ablation only.
# ============================================================

import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE_DIAGNOSTIC = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL_DIAGNOSTIC = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

NEW_CHECKPOINT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_pcccenter1000_v2/"
    "best.pt"
)


# ------------------------------------------------------------
# Existing matched baseline outputs.
# ------------------------------------------------------------

BASELINE_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "scheduleprobe1000_evaluation_v1"
)

BASELINE_METRIC = (
    BASELINE_ROOT
    / "metrics"
    / "phase5c_metric_summary.json"
)

BASELINE_ORACLE = (
    BASELINE_ROOT
    / "oracle"
    / "oracle_component_diagnostic.json"
)

BASELINE_COUNTERFACTUAL = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "counterfactual_center_diagnostic_v1/"
    "counterfactual_center_diagnostic.json"
)


# ------------------------------------------------------------
# New output root.
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "pcccenter1000_v2_evaluation"
)

METRIC_OUT = (
    ROOT
    / "metrics"
)

ORACLE_OUT = (
    ROOT
    / "oracle"
)

COUNTERFACTUAL_OUT = (
    ROOT
    / "counterfactual"
)


for required in (
    PROJECT_ROOT,
    NEW_CHECKPOINT,
    BASELINE_METRIC,
    BASELINE_ORACLE,
    BASELINE_COUNTERFACTUAL,
    EVALUATOR,
    ORACLE_DIAGNOSTIC,
    COUNTERFACTUAL_DIAGNOSTIC,
    DECODER,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


def sha256(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


DECODER_SHA256 = sha256(
    DECODER
)


# The previously verified corrected decoder should still be active.
decoder_text = DECODER.read_text(
    encoding="utf-8"
)

for marker in (
    "def local_peak_suppression(",
    "local_peak_kernel: int = 3",
    "center = local_peak_suppression(",
):
    if marker not in decoder_text:
        raise RuntimeError(
            "Corrected Phase-5C decoder is no longer active."
        )


if ROOT.exists():
    shutil.rmtree(
        ROOT
    )

ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. INSTANCE METRICS
# ============================================================

metric_cmd = [
    sys.executable,
    str(EVALUATOR),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(NEW_CHECKPOINT),

    "--output-dir",
    str(METRIC_OUT),

    "--split",
    "val",

    "--max-scenes",
    "200",

    "--seed",
    "20260917",

    "--num-workers",
    "2",

    "--decode-min-score",
    "0.001",

    "--operating-score",
    "0.05",

    "--topk",
    "100",

    "--nms-iou",
    "0.5",

    "--mask-threshold",
    "0.5",

    "--match-iou",
    "0.5",
]


print(
    "============================================================"
)

print(
    "PCC-CENTER v2 — INSTANCE METRICS"
)

print(
    "============================================================"
)

print(
    "Decoder SHA256:"
)

print(
    DECODER_SHA256
)

print()


subprocess.run(
    metric_cmd,
    check=True,
)


# ============================================================
# 2. ORACLE COMPONENT DIAGNOSTIC
# ============================================================

oracle_cmd = [
    sys.executable,
    str(ORACLE_DIAGNOSTIC),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(NEW_CHECKPOINT),

    "--evaluator",
    str(EVALUATOR),

    "--output-dir",
    str(ORACLE_OUT),

    "--split",
    "val",

    "--max-scenes",
    "200",

    "--seed",
    "20260917",

    "--num-workers",
    "2",

    "--stride",
    "4",

    "--peak-kernel",
    "3",

    "--topk",
    "100",

    "--mask-threshold",
    "0.5",
]


print()
print(
    "============================================================"
)

print(
    "PCC-CENTER v2 — ORACLE DIAGNOSTIC"
)

print(
    "============================================================"
)


subprocess.run(
    oracle_cmd,
    check=True,
)


# ============================================================
# 3. COUNTERFACTUAL CENTER DIAGNOSTIC
# ============================================================

counterfactual_cmd = [
    sys.executable,
    str(COUNTERFACTUAL_DIAGNOSTIC),

    "--project-root",
    str(PROJECT_ROOT),

    "--checkpoint",
    str(NEW_CHECKPOINT),

    "--evaluator",
    str(EVALUATOR),

    "--output-dir",
    str(COUNTERFACTUAL_OUT),

    "--split",
    "val",

    "--max-scenes",
    "200",

    "--seed",
    "20260917",

    "--num-workers",
    "2",

    "--stride",
    "4",

    "--peak-kernel",
    "3",

    "--window-radius-feature",
    "2",
]


print()
print(
    "============================================================"
)

print(
    "PCC-CENTER v2 — COUNTERFACTUAL DIAGNOSTIC"
)

print(
    "============================================================"
)


subprocess.run(
    counterfactual_cmd,
    check=True,
)


# ============================================================
# LOAD RESULTS
# ============================================================

NEW_METRIC = (
    METRIC_OUT
    / "phase5c_metric_summary.json"
)

NEW_ORACLE = (
    ORACLE_OUT
    / "oracle_component_diagnostic.json"
)

NEW_COUNTERFACTUAL = (
    COUNTERFACTUAL_OUT
    / "counterfactual_center_diagnostic.json"
)


def load_json(path: Path):
    if not path.exists():
        raise RuntimeError(
            f"Expected output missing:\n{path}"
        )

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


old_metric = load_json(
    BASELINE_METRIC
)

new_metric = load_json(
    NEW_METRIC
)

old_oracle = load_json(
    BASELINE_ORACLE
)

new_oracle = load_json(
    NEW_ORACLE
)

old_cf = load_json(
    BASELINE_COUNTERFACTUAL
)

new_cf = load_json(
    NEW_COUNTERFACTUAL
)


# ============================================================
# SAMPLE-EQUIVALENCE CHECKS
# ============================================================

if (
    old_metric[
        "sample_particle_count_distribution"
    ]
    !=
    new_metric[
        "sample_particle_count_distribution"
    ]
):
    raise RuntimeError(
        "Particle-count distribution mismatch."
    )


if (
    old_metric[
        "sample_fruit_distribution"
    ]
    !=
    new_metric[
        "sample_fruit_distribution"
    ]
):
    raise RuntimeError(
        "Fruit distribution mismatch."
    )


if (
    int(old_oracle["particles"])
    !=
    int(new_oracle["particles"])
):
    raise RuntimeError(
        "Oracle particle-count mismatch."
    )


if (
    int(old_cf["particles"])
    !=
    int(new_cf["particles"])
):
    raise RuntimeError(
        "Counterfactual particle-count mismatch."
    )


# ============================================================
# INSTANCE METRIC TABLE
# ============================================================

def metric_values(x):

    op = x[
        "operating_point"
    ]

    ap = x[
        "mask_ap"
    ]

    sweep = x[
        "validation_threshold_sweep"
    ]

    return {
        "AP50":
            float(
                ap["mask_ap50"]
            ),

        "AP75":
            float(
                ap["mask_ap75"]
            ),

        "AP50:95":
            float(
                ap["mask_ap50_95"]
            ),

        "Precision@0.05":
            float(
                op["precision"]
            ),

        "Recall@0.05":
            float(
                op["recall"]
            ),

        "F1@0.05":
            float(
                op["f1"]
            ),

        "MatchedDice":
            float(
                op[
                    "matched_mask_dice_mean"
                ]
            ),

        "CountMAE":
            float(
                op["count_mae"]
            ),

        "EmptyFP/image":
            float(
                op[
                    "empty_scene_fp_per_image"
                ]
            ),

        "BestThreshold":
            float(
                sweep[
                    "best_threshold_by_f1"
                ]
            ),

        "BestF1":
            float(
                sweep[
                    "best_f1"
                ]
            ),
    }


baseline_m = metric_values(
    old_metric
)

pcc_m = metric_values(
    new_metric
)


print()
print(
    "=================================================================="
)

print(
    "PCC-CENTER CONTROLLED ABLATION — INSTANCE METRICS"
)

print(
    "=================================================================="
)

print(
    f"{'Metric':<22}"
    f"{'baseline':>14}"
    f"{'PCC-center':>14}"
    f"{'Delta':>14}"
)

print(
    "-" * 64
)


for key in baseline_m:

    old = baseline_m[key]
    new = pcc_m[key]

    print(
        f"{key:<22}"
        f"{old:>14.4f}"
        f"{new:>14.4f}"
        f"{(new - old):>+14.4f}"
    )


# ============================================================
# ORACLE TABLE
# ============================================================

def oracle_values(x):

    center = x[
        "center_head"
    ]

    geom = x[
        "oracle_gt_cell_geometry"
    ]

    mask = x[
        "oracle_gt_cell_mask"
    ]

    empty = x[
        "empty_scene_center_behavior"
    ]

    return {
        "GTCenterScoreMean":
            float(
                center[
                    "gt_cell_score_stats"
                ]["mean"]
            ),

        "PeakRecall<=4px":
            float(
                center[
                    "top100_peak_recall_within_4px"
                ]
            ),

        "PeakRecall<=8px":
            float(
                center[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall<=16px":
            float(
                center[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "CenterErrorPx":
            float(
                geom[
                    "center_error_px_stats"
                ]["mean"]
            ),

        "BoxIoUMean":
            float(
                geom[
                    "box_iou_stats"
                ]["mean"]
            ),

        "BoxIoU>=0.50":
            float(
                geom[
                    "box_iou_ge_0_50"
                ]
            ),

        "WidthRatioMedian":
            float(
                geom[
                    "pred_to_gt_width_ratio_stats"
                ]["median"]
            ),

        "HeightRatioMedian":
            float(
                geom[
                    "pred_to_gt_height_ratio_stats"
                ]["median"]
            ),

        "GTBoxMaskIoUMean":
            float(
                mask[
                    "gt_box_crop_mask_iou_stats"
                ]["mean"]
            ),

        "GTBoxMaskIoU>=0.50":
            float(
                mask[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),

        "EmptyMaxPeakMean":
            float(
                empty[
                    "max_peak_score_stats"
                ]["mean"]
            ),

        "EmptyPeaks>=0.05":
            float(
                empty[
                    "mean_local_peak_count_by_threshold"
                ]["0.05"]
            ),

        "EmptyPeaks>=0.10":
            float(
                empty[
                    "mean_local_peak_count_by_threshold"
                ]["0.10"]
            ),

        "EmptyPeaks>=0.20":
            float(
                empty[
                    "mean_local_peak_count_by_threshold"
                ]["0.20"]
            ),
    }


baseline_o = oracle_values(
    old_oracle
)

pcc_o = oracle_values(
    new_oracle
)


print()
print(
    "=================================================================="
)

print(
    "PCC-CENTER CONTROLLED ABLATION — ORACLE COMPONENTS"
)

print(
    "=================================================================="
)

print(
    f"{'Diagnostic':<24}"
    f"{'baseline':>14}"
    f"{'PCC-center':>14}"
    f"{'Delta':>14}"
)

print(
    "-" * 66
)


for key in baseline_o:

    old = baseline_o[key]
    new = pcc_o[key]

    print(
        f"{key:<24}"
        f"{old:>14.4f}"
        f"{new:>14.4f}"
        f"{(new - old):>+14.4f}"
    )


# ============================================================
# COUNTERFACTUAL TABLE
# ============================================================

def counterfactual_values(x):

    exact = x[
        "exact_gt_cell"
    ]

    local = x[
        "local_window_peak"
    ]

    scene = x[
        "scene_level"
    ]

    return {
        "CleanGTScoreMean":
            float(
                exact[
                    "clean_score_stats"
                ]["mean"]
            ),

        "CompositeGTScoreMean":
            float(
                exact[
                    "composite_score_stats"
                ]["mean"]
            ),

        "ExactLiftMean":
            float(
                exact[
                    "composite_minus_clean_stats"
                ]["mean"]
            ),

        "ExactComp>Clean":
            float(
                exact[
                    "fraction_composite_gt_clean"
                ]
            ),

        "ExactLift>=0.01":
            float(
                exact[
                    "fraction_lift_ge_0_01"
                ]
            ),

        "ExactLift>=0.05":
            float(
                exact[
                    "fraction_lift_ge_0_05"
                ]
            ),

        "LocalCleanMean":
            float(
                local[
                    "clean_peak_stats"
                ]["mean"]
            ),

        "LocalCompositeMean":
            float(
                local[
                    "composite_peak_stats"
                ]["mean"]
            ),

        "LocalLiftMean":
            float(
                local[
                    "composite_minus_clean_stats"
                ]["mean"]
            ),

        "LocalComp>Clean":
            float(
                local[
                    "fraction_composite_gt_clean"
                ]
            ),

        "CleanMaxPeakMean":
            float(
                scene[
                    "clean_max_peak_stats"
                ]["mean"]
            ),

        "OutsideMaxPeakMean":
            float(
                scene[
                    "composite_outside_effect_max_peak_stats"
                ]["mean"]
            ),

        "EffectMaxPeakMean":
            float(
                scene[
                    "composite_effect_max_peak_stats"
                ]["mean"]
            ),

        "CleanPeaks>=0.05":
            float(
                scene[
                    "clean_mean_peak_count_by_threshold"
                ]["0.05"]
            ),

        "OutsidePeaks>=0.05":
            float(
                scene[
                    "composite_outside_effect_mean_peak_count_by_threshold"
                ]["0.05"]
            ),

        "EffectPeaks>=0.05":
            float(
                scene[
                    "composite_effect_mean_peak_count_by_threshold"
                ]["0.05"]
            ),

        "CleanPeaks>=0.10":
            float(
                scene[
                    "clean_mean_peak_count_by_threshold"
                ]["0.1"]
            ),

        "OutsidePeaks>=0.10":
            float(
                scene[
                    "composite_outside_effect_mean_peak_count_by_threshold"
                ]["0.1"]
            ),

        "EffectPeaks>=0.10":
            float(
                scene[
                    "composite_effect_mean_peak_count_by_threshold"
                ]["0.1"]
            ),
    }


baseline_c = counterfactual_values(
    old_cf
)

pcc_c = counterfactual_values(
    new_cf
)


print()
print(
    "=================================================================="
)

print(
    "PCC-CENTER CONTROLLED ABLATION — COUNTERFACTUAL RESPONSE"
)

print(
    "=================================================================="
)

print(
    f"{'Diagnostic':<24}"
    f"{'baseline':>14}"
    f"{'PCC-center':>14}"
    f"{'Delta':>14}"
)

print(
    "-" * 66
)


for key in baseline_c:

    old = baseline_c[key]
    new = pcc_c[key]

    print(
        f"{key:<24}"
        f"{old:>14.4f}"
        f"{new:>14.4f}"
        f"{(new - old):>+14.4f}"
    )


# ============================================================
# SAVE CONTROLLED COMPARISON RECORD
# ============================================================

comparison = {
    "phase":
        "5C",

    "paper_result":
        False,

    "experiment":
        "pcc_center_1000_step_controlled_ablation_v2",

    "decoder_sha256":
        DECODER_SHA256,

    "controlled_change":
        "PCC-center auxiliary objective only",

    "matched_training": {
        "train_steps_per_rank":
            1000,

        "optimizer_updates":
            250,

        "warmup_optimizer_steps":
            50,

        "learning_rate":
            3e-4,

        "post_warmup_schedule":
            "constant",

        "native_tile":
            1024,

        "world_size":
            2,

        "batch_per_gpu":
            1,

        "gradient_accumulation":
            4,

        "effective_synthetic_batch":
            8,

        "amp":
            "fp16",

        "amp_initial_scale":
            32,

        "architecture_changed":
            False,

        "instance_loss_changed":
            False,

        "original_pcc_changed":
            False,

        "crpa_changed":
            False,

        "renderer_changed":
            False,

        "data_changed":
            False,
    },

    "baseline_metrics":
        baseline_m,

    "pcc_center_metrics":
        pcc_m,

    "baseline_oracle":
        baseline_o,

    "pcc_center_oracle":
        pcc_o,

    "baseline_counterfactual":
        baseline_c,

    "pcc_center_counterfactual":
        pcc_c,

    "particle_count_distribution":
        new_metric[
            "sample_particle_count_distribution"
        ],

    "fruit_distribution":
        new_metric[
            "sample_fruit_distribution"
        ],
}


comparison_path = (
    ROOT
    / "pcc_center_controlled_ablation_v2.json"
)


comparison_path.write_text(
    json.dumps(
        comparison,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "Particle-count distribution:"
)

print(
    comparison[
        "particle_count_distribution"
    ]
)


print()
print(
    "Fruit distribution:"
)

print(
    comparison[
        "fruit_distribution"
    ]
)


print()
print(
    "Comparison JSON:"
)

print(
    comparison_path
)


print()
print(
    "=================================================================="
)

print(
    "PHASE 5C PCC-CENTER CONTROLLED ABLATION v2: PASS"
)

print(
    "=================================================================="
)

print(
    "Do not launch longer training until these tables are reviewed."
)

PCC-CENTER v2 — INSTANCE METRICS
Decoder SHA256:
9799eb1be568e3d6e0ce82b7c99df7a0ed05d271cd44ff24b612526cc18537d1

eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=100 op_pred=21 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=100 op_pred=23 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=99 op_pred=20 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=100 op_pred=31 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=100 op_pred=27 elapsed=0.6m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=100 op_pred=7 elapsed=0.7m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=99 op_pred=7 elapsed=0.8m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=99 op_pred=3 elapsed=0.9m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=100 op_pred=22 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=99 op_pred=10 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0d7548bc gt=0 decoded=100 op_

In [47]:
# ============================================================
# PHASE 5C — PCC-CENTER RELATIVE-ONLY ABLATION
#
# Uses the already verified/stable PCC-center v2 trainer.
#
# Matched to baseline and v2:
#   1000 train steps/rank
#   250 optimizer updates
#   warmup 50
#   constant 3e-4 thereafter
#   T4 x2
#   native 1024
#   batch/GPU 1
#   accumulation 4
#   effective batch 8
#   AMP FP16 initial scale 32
#
# ONLY CHANGE FROM PCC-center v2:
#
#   clean_hard_negative_weight = 0
#   composite_outside_hard_negative_weight = 0
#
# Remaining PCC-center terms:
#   + counterfactual GT-center lift
#   + positive-vs-background ranking
#
# No code modification.
# ============================================================

import hashlib
import json
import shutil
import subprocess
import time
from pathlib import Path

import torch


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5c_pcc_center.py"
)

BASE_CONFIG = Path(
    "/kaggle/working/"
    "phase5c_scheduleprobe1000_v1.json"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_pcccenter_relative1000_v1.json"
)

RUN_NAME = (
    "dasonet_phase5c_pcccenter_relative1000_v1"
)

OUT = (
    Path(
        "/kaggle/working/"
        "dasonet_phase5b"
    )
    / RUN_NAME
)


def sha256(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


for required in (
    TRAINER,
    PROJECT_ROOT,
    BASE_CONFIG,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


# ------------------------------------------------------------
# Verify the already-proven v2 trainer.
# ------------------------------------------------------------

trainer_text = TRAINER.read_text(
    encoding="utf-8"
)

required_markers = [
    "DASO-Net Phase 5C PCC-Center trainer v2",
    "clean_center_logits",
    "PCCCenterAuxLoss",
    "self.core.pred_tower(",
    "self.core.center_head(",
]

missing = [
    x
    for x in required_markers
    if x not in trainer_text
]

if missing:
    raise RuntimeError(
        "Verified PCC-center v2 trainer is not active.\n"
        f"Missing: {missing}"
    )


trainer_sha = sha256(
    TRAINER
)


# ------------------------------------------------------------
# Start from exact 1000-step baseline configuration.
# ------------------------------------------------------------

cfg = json.loads(
    BASE_CONFIG.read_text(
        encoding="utf-8"
    )
)

cfg["experiment_name"] = RUN_NAME


# Frozen runtime.
cfg["data"][
    "per_rank_synthetic_batch"
] = 1

cfg["data"][
    "per_rank_anchor_batch"
] = 2

cfg["data"][
    "synthetic_workers_per_rank"
] = 2

cfg["data"][
    "anchor_workers_per_rank"
] = 1

cfg["data"][
    "prefetch_factor"
] = 2


cfg["optimization"][
    "gradient_accumulation_steps"
] = 4

cfg["optimization"][
    "amp"
] = True

cfg["optimization"][
    "learning_rate"
] = 3e-4

cfg["optimization"][
    "warmup_optimizer_steps"
] = 50

cfg["optimization"][
    "min_lr_ratio"
] = 1.0


# ------------------------------------------------------------
# Relative-only PCC-center.
#
# SAME lift/rank settings as rejected v2.
# ONLY remove absolute suppression.
# ------------------------------------------------------------

cfg.setdefault(
    "loss",
    {}
)

cfg["loss"][
    "pcc_center"
] = {
    "enabled":
        True,

    "stride":
        4,

    "guard_feature_pixels":
        2,

    "hard_negative_topk":
        64,

    "lift_margin_logit":
        0.50,

    "rank_margin_logit":
        1.00,

    "lift_weight":
        0.25,

    "rank_weight":
        0.25,

    "clean_hard_negative_weight":
        0.00,

    "composite_outside_hard_negative_weight":
        0.00,
}


CONFIG.write_text(
    json.dumps(
        cfg,
        indent=2,
    ),
    encoding="utf-8",
)

config_sha = sha256(
    CONFIG
)


if OUT.exists():
    shutil.rmtree(
        OUT
    )


cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(TRAINER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "1",

    "--max-train-steps",
    "1000",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    "none",
]


print(
    "============================================================"
)

print(
    "PHASE 5C — RELATIVE-ONLY PCC-CENTER 1000-STEP ABLATION"
)

print(
    "============================================================"
)

print(
    "Trainer SHA256:"
)

print(
    trainer_sha
)

print()

print(
    "Config SHA256:"
)

print(
    config_sha
)

print()

print(
    "Lift weight:                0.25"
)

print(
    "Ranking weight:             0.25"
)

print(
    "Clean absolute suppression: 0.00"
)

print(
    "Outside abs. suppression:   0.00"
)

print()

print(
    "Train steps/rank:           1000"
)

print(
    "Optimizer updates:          250"
)

print(
    "Warmup updates:             50"
)

print(
    "Post-warmup LR:             3e-4 constant"
)

print(
    "Effective batch:            8"
)

print(
    "AMP initial scale:          32"
)

print()

print(
    " ".join(cmd)
)

print()


start = time.time()


subprocess.run(
    cmd,
    check=True,
)


elapsed = time.time() - start


status_path = (
    OUT
    / "run_status.json"
)

last_path = (
    OUT
    / "last.pt"
)

best_path = (
    OUT
    / "best.pt"
)


for required in (
    status_path,
    last_path,
    best_path,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing run artifact:\n{required}"
        )


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:
    raise RuntimeError(
        f"Unexpected AMP skips: {amp_skips}"
    )


last_ckpt = torch.load(
    last_path,
    map_location="cpu",
    weights_only=False,
)

scaler_state = last_ckpt.get(
    "scaler_state_dict",
    {},
)


print()
print(
    "============================================================"
)

print(
    "RELATIVE-ONLY PCC-CENTER TRAINING COMPLETE"
)

print(
    "============================================================"
)

print(
    f"Wall time:                  "
    f"{elapsed / 60.0:.2f} min"
)

print(
    "AMP skipped updates:        ",
    amp_skips,
)

print(
    "Final scaler scale:          ",
    scaler_state.get(
        "scale"
    ),
)

print(
    "Best validation total loss:  ",
    status.get(
        "best_val_total_loss"
    ),
)

print(
    "Best checkpoint:"
)

print(
    best_path
)

print()

print(
    "========================================"
)

print(
    "PHASE 5C RELATIVE-ONLY PCC-CENTER TRAINING: PASS"
)

print(
    "========================================"
)

PHASE 5C — RELATIVE-ONLY PCC-CENTER 1000-STEP ABLATION
Trainer SHA256:
0ce021074170332be6fc682b076507b63c6dafe40af7c50fa5909f8232591787

Config SHA256:
9b31d7c4d0e2b987413c9a7244343b3066252c1114635b91e7aa5a2626837c3d

Lift weight:                0.25
Ranking weight:             0.25
Clean absolute suppression: 0.00
Outside abs. suppression:   0.00

Train steps/rank:           1000
Optimizer updates:          250
Warmup updates:             50
Post-warmup LR:             3e-4 constant
Effective batch:            8
AMP initial scale:          32

torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5c_pcc_center.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_pcccenter_relative1000_v1.json --output-root /kaggle/working --epochs 1 --max-train-steps 1000 --max-val-steps 100 --accum 4 --num-workers 2 --log-every 100 --resume none



W0913 11:28:34.394000 1882 torch/distributed/run.py:852] 
W0913 11:28:34.394000 1882 torch/distributed/run.py:852] *****************************************
W0913 11:28:34.394000 1882 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 11:28:34.394000 1882 torch/distributed/run.py:852] *****************************************
[W913 11:28:34.797689786 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — PCC-CENTER TRAINER v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC-center configuration:
{
  "enabled": true,
  "stride": 4,
  "guard_feature_pixels": 2,
  "hard_negative_topk": 64,
  "lift_margin_logit": 0.5,
  "rank_margin_logit": 1.0,
  "lift_weight": 0.25,
  "rank_weight": 0.25,
  "clean_hard_negative_weight": 0.0,
  "composite_outside_hard_negative_weight": 0.0
}

Clean backbone/FPN extraction count: 1
Clean pred_tower + center_head:      enabled
Second clean backbone pass:          NO
Architecture parameters changed:     NO
Instance loss changed:               NO
Original feature PCC changed:        NO
CRPA changed:                        NO
Renderer/data changed:               NO
AMP initial scale:                   32
DASO-Net Phase 5C — PCC-CENTER TRAINER v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

PCC-center configuration:
{
  "enabled": true,
  "stride": 4,
  "gu

[W913 11:28:37.477181382 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 11:28:37.497909672 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_pcccenter_relative1000_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      250
Smoke mode:                 False
epoch=00 step=00000/01000 loss=29.6886 inst=27.1603 pcc=1.9242 crpa=0.6041 lr=6.00e-06 vram_alloc=9.48GB vram_res=9.75GB
epoch=00 step=00100/01000 loss=8.1297 inst=6.8502 pcc=0.6983 crpa=0.5812 lr=1.56e-04 vram_alloc=9.64GB vram_res=9.75GB
epoch=00 step=00200/01000 loss=0.4457 inst=0.0984 pcc=0.0000 cr

In [48]:
# ============================================================
# PHASE 5C — RELATIVE-ONLY PCC-CENTER EVALUATION
#
# Three-way controlled comparison:
#
#   1. Baseline 1000-step schedule probe
#   2. PCC-center v2 with absolute suppression
#   3. PCC-center relative-only
#
# Exact same deterministic 200 validation scenes.
# ============================================================

import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)


NEW_CHECKPOINT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_pcccenter_relative1000_v1/"
    "best.pt"
)


BASE_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "scheduleprobe1000_evaluation_v1"
)

BASE_METRIC = (
    BASE_ROOT
    / "metrics"
    / "phase5c_metric_summary.json"
)

BASE_ORACLE = (
    BASE_ROOT
    / "oracle"
    / "oracle_component_diagnostic.json"
)

BASE_CF = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "counterfactual_center_diagnostic_v1/"
    "counterfactual_center_diagnostic.json"
)


V2_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "pcccenter1000_v2_evaluation"
)

V2_METRIC = (
    V2_ROOT
    / "metrics"
    / "phase5c_metric_summary.json"
)

V2_ORACLE = (
    V2_ROOT
    / "oracle"
    / "oracle_component_diagnostic.json"
)

V2_CF = (
    V2_ROOT
    / "counterfactual"
    / "counterfactual_center_diagnostic.json"
)


ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "pcccenter_relative1000_v1_evaluation"
)

METRIC_OUT = (
    ROOT
    / "metrics"
)

ORACLE_OUT = (
    ROOT
    / "oracle"
)

CF_OUT = (
    ROOT
    / "counterfactual"
)


for required in (
    NEW_CHECKPOINT,
    BASE_METRIC,
    BASE_ORACLE,
    BASE_CF,
    V2_METRIC,
    V2_ORACLE,
    V2_CF,
    EVALUATOR,
    ORACLE,
    COUNTERFACTUAL,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


if ROOT.exists():
    shutil.rmtree(
        ROOT
    )

ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# New model: instance evaluation
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        str(EVALUATOR),

        "--project-root",
        str(PROJECT_ROOT),

        "--checkpoint",
        str(NEW_CHECKPOINT),

        "--output-dir",
        str(METRIC_OUT),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--decode-min-score",
        "0.001",

        "--operating-score",
        "0.05",

        "--topk",
        "100",

        "--nms-iou",
        "0.5",

        "--mask-threshold",
        "0.5",

        "--match-iou",
        "0.5",
    ],
    check=True,
)


# ------------------------------------------------------------
# New model: oracle diagnostic
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        str(ORACLE),

        "--project-root",
        str(PROJECT_ROOT),

        "--checkpoint",
        str(NEW_CHECKPOINT),

        "--evaluator",
        str(EVALUATOR),

        "--output-dir",
        str(ORACLE_OUT),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--stride",
        "4",

        "--peak-kernel",
        "3",

        "--topk",
        "100",

        "--mask-threshold",
        "0.5",
    ],
    check=True,
)


# ------------------------------------------------------------
# New model: counterfactual diagnostic
# ------------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        str(COUNTERFACTUAL),

        "--project-root",
        str(PROJECT_ROOT),

        "--checkpoint",
        str(NEW_CHECKPOINT),

        "--evaluator",
        str(EVALUATOR),

        "--output-dir",
        str(CF_OUT),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--stride",
        "4",

        "--peak-kernel",
        "3",

        "--window-radius-feature",
        "2",
    ],
    check=True,
)


def load(path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


base_metric = load(
    BASE_METRIC
)

v2_metric = load(
    V2_METRIC
)

rel_metric = load(
    METRIC_OUT
    / "phase5c_metric_summary.json"
)


base_oracle = load(
    BASE_ORACLE
)

v2_oracle = load(
    V2_ORACLE
)

rel_oracle = load(
    ORACLE_OUT
    / "oracle_component_diagnostic.json"
)


base_cf = load(
    BASE_CF
)

v2_cf = load(
    V2_CF
)

rel_cf = load(
    CF_OUT
    / "counterfactual_center_diagnostic.json"
)


# ------------------------------------------------------------
# Verify exact validation sample.
# ------------------------------------------------------------

for other in (
    v2_metric,
    rel_metric,
):
    if (
        other[
            "sample_particle_count_distribution"
        ]
        !=
        base_metric[
            "sample_particle_count_distribution"
        ]
    ):
        raise RuntimeError(
            "Particle-count sample mismatch."
        )

    if (
        other[
            "sample_fruit_distribution"
        ]
        !=
        base_metric[
            "sample_fruit_distribution"
        ]
    ):
        raise RuntimeError(
            "Fruit sample mismatch."
        )


# ============================================================
# INSTANCE METRICS
# ============================================================

def mv(x):
    op = x["operating_point"]
    ap = x["mask_ap"]
    sw = x[
        "validation_threshold_sweep"
    ]

    return {
        "AP50":
            float(ap["mask_ap50"]),

        "AP75":
            float(ap["mask_ap75"]),

        "AP50:95":
            float(ap["mask_ap50_95"]),

        "Recall@0.05":
            float(op["recall"]),

        "F1@0.05":
            float(op["f1"]),

        "MatchedDice":
            float(
                op[
                    "matched_mask_dice_mean"
                ]
            ),

        "CountMAE":
            float(op["count_mae"]),

        "EmptyFP/image":
            float(
                op[
                    "empty_scene_fp_per_image"
                ]
            ),

        "BestThreshold":
            float(
                sw[
                    "best_threshold_by_f1"
                ]
            ),

        "BestF1":
            float(sw["best_f1"]),
    }


bm = mv(base_metric)
vm = mv(v2_metric)
rm = mv(rel_metric)


print()
print(
    "=========================================================================="
)

print(
    "PCC-CENTER THREE-WAY COMPARISON — INSTANCE METRICS"
)

print(
    "=========================================================================="
)

print(
    f"{'Metric':<22}"
    f"{'baseline':>13}"
    f"{'v2-absolute':>15}"
    f"{'relative-only':>16}"
    f"{'rel-base':>13}"
)

print(
    "-" * 79
)


for key in bm:
    print(
        f"{key:<22}"
        f"{bm[key]:>13.4f}"
        f"{vm[key]:>15.4f}"
        f"{rm[key]:>16.4f}"
        f"{(rm[key] - bm[key]):>+13.4f}"
    )


# ============================================================
# ORACLE
# ============================================================

def ov(x):
    c = x["center_head"]
    g = x[
        "oracle_gt_cell_geometry"
    ]
    m = x[
        "oracle_gt_cell_mask"
    ]
    e = x[
        "empty_scene_center_behavior"
    ]

    return {
        "GTCenterScoreMean":
            float(
                c[
                    "gt_cell_score_stats"
                ]["mean"]
            ),

        "PeakRecall<=4px":
            float(
                c[
                    "top100_peak_recall_within_4px"
                ]
            ),

        "PeakRecall<=8px":
            float(
                c[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall<=16px":
            float(
                c[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "BoxIoUMean":
            float(
                g[
                    "box_iou_stats"
                ]["mean"]
            ),

        "WidthRatioMedian":
            float(
                g[
                    "pred_to_gt_width_ratio_stats"
                ]["median"]
            ),

        "HeightRatioMedian":
            float(
                g[
                    "pred_to_gt_height_ratio_stats"
                ]["median"]
            ),

        "GTBoxMaskIoUMean":
            float(
                m[
                    "gt_box_crop_mask_iou_stats"
                ]["mean"]
            ),

        "GTBoxMaskIoU>=0.50":
            float(
                m[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),

        "EmptyMaxPeakMean":
            float(
                e[
                    "max_peak_score_stats"
                ]["mean"]
            ),

        "EmptyPeaks>=0.05":
            float(
                e[
                    "mean_local_peak_count_by_threshold"
                ]["0.05"]
            ),

        "EmptyPeaks>=0.10":
            float(
                e[
                    "mean_local_peak_count_by_threshold"
                ]["0.10"]
            ),
    }


bo = ov(base_oracle)
vo = ov(v2_oracle)
ro = ov(rel_oracle)


print()
print(
    "=========================================================================="
)

print(
    "PCC-CENTER THREE-WAY COMPARISON — ORACLE COMPONENTS"
)

print(
    "=========================================================================="
)

print(
    f"{'Diagnostic':<24}"
    f"{'baseline':>13}"
    f"{'v2-absolute':>15}"
    f"{'relative-only':>16}"
    f"{'rel-base':>13}"
)

print(
    "-" * 81
)


for key in bo:
    print(
        f"{key:<24}"
        f"{bo[key]:>13.4f}"
        f"{vo[key]:>15.4f}"
        f"{ro[key]:>16.4f}"
        f"{(ro[key] - bo[key]):>+13.4f}"
    )


# ============================================================
# COUNTERFACTUAL
# ============================================================

def cv(x):
    e = x["exact_gt_cell"]
    l = x["local_window_peak"]
    s = x["scene_level"]

    return {
        "CleanGTScoreMean":
            float(
                e[
                    "clean_score_stats"
                ]["mean"]
            ),

        "CompositeGTScoreMean":
            float(
                e[
                    "composite_score_stats"
                ]["mean"]
            ),

        "ExactLiftMean":
            float(
                e[
                    "composite_minus_clean_stats"
                ]["mean"]
            ),

        "ExactComp>Clean":
            float(
                e[
                    "fraction_composite_gt_clean"
                ]
            ),

        "ExactLift>=0.01":
            float(
                e[
                    "fraction_lift_ge_0_01"
                ]
            ),

        "LocalLiftMean":
            float(
                l[
                    "composite_minus_clean_stats"
                ]["mean"]
            ),

        "LocalComp>Clean":
            float(
                l[
                    "fraction_composite_gt_clean"
                ]
            ),

        "CleanMaxPeakMean":
            float(
                s[
                    "clean_max_peak_stats"
                ]["mean"]
            ),

        "OutsideMaxPeakMean":
            float(
                s[
                    "composite_outside_effect_max_peak_stats"
                ]["mean"]
            ),

        "EffectMaxPeakMean":
            float(
                s[
                    "composite_effect_max_peak_stats"
                ]["mean"]
            ),

        "CleanPeaks>=0.05":
            float(
                s[
                    "clean_mean_peak_count_by_threshold"
                ]["0.05"]
            ),

        "OutsidePeaks>=0.05":
            float(
                s[
                    "composite_outside_effect_mean_peak_count_by_threshold"
                ]["0.05"]
            ),
    }


bc = cv(base_cf)
vc = cv(v2_cf)
rc = cv(rel_cf)


print()
print(
    "=========================================================================="
)

print(
    "PCC-CENTER THREE-WAY COMPARISON — COUNTERFACTUAL RESPONSE"
)

print(
    "=========================================================================="
)

print(
    f"{'Diagnostic':<24}"
    f"{'baseline':>13}"
    f"{'v2-absolute':>15}"
    f"{'relative-only':>16}"
    f"{'rel-base':>13}"
)

print(
    "-" * 81
)


for key in bc:
    print(
        f"{key:<24}"
        f"{bc[key]:>13.4f}"
        f"{vc[key]:>15.4f}"
        f"{rc[key]:>16.4f}"
        f"{(rc[key] - bc[key]):>+13.4f}"
    )


# ============================================================
# SAVE RECORD
# ============================================================

record = {
    "phase":
        "5C",

    "paper_result":
        False,

    "experiment":
        "pcc_center_relative_only_1000_step_ablation",

    "scientific_question":
        (
            "Did the absolute hard-negative suppression terms "
            "cause the positive-response collapse observed in "
            "PCC-center v2?"
        ),

    "relative_only_change": {
        "lift_weight":
            0.25,

        "rank_weight":
            0.25,

        "clean_hard_negative_weight":
            0.0,

        "composite_outside_hard_negative_weight":
            0.0,
    },

    "baseline_metrics":
        bm,

    "absolute_v2_metrics":
        vm,

    "relative_only_metrics":
        rm,

    "baseline_oracle":
        bo,

    "absolute_v2_oracle":
        vo,

    "relative_only_oracle":
        ro,

    "baseline_counterfactual":
        bc,

    "absolute_v2_counterfactual":
        vc,

    "relative_only_counterfactual":
        rc,

    "particle_count_distribution":
        rel_metric[
            "sample_particle_count_distribution"
        ],

    "fruit_distribution":
        rel_metric[
            "sample_fruit_distribution"
        ],
}


record_path = (
    ROOT
    / "pcc_center_three_way_comparison.json"
)


record_path.write_text(
    json.dumps(
        record,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "Particle-count distribution:"
)

print(
    record[
        "particle_count_distribution"
    ]
)

print()
print(
    "Fruit distribution:"
)

print(
    record[
        "fruit_distribution"
    ]
)

print()
print(
    "Comparison JSON:"
)

print(
    record_path
)

print()
print(
    "=========================================================================="
)

print(
    "PHASE 5C RELATIVE-ONLY PCC-CENTER EVALUATION: PASS"
)

print(
    "=========================================================================="
)

print(
    "Do not launch longer training until this result is reviewed."
)

eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=100 op_pred=28 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=100 op_pred=28 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=100 op_pred=32 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=100 op_pred=46 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=100 op_pred=50 elapsed=0.5m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=100 op_pred=10 elapsed=0.6m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=99 op_pred=9 elapsed=0.7m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=99 op_pred=7 elapsed=0.8m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=97 op_pred=50 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=100 op_pred=23 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0d7548bc gt=0 decoded=99 op_pred=10 elapsed=1.1m
eval 0111/0200 scene=858baec99120971a3e47 gt=3 decoded=100 op_pred=1 elapsed=1.2m
eval 0121/0

In [49]:
%%writefile /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/launch_phase5c_checkpointed_baseline.py
from __future__ import annotations

import importlib.util
import shutil
import sys
from pathlib import Path

import torch


# ============================================================
# DASO-Net Phase 5C
# Checkpointed ORIGINAL-LOSS baseline launcher
#
# Does NOT modify:
#   architecture
#   InstanceSegLoss
#   PCCLoss
#   CRPALoss
#   renderer
#   datasets
#   base trainer
#
# Adds only:
#   1. AMP initial scale = 32
#   2. one compact checkpoint snapshot after each epoch
#
# Development infrastructure only.
# ============================================================


THIS_FILE = Path(__file__).resolve()

CODE_ROOT = THIS_FILE.parents[1]

BASE_TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)


if not BASE_TRAINER.exists():
    raise FileNotFoundError(
        f"Base trainer missing:\n{BASE_TRAINER}"
    )


def load_base_trainer():

    spec = importlib.util.spec_from_file_location(
        "phase5c_checkpointed_base",
        BASE_TRAINER,
    )

    if (
        spec is None
        or spec.loader is None
    ):
        raise RuntimeError(
            "Could not import base Phase-5B trainer."
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


base = load_base_trainer()


# ============================================================
# AMP scale 32
# ============================================================

_ORIGINAL_GRAD_SCALER = (
    torch.amp.GradScaler
)


class StableGradScaler(
    _ORIGINAL_GRAD_SCALER
):

    def __init__(
        self,
        *args,
        **kwargs,
    ):

        kwargs.setdefault(
            "init_scale",
            32.0,
        )

        super().__init__(
            *args,
            **kwargs,
        )


torch.amp.GradScaler = (
    StableGradScaler
)


# ============================================================
# Snapshot AFTER normal last.pt save.
#
# Base trainer calls save_checkpoint for:
#   last.pt
#   best.pt
#
# We snapshot ONLY last.pt so exactly one chronological
# checkpoint is retained per completed epoch.
# ============================================================

_ORIGINAL_SAVE_CHECKPOINT = (
    base.save_checkpoint
)


def snapshotting_save_checkpoint(
    path,
    bundle,
    optimizer,
    scheduler,
    scaler,
    epoch,
    optimizer_step,
    best_metric,
    cfg,
):

    _ORIGINAL_SAVE_CHECKPOINT(
        path,
        bundle,
        optimizer,
        scheduler,
        scaler,
        epoch,
        optimizer_step,
        best_metric,
        cfg,
    )

    path = Path(
        path
    )

    if path.name != "last.pt":
        return

    snapshot_dir = (
        path.parent
        / "snapshots"
    )

    snapshot_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    snapshot_path = (
        snapshot_dir
        / (
            f"epoch_{int(epoch):03d}"
            f"_opt_{int(optimizer_step):06d}.pt"
        )
    )

    shutil.copy2(
        path,
        snapshot_path,
    )

    print(
        "Phase 5C checkpoint snapshot:",
        snapshot_path,
        flush=True,
    )


base.save_checkpoint = (
    snapshotting_save_checkpoint
)


# ============================================================
# ENTRY
# ============================================================

if __name__ == "__main__":

    print(
        "============================================================"
    )

    print(
        "DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE"
    )

    print(
        "============================================================"
    )

    print(
        "Base trainer:"
    )

    print(
        BASE_TRAINER
    )

    print()

    print(
        "Model architecture change:  NO"
    )

    print(
        "Instance loss change:       NO"
    )

    print(
        "Original PCC change:        NO"
    )

    print(
        "CRPA change:                NO"
    )

    print(
        "PCC-center active:          NO"
    )

    print(
        "AMP initial scale:          32"
    )

    print(
        "Epoch snapshots:            YES"
    )

    print(
        "============================================================"
    )

    base.main()

Writing /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/launch_phase5c_checkpointed_baseline.py


In [50]:
# ============================================================
# PHASE 5C — CHECKPOINTED BASELINE LEARNING CURVE
#
# ORIGINAL model/loss system:
#   InstanceSegLoss
#   original feature/embedding PCC
#   CRPA
#
# NO PCC-center.
#
# 4 epochs
# 1000 train steps/rank/epoch
# accumulation = 4
#
# Snapshots:
#   250 optimizer updates
#   500 optimizer updates
#   750 optimizer updates
#   1000 optimizer updates
#
# Fresh initialization.
# ============================================================

import hashlib
import json
import shutil
import subprocess
import time
from pathlib import Path

import torch


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

BASE_CONFIG = Path(
    "/kaggle/working/"
    "phase5c_scheduleprobe1000_v1.json"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_baseline_curve1000updates_v1.json"
)

RUN_NAME = (
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

OUT = (
    Path(
        "/kaggle/working/"
        "dasonet_phase5b"
    )
    / RUN_NAME
)


def sha256(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


for required in (
    LAUNCHER,
    BASE_CONFIG,
    PROJECT_ROOT,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


# ------------------------------------------------------------
# Build exact original-loss configuration.
# ------------------------------------------------------------

cfg = json.loads(
    BASE_CONFIG.read_text(
        encoding="utf-8"
    )
)


cfg[
    "experiment_name"
] = RUN_NAME


# Absolutely remove development-only PCC-center config
# if it somehow appears in the source config.
cfg.setdefault(
    "loss",
    {}
)

cfg[
    "loss"
].pop(
    "pcc_center",
    None,
)


# ------------------------------------------------------------
# Frozen hardware/runtime setup.
# ------------------------------------------------------------

cfg["data"][
    "per_rank_synthetic_batch"
] = 1

cfg["data"][
    "per_rank_anchor_batch"
] = 2

cfg["data"][
    "synthetic_workers_per_rank"
] = 2

cfg["data"][
    "anchor_workers_per_rank"
] = 1

cfg["data"][
    "prefetch_factor"
] = 2


cfg["optimization"][
    "gradient_accumulation_steps"
] = 4

cfg["optimization"][
    "amp"
] = True

cfg["optimization"][
    "learning_rate"
] = 3e-4

cfg["optimization"][
    "warmup_optimizer_steps"
] = 50

# Constant 3e-4 after warmup.
cfg["optimization"][
    "min_lr_ratio"
] = 1.0


CONFIG.write_text(
    json.dumps(
        cfg,
        indent=2,
    ),
    encoding="utf-8",
)


if OUT.exists():
    shutil.rmtree(
        OUT
    )


print(
    "============================================================"
)

print(
    "PHASE 5C — ORIGINAL-PCC BASELINE LEARNING CURVE"
)

print(
    "============================================================"
)

print(
    "Launcher SHA256:"
)

print(
    sha256(
        LAUNCHER
    )
)

print()

print(
    "Config SHA256:"
)

print(
    sha256(
        CONFIG
    )
)

print()

print(
    "Epochs:                   4"
)

print(
    "Train steps/rank/epoch:   1000"
)

print(
    "Updates/epoch:            250"
)

print(
    "Total optimizer updates:  1000"
)

print(
    "Warm-up updates:          50"
)

print(
    "Post-warm-up LR:          3e-4 constant"
)

print(
    "Effective batch:          8"
)

print(
    "PCC-center:               DISABLED"
)

print()


cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(LAUNCHER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "4",

    "--max-train-steps",
    "1000",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    "none",
]


print(
    " ".join(
        cmd
    )
)

print()


start = time.time()


subprocess.run(
    cmd,
    check=True,
)


elapsed = (
    time.time()
    - start
)


status_path = (
    OUT
    / "run_status.json"
)

history_path = (
    OUT
    / "history.csv"
)

snapshot_dir = (
    OUT
    / "snapshots"
)


for required in (
    status_path,
    history_path,
    snapshot_dir,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing output:\n{required}"
        )


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:
    raise RuntimeError(
        f"Unexpected AMP skips: {amp_skips}"
    )


expected_snapshots = [
    snapshot_dir
    / "epoch_000_opt_000250.pt",

    snapshot_dir
    / "epoch_001_opt_000500.pt",

    snapshot_dir
    / "epoch_002_opt_000750.pt",

    snapshot_dir
    / "epoch_003_opt_001000.pt",
]


for path in expected_snapshots:

    if not path.exists():
        raise RuntimeError(
            f"Expected snapshot missing:\n{path}"
        )


last_ckpt = torch.load(
    expected_snapshots[-1],
    map_location="cpu",
    weights_only=False,
)


print()
print(
    "============================================================"
)

print(
    "BASELINE LEARNING-CURVE TRAINING COMPLETE"
)

print(
    "============================================================"
)

print(
    f"Wall time:                "
    f"{elapsed / 60.0:.2f} min"
)

print(
    "AMP skipped updates:      ",
    amp_skips,
)

print(
    "Final optimizer step:      ",
    last_ckpt.get(
        "optimizer_step"
    ),
)

print(
    "Final scaler scale:        ",
    last_ckpt.get(
        "scaler_state_dict",
        {},
    ).get(
        "scale"
    ),
)

print(
    "Best val total loss:       ",
    status.get(
        "best_val_total_loss"
    ),
)

print()

print(
    "Snapshots:"
)

for path in expected_snapshots:
    print(
        path
    )


print()
print(
    "========================================"
)

print(
    "PHASE 5C BASELINE 1000-UPDATES: PASS"
)

print(
    "========================================"
)

PHASE 5C — ORIGINAL-PCC BASELINE LEARNING CURVE
Launcher SHA256:
59d3768e507a2356d0ebc63652c5edc3ec32a63fb14612ffbd1690e82ddaafc9

Config SHA256:
a5f3671106ddcd09ccd20bb947ee97d3ed84dc168667015a250a7bedc9cef979

Epochs:                   4
Train steps/rank/epoch:   1000
Updates/epoch:            250
Total optimizer updates:  1000
Warm-up updates:          50
Post-warm-up LR:          3e-4 constant
Effective batch:          8
PCC-center:               DISABLED

torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/launch_phase5c_checkpointed_baseline.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_baseline_curve1000updates_v1.json --output-root /kaggle/working --epochs 4 --max-train-steps 1000 --max-val-steps 100 --accum 4 --num-workers 2 --log-every 100 --resume none



W0913 12:00:14.964000 2109 torch/distributed/run.py:852] 
W0913 12:00:14.964000 2109 torch/distributed/run.py:852] *****************************************
W0913 12:00:14.964000 2109 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 12:00:14.964000 2109 torch/distributed/run.py:852] *****************************************
[W913 12:00:15.336915953 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES


[W913 12:00:17.113379506 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 12:00:17.131575737 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      1000
Smoke mode:                 False
epoch=00 step=00000/01000 loss=27.7694 inst=27.1653 pcc=0.0000 crpa=0.6041 lr=6.00e-06 vram_alloc=8.84GB vram_res=9.10GB
epoch=00 step=00100/01000 loss=7.4272 inst=6.8451 pcc=0.0000 crpa=0.5820 lr=1.56e-04 vram_alloc=8.96GB vram_res=9.10GB
epoch=00 step=00200/01000 loss=0.4607 inst=0.1137 pcc=0.000

In [51]:
# ============================================================
# PHASE 5C — BASELINE LEARNING-CURVE EVALUATION
#
# Snapshots:
#   250 / 500 / 750 / 1000 optimizer updates
#
# SAME deterministic 200 synthetic validation scenes.
#
# Development evaluation only.
# Does NOT touch any external real-MP test set.
# ============================================================

import json
import shutil
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)


RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)


EVAL_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "baseline_curve1000updates_v1"
)


snapshots = {
    250:
        SNAPSHOT_DIR
        / "epoch_000_opt_000250.pt",

    500:
        SNAPSHOT_DIR
        / "epoch_001_opt_000500.pt",

    750:
        SNAPSHOT_DIR
        / "epoch_002_opt_000750.pt",

    1000:
        SNAPSHOT_DIR
        / "epoch_003_opt_001000.pt",
}


for path in (
    EVALUATOR,
    ORACLE,
    COUNTERFACTUAL,
    *snapshots.values(),
):
    if not path.exists():
        raise RuntimeError(
            f"Missing required path:\n{path}"
        )


if EVAL_ROOT.exists():
    shutil.rmtree(
        EVAL_ROOT
    )


EVAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


records = {}


for optimizer_step, checkpoint in snapshots.items():

    print()
    print(
        "============================================================"
    )

    print(
        f"EVALUATING BASELINE @ {optimizer_step} OPTIMIZER UPDATES"
    )

    print(
        "============================================================"
    )


    step_root = (
        EVAL_ROOT
        / f"opt_{optimizer_step:04d}"
    )

    metric_out = (
        step_root
        / "metrics"
    )

    oracle_out = (
        step_root
        / "oracle"
    )


    # --------------------------------------------------------
    # Exact instance metrics
    # --------------------------------------------------------

    subprocess.run(
        [
            sys.executable,
            str(EVALUATOR),

            "--project-root",
            str(PROJECT_ROOT),

            "--checkpoint",
            str(checkpoint),

            "--output-dir",
            str(metric_out),

            "--split",
            "val",

            "--max-scenes",
            "200",

            "--seed",
            "20260917",

            "--num-workers",
            "2",

            "--decode-min-score",
            "0.001",

            "--operating-score",
            "0.05",

            "--topk",
            "100",

            "--nms-iou",
            "0.5",

            "--mask-threshold",
            "0.5",

            "--match-iou",
            "0.5",
        ],
        check=True,
    )


    # --------------------------------------------------------
    # Oracle component diagnostic
    # --------------------------------------------------------

    subprocess.run(
        [
            sys.executable,
            str(ORACLE),

            "--project-root",
            str(PROJECT_ROOT),

            "--checkpoint",
            str(checkpoint),

            "--evaluator",
            str(EVALUATOR),

            "--output-dir",
            str(oracle_out),

            "--split",
            "val",

            "--max-scenes",
            "200",

            "--seed",
            "20260917",

            "--num-workers",
            "2",

            "--stride",
            "4",

            "--peak-kernel",
            "3",

            "--topk",
            "100",

            "--mask-threshold",
            "0.5",
        ],
        check=True,
    )


    metric = json.loads(
        (
            metric_out
            / "phase5c_metric_summary.json"
        ).read_text(
            encoding="utf-8"
        )
    )


    oracle = json.loads(
        (
            oracle_out
            / "oracle_component_diagnostic.json"
        ).read_text(
            encoding="utf-8"
        )
    )


    ap = metric[
        "mask_ap"
    ]

    op = metric[
        "operating_point"
    ]

    sweep = metric[
        "validation_threshold_sweep"
    ]

    center = oracle[
        "center_head"
    ]

    geom = oracle[
        "oracle_gt_cell_geometry"
    ]

    mask = oracle[
        "oracle_gt_cell_mask"
    ]

    empty = oracle[
        "empty_scene_center_behavior"
    ]


    records[
        optimizer_step
    ] = {
        "checkpoint":
            str(
                checkpoint
            ),

        "AP50":
            float(
                ap[
                    "mask_ap50"
                ]
            ),

        "AP75":
            float(
                ap[
                    "mask_ap75"
                ]
            ),

        "AP50_95":
            float(
                ap[
                    "mask_ap50_95"
                ]
            ),

        "BestF1":
            float(
                sweep[
                    "best_f1"
                ]
            ),

        "BestThreshold":
            float(
                sweep[
                    "best_threshold_by_f1"
                ]
            ),

        "CountMAE_at_005":
            float(
                op[
                    "count_mae"
                ]
            ),

        "EmptyFP_at_005":
            float(
                op[
                    "empty_scene_fp_per_image"
                ]
            ),

        "PeakRecall4":
            float(
                center[
                    "top100_peak_recall_within_4px"
                ]
            ),

        "PeakRecall8":
            float(
                center[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall16":
            float(
                center[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "GTCenterScore":
            float(
                center[
                    "gt_cell_score_stats"
                ][
                    "mean"
                ]
            ),

        "BoxIoUMean":
            float(
                geom[
                    "box_iou_stats"
                ][
                    "mean"
                ]
            ),

        "WidthRatioMedian":
            float(
                geom[
                    "pred_to_gt_width_ratio_stats"
                ][
                    "median"
                ]
            ),

        "HeightRatioMedian":
            float(
                geom[
                    "pred_to_gt_height_ratio_stats"
                ][
                    "median"
                ]
            ),

        "GTBoxMaskIoUMean":
            float(
                mask[
                    "gt_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),

        "GTBoxMaskIoU50":
            float(
                mask[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),

        "EmptyMaxPeak":
            float(
                empty[
                    "max_peak_score_stats"
                ][
                    "mean"
                ]
            ),

        "EmptyPeaks010":
            float(
                empty[
                    "mean_local_peak_count_by_threshold"
                ][
                    "0.10"
                ]
            ),
    }


# ============================================================
# Verify frozen sample identity across snapshots.
# ============================================================

base_distribution = None
base_fruit_distribution = None


for optimizer_step in snapshots:

    metric_path = (
        EVAL_ROOT
        / f"opt_{optimizer_step:04d}"
        / "metrics"
        / "phase5c_metric_summary.json"
    )

    metric = json.loads(
        metric_path.read_text(
            encoding="utf-8"
        )
    )

    pc = metric[
        "sample_particle_count_distribution"
    ]

    fr = metric[
        "sample_fruit_distribution"
    ]


    if base_distribution is None:
        base_distribution = pc
        base_fruit_distribution = fr

    if pc != base_distribution:
        raise RuntimeError(
            "Particle distribution mismatch across checkpoints."
        )

    if fr != base_fruit_distribution:
        raise RuntimeError(
            "Fruit distribution mismatch across checkpoints."
        )


# ============================================================
# Learning-curve table
# ============================================================

print()
print(
    "=========================================================================================="
)

print(
    "PHASE 5C — ORIGINAL-PCC BASELINE LEARNING CURVE"
)

print(
    "=========================================================================================="
)

print(
    f"{'Updates':>8}"
    f"{'AP50':>10}"
    f"{'AP50:95':>11}"
    f"{'BestF1':>10}"
    f"{'PeakR@8':>10}"
    f"{'BoxIoU':>10}"
    f"{'MaskIoU':>10}"
    f"{'EmptyFP@.05':>14}"
)

print(
    "-" * 83
)


for step in sorted(
    records
):

    r = records[
        step
    ]

    print(
        f"{step:>8d}"
        f"{r['AP50']:>10.4f}"
        f"{r['AP50_95']:>11.4f}"
        f"{r['BestF1']:>10.4f}"
        f"{r['PeakRecall8']:>10.4f}"
        f"{r['BoxIoUMean']:>10.4f}"
        f"{r['GTBoxMaskIoUMean']:>10.4f}"
        f"{r['EmptyFP_at_005']:>14.2f}"
    )


print()
print(
    "------------------------------------------------------------------------------------------"
)

print(
    f"{'Updates':>8}"
    f"{'GTScore':>10}"
    f"{'PeakR@4':>10}"
    f"{'PeakR@16':>10}"
    f"{'W ratio':>10}"
    f"{'H ratio':>10}"
    f"{'Mask>=.5':>11}"
    f"{'EmptyPk>.1':>13}"
)

print(
    "-" * 83
)


for step in sorted(
    records
):

    r = records[
        step
    ]

    print(
        f"{step:>8d}"
        f"{r['GTCenterScore']:>10.4f}"
        f"{r['PeakRecall4']:>10.4f}"
        f"{r['PeakRecall16']:>10.4f}"
        f"{r['WidthRatioMedian']:>10.4f}"
        f"{r['HeightRatioMedian']:>10.4f}"
        f"{r['GTBoxMaskIoU50']:>11.4f}"
        f"{r['EmptyPeaks010']:>13.2f}"
    )


# ============================================================
# Provisional development checkpoint selection.
#
# Primary: AP50:95
# Secondary: BestF1
#
# This is ONLY on the 200-scene synthetic validation diagnostic.
# It is not final checkpoint selection.
# ============================================================

best_step = max(
    records,
    key=lambda step: (
        records[
            step
        ][
            "AP50_95"
        ],
        records[
            step
        ][
            "BestF1"
        ],
    ),
)


best_checkpoint = Path(
    records[
        best_step
    ][
        "checkpoint"
    ]
)


print()
print(
    "Provisional best development checkpoint:"
)

print(
    f"  optimizer updates: {best_step}"
)

print(
    f"  AP50:95:           "
    f"{records[best_step]['AP50_95']:.4f}"
)

print(
    f"  BestF1:            "
    f"{records[best_step]['BestF1']:.4f}"
)

print(
    f"  checkpoint:        {best_checkpoint}"
)


# ============================================================
# Counterfactual diagnostic on provisional best checkpoint.
# ============================================================

cf_out = (
    EVAL_ROOT
    / "best_counterfactual"
)


subprocess.run(
    [
        sys.executable,
        str(COUNTERFACTUAL),

        "--project-root",
        str(PROJECT_ROOT),

        "--checkpoint",
        str(best_checkpoint),

        "--evaluator",
        str(EVALUATOR),

        "--output-dir",
        str(cf_out),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--stride",
        "4",

        "--peak-kernel",
        "3",

        "--window-radius-feature",
        "2",
    ],
    check=True,
)


cf = json.loads(
    (
        cf_out
        / "counterfactual_center_diagnostic.json"
    ).read_text(
        encoding="utf-8"
    )
)


print()
print(
    "============================================================"
)

print(
    "BEST-CHECKPOINT COUNTERFACTUAL SUMMARY"
)

print(
    "============================================================"
)

print(
    "Exact GT-cell lift mean:",
    f"{cf['exact_gt_cell']['composite_minus_clean_stats']['mean']:+.4f}",
)

print(
    "Composite > clean:",
    f"{cf['exact_gt_cell']['fraction_composite_gt_clean']:.4f}",
)

print(
    "Local peak lift mean:",
    f"{cf['local_window_peak']['composite_minus_clean_stats']['mean']:+.4f}",
)

print(
    "Clean max peak mean:",
    f"{cf['scene_level']['clean_max_peak_stats']['mean']:.4f}",
)

print(
    "Effect max peak mean:",
    f"{cf['scene_level']['composite_effect_max_peak_stats']['mean']:.4f}",
)


# ============================================================
# Save complete learning-curve record.
# ============================================================

record = {
    "phase":
        "5C",

    "paper_result":
        False,

    "experiment":
        "original_pcc_baseline_learning_curve_1000_updates",

    "pcc_center":
        "rejected_and_disabled",

    "optimizer_update_checkpoints":
        records,

    "provisional_best_optimizer_step":
        int(
            best_step
        ),

    "provisional_best_checkpoint":
        str(
            best_checkpoint
        ),

    "selection_rule":
        (
            "200-scene synthetic validation diagnostic: "
            "maximize AP50:95, BestF1 secondary"
        ),

    "particle_count_distribution":
        base_distribution,

    "fruit_distribution":
        base_fruit_distribution,

    "best_checkpoint_counterfactual":
        cf,
}


record_path = (
    EVAL_ROOT
    / "baseline_learning_curve_summary.json"
)


record_path.write_text(
    json.dumps(
        record,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "Summary JSON:"
)

print(
    record_path
)

print()
print(
    "========================================"
)

print(
    "PHASE 5C BASELINE LEARNING CURVE: PASS"
)

print(
    "========================================"
)

print(
    "Do not start >1000 optimizer updates until this curve is reviewed."
)


EVALUATING BASELINE @ 250 OPTIMIZER UPDATES
eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=99 op_pred=99 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=100 op_pred=100 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=100 op_pred=100 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=100 op_pred=100 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=100 op_pred=100 elapsed=0.5m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=100 op_pred=36 elapsed=0.6m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=96 op_pred=48 elapsed=0.7m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=100 op_pred=54 elapsed=0.8m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=99 op_pred=99 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=100 op_pred=81 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0d7548bc gt=0 decoded=100 op_pred=100 elapsed=1.1m
eval 0111/0200 scene=858baec99120971a3e4

In [53]:
%%writefile /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/launch_phase5c_checkpointed_baseline.py
from __future__ import annotations

import importlib.util
import shutil
import sys
from pathlib import Path

import torch
from torch.nn.parallel import DistributedDataParallel as DDP


# ============================================================
# DASO-Net Phase 5C
# CHECKPOINTED ORIGINAL-PCC BASELINE LAUNCHER v2
#
# COMPLETE replacement.
#
# Purpose
# -------
# Preserve the hardened Phase-5B trainer while adding only:
#
#   1. AMP initial scale = 32
#   2. chronological epoch snapshots
#   3. PyTorch 2.6-compatible trusted checkpoint resume
#
# Scientific model/training components remain unchanged:
#
#   architecture     : unchanged
#   InstanceSegLoss  : unchanged
#   PCCLoss          : unchanged
#   CRPALoss         : unchanged
#   renderer         : unchanged
#   datasets         : unchanged
#
# Why load with weights_only=False?
# ---------------------------------
# DASO-Net checkpoints contain:
#   - model states
#   - optimizer state
#   - scheduler state
#   - GradScaler state
#   - configuration
#   - RNG metadata
#
# PyTorch 2.6 changed torch.load() default weights_only=True.
# These files are generated locally by our own trusted training
# pipeline, therefore full checkpoint deserialization is required.
# ============================================================


THIS_FILE = Path(__file__).resolve()

CODE_ROOT = THIS_FILE.parents[1]

BASE_TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)


if not BASE_TRAINER.exists():
    raise FileNotFoundError(
        f"Base trainer missing:\n{BASE_TRAINER}"
    )


# ============================================================
# Import current hardened Phase-5B trainer.
# ============================================================

def load_base_trainer():

    spec = importlib.util.spec_from_file_location(
        "phase5c_checkpointed_base",
        BASE_TRAINER,
    )

    if (
        spec is None
        or spec.loader is None
    ):
        raise RuntimeError(
            "Could not import hardened Phase-5B trainer."
        )

    module = importlib.util.module_from_spec(
        spec
    )

    spec.loader.exec_module(
        module
    )

    return module


base = load_base_trainer()


# ============================================================
# PyTorch 2.6-compatible trusted checkpoint loader.
#
# This reproduces the CURRENT Phase-5B load_checkpoint()
# semantics exactly, except that weights_only=False is explicit.
#
# We intentionally do NOT alter optimizer/scheduler/RNG behavior
# beyond fixing checkpoint deserialization.
# ============================================================

def trusted_load_checkpoint(
    path: Path,
    bundle: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    scaler: torch.amp.GradScaler,
    device: torch.device,
) -> tuple[int, int, float]:

    path = Path(
        path
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Resume checkpoint missing:\n{path}"
        )

    # IMPORTANT:
    # This checkpoint was produced by our own DASO-Net run.
    # Full deserialization is required for optimizer/scheduler/
    # scaler metadata under PyTorch >= 2.6.
    payload = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )

    module = (
        bundle.module
        if isinstance(
            bundle,
            DDP,
        )
        else bundle
    )

    module.load_state_dict(
        payload[
            "bundle_state_dict"
        ],
        strict=True,
    )

    optimizer.load_state_dict(
        payload[
            "optimizer_state_dict"
        ]
    )

    scheduler.load_state_dict(
        payload[
            "scheduler_state_dict"
        ]
    )

    scaler.load_state_dict(
        payload.get(
            "scaler_state_dict",
            {},
        )
    )

    start_epoch = (
        int(
            payload[
                "epoch"
            ]
        )
        + 1
    )

    optimizer_step = int(
        payload.get(
            "optimizer_step",
            0,
        )
    )

    best_metric = float(
        payload.get(
            "best_metric",
            float(
                "inf"
            ),
        )
    )

    return (
        start_epoch,
        optimizer_step,
        best_metric,
    )


# Install corrected checkpoint loader into the imported trainer.
base.load_checkpoint = (
    trusted_load_checkpoint
)


# ============================================================
# AMP initial scale = 32.
#
# This preserves the already-validated T4 setup.
# ============================================================

_ORIGINAL_GRAD_SCALER = (
    torch.amp.GradScaler
)


class StableGradScaler(
    _ORIGINAL_GRAD_SCALER
):

    def __init__(
        self,
        *args,
        **kwargs,
    ):

        kwargs.setdefault(
            "init_scale",
            32.0,
        )

        super().__init__(
            *args,
            **kwargs,
        )


torch.amp.GradScaler = (
    StableGradScaler
)


# ============================================================
# Epoch snapshotting.
#
# The hardened trainer normally writes:
#   last.pt
#   best.pt
#
# Whenever last.pt is written after an epoch, preserve a compact
# chronological snapshot:
#
#   snapshots/
#       epoch_000_opt_000250.pt
#       epoch_001_opt_000500.pt
#       ...
# ============================================================

_ORIGINAL_SAVE_CHECKPOINT = (
    base.save_checkpoint
)


def snapshotting_save_checkpoint(
    path: Path,
    bundle: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    scaler: torch.amp.GradScaler,
    epoch: int,
    optimizer_step: int,
    best_metric: float,
    cfg: dict,
) -> None:

    _ORIGINAL_SAVE_CHECKPOINT(
        path,
        bundle,
        optimizer,
        scheduler,
        scaler,
        epoch,
        optimizer_step,
        best_metric,
        cfg,
    )

    path = Path(
        path
    )

    # Snapshot only the chronological last.pt, not best.pt.
    if path.name != "last.pt":
        return

    snapshot_dir = (
        path.parent
        / "snapshots"
    )

    snapshot_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    snapshot_path = (
        snapshot_dir
        / (
            f"epoch_{int(epoch):03d}"
            f"_opt_{int(optimizer_step):06d}.pt"
        )
    )

    shutil.copy2(
        path,
        snapshot_path,
    )

    print(
        "Phase 5C checkpoint snapshot:",
        snapshot_path,
        flush=True,
    )


base.save_checkpoint = (
    snapshotting_save_checkpoint
)


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":

    print(
        "============================================================"
    )

    print(
        "DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2"
    )

    print(
        "============================================================"
    )

    print(
        "Base trainer:"
    )

    print(
        BASE_TRAINER
    )

    print()

    print(
        "Model architecture change:  NO"
    )

    print(
        "Instance loss change:       NO"
    )

    print(
        "Original PCC change:        NO"
    )

    print(
        "CRPA change:                NO"
    )

    print(
        "PCC-center active:          NO"
    )

    print(
        "AMP initial scale:          32"
    )

    print(
        "Epoch snapshots:            YES"
    )

    print(
        "PyTorch 2.6 resume fix:     ENABLED"
    )

    print(
        "Checkpoint trust scope:     locally generated DASO-Net files"
    )

    print(
        "============================================================"
    )

    base.main()

Overwriting /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/launch_phase5c_checkpointed_baseline.py


In [54]:
# ============================================================
# PHASE 5C — CONTINUE ORIGINAL-PCC BASELINE
#
# Resume safely from:
#   epoch 3 / optimizer update 1000
#
# Continue through:
#   epoch 4 -> 1250 updates
#   epoch 5 -> 1500 updates
#
# Uses checkpointed launcher v2 with explicit
# torch.load(..., weights_only=False).
#
# The failed previous attempt performed ZERO optimizer updates.
# Existing 250/500/750/1000 snapshots are preserved.
# ============================================================

import json
import subprocess
import sys
import time
from pathlib import Path

import torch


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

CONFIG = Path(
    "/kaggle/working/"
    "phase5c_baseline_curve1000updates_v1.json"
)

RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

RESUME = (
    RUN_ROOT
    / "snapshots"
    / "epoch_003_opt_001000.pt"
)

SNAP_1250 = (
    RUN_ROOT
    / "snapshots"
    / "epoch_004_opt_001250.pt"
)

SNAP_1500 = (
    RUN_ROOT
    / "snapshots"
    / "epoch_005_opt_001500.pt"
)


for required in (
    PROJECT_ROOT,
    LAUNCHER,
    CONFIG,
    RESUME,
):
    if not required.exists():
        raise RuntimeError(
            f"Missing required path:\n{required}"
        )


# ------------------------------------------------------------
# Verify launcher replacement.
# ------------------------------------------------------------

launcher_text = LAUNCHER.read_text(
    encoding="utf-8"
)


required_markers = [
    "CHECKPOINTED ORIGINAL-PCC BASELINE LAUNCHER v2",
    "def trusted_load_checkpoint(",
    "weights_only=False",
    "base.load_checkpoint =",
    "snapshotting_save_checkpoint",
]


missing = [
    marker
    for marker in required_markers
    if marker not in launcher_text
]


if missing:
    raise RuntimeError(
        "Corrected launcher v2 verification failed.\n"
        f"Missing markers: {missing}"
    )


# Syntax validation.
subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(LAUNCHER),
    ],
    check=True,
)


# ------------------------------------------------------------
# Verify trusted resume checkpoint.
# ------------------------------------------------------------

resume_ckpt = torch.load(
    RESUME,
    map_location="cpu",
    weights_only=False,
)


resume_epoch = int(
    resume_ckpt.get(
        "epoch",
        -1,
    )
)

resume_step = int(
    resume_ckpt.get(
        "optimizer_step",
        -1,
    )
)


if resume_epoch != 3:
    raise RuntimeError(
        f"Expected resume epoch 3, got {resume_epoch}."
    )


if resume_step != 1000:
    raise RuntimeError(
        f"Expected optimizer step 1000, got {resume_step}."
    )


# ------------------------------------------------------------
# Do NOT delete the run directory.
#
# Existing history + snapshots must be retained.
# ------------------------------------------------------------

print(
    "============================================================"
)

print(
    "PHASE 5C — CONTINUING BASELINE 1000 -> 1500 UPDATES"
)

print(
    "============================================================"
)

print(
    "Launcher v2 verification: PASS"
)

print(
    "Python syntax:             PASS"
)

print(
    "Resume checkpoint load:    PASS"
)

print()

print(
    "Resume checkpoint:"
)

print(
    RESUME
)

print()

print(
    "Resume optimizer step:",
    resume_step,
)

print(
    "Resume epoch:",
    resume_epoch,
)

print(
    "Next epochs: 4, 5"
)

print(
    "Updates per epoch: 250"
)

print(
    "Final update target: 1500"
)

print(
    "PCC-center: DISABLED"
)

print()


cmd = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",

    str(LAUNCHER),

    "--project-root",
    str(PROJECT_ROOT),

    "--config",
    str(CONFIG),

    "--output-root",
    "/kaggle/working",

    "--epochs",
    "6",

    "--max-train-steps",
    "1000",

    "--max-val-steps",
    "100",

    "--accum",
    "4",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    str(RESUME),
]


print(
    " ".join(
        cmd
    )
)

print()


start = time.time()


subprocess.run(
    cmd,
    check=True,
)


elapsed = (
    time.time()
    - start
)


# ------------------------------------------------------------
# Validate continuation outputs.
# ------------------------------------------------------------

for expected in (
    SNAP_1250,
    SNAP_1500,
):
    if not expected.exists():
        raise RuntimeError(
            f"Expected snapshot missing:\n{expected}"
        )


ckpt_1250 = torch.load(
    SNAP_1250,
    map_location="cpu",
    weights_only=False,
)

ckpt_1500 = torch.load(
    SNAP_1500,
    map_location="cpu",
    weights_only=False,
)


if int(
    ckpt_1250.get(
        "optimizer_step",
        -1,
    )
) != 1250:
    raise RuntimeError(
        "1250 checkpoint has wrong optimizer step."
    )


if int(
    ckpt_1500.get(
        "optimizer_step",
        -1,
    )
) != 1500:
    raise RuntimeError(
        "1500 checkpoint has wrong optimizer step."
    )


status_path = (
    RUN_ROOT
    / "run_status.json"
)


if not status_path.exists():
    raise RuntimeError(
        "run_status.json missing after continuation."
    )


status = json.loads(
    status_path.read_text(
        encoding="utf-8"
    )
)


amp_skips = int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if amp_skips != 0:
    raise RuntimeError(
        f"Unexpected AMP skips: {amp_skips}"
    )


print()
print(
    "============================================================"
)

print(
    "BASELINE CONTINUATION COMPLETE"
)

print(
    "============================================================"
)

print(
    f"Continuation wall time: "
    f"{elapsed / 60.0:.2f} min"
)

print(
    "1250 checkpoint optimizer step:",
    ckpt_1250[
        "optimizer_step"
    ],
)

print(
    "1500 checkpoint optimizer step:",
    ckpt_1500[
        "optimizer_step"
    ],
)

print(
    "Final scaler scale:",
    ckpt_1500.get(
        "scaler_state_dict",
        {},
    ).get(
        "scale"
    ),
)

print(
    "AMP skipped updates:",
    amp_skips,
)

print()

print(
    "1250 checkpoint:"
)

print(
    SNAP_1250
)

print()

print(
    "1500 checkpoint:"
)

print(
    SNAP_1500
)

print()
print(
    "========================================"
)

print(
    "PHASE 5C BASELINE 1500-UPDATES: PASS"
)

print(
    "========================================"
)

PHASE 5C — CONTINUING BASELINE 1000 -> 1500 UPDATES
Launcher v2 verification: PASS
Python syntax:             PASS
Resume checkpoint load:    PASS

Resume checkpoint:
/kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapshots/epoch_003_opt_001000.pt

Resume optimizer step: 1000
Resume epoch: 3
Next epochs: 4, 5
Updates per epoch: 250
Final update target: 1500
PCC-center: DISABLED

torchrun --standalone --nproc_per_node=2 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/launch_phase5c_checkpointed_baseline.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A --config /kaggle/working/phase5c_baseline_curve1000updates_v1.json --output-root /kaggle/working --epochs 6 --max-train-steps 1000 --max-val-steps 100 --accum 4 --num-workers 2 --log-every 100 --resume /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapshots/epoch_003_opt_001000.pt



W0913 13:47:07.577000 2550 torch/distributed/run.py:852] 
W0913 13:47:07.577000 2550 torch/distributed/run.py:852] *****************************************
W0913 13:47:07.577000 2550 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0913 13:47:07.577000 2550 torch/distributed/run.py:852] *****************************************
[W913 13:47:07.210825718 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files
DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files


[W913 13:47:10.976984629 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W913 13:47:10.996345511 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


Resumed from: /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapshots/epoch_003_opt_001000.pt
=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      1500
Smoke mode:                 False
epoch=04 step=00000/01000 loss=2.1859 inst=2.1307 pcc=0.0000 crpa=0.0551 lr=3.00e-04 vram_alloc=8.85GB vram_res=9.12GB
epoch=04 step=00100/01000 loss=0.3096 inst=0.2283 pcc=0

In [55]:
# ============================================================
# PHASE 5C — EXTENDED BASELINE LEARNING CURVE
#
# Existing:
#   250 / 500 / 750 / 1000
#
# New:
#   1250 / 1500
#
# SAME frozen deterministic 200 validation scenes.
# ============================================================

import json
import subprocess
import sys
from pathlib import Path


CODE_ROOT = Path(
    "/kaggle/working/"
    "DASO-Net-Phase5B-KaggleTraining"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)


RUN_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5b/"
    "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

EVAL_ROOT = Path(
    "/kaggle/working/"
    "dasonet_phase5c/"
    "baseline_curve1000updates_v1"
)


NEW_SNAPSHOTS = {
    1250:
        SNAPSHOT_DIR
        / "epoch_004_opt_001250.pt",

    1500:
        SNAPSHOT_DIR
        / "epoch_005_opt_001500.pt",
}


for path in (
    EVALUATOR,
    ORACLE,
    COUNTERFACTUAL,
    *NEW_SNAPSHOTS.values(),
):
    if not path.exists():
        raise RuntimeError(
            f"Missing:\n{path}"
        )


def evaluate_snapshot(
    optimizer_step,
    checkpoint,
):

    print()
    print(
        "============================================================"
    )

    print(
        f"EVALUATING BASELINE @ {optimizer_step} OPTIMIZER UPDATES"
    )

    print(
        "============================================================"
    )


    step_root = (
        EVAL_ROOT
        / f"opt_{optimizer_step:04d}"
    )

    metric_out = (
        step_root
        / "metrics"
    )

    oracle_out = (
        step_root
        / "oracle"
    )


    subprocess.run(
        [
            sys.executable,
            str(EVALUATOR),

            "--project-root",
            str(PROJECT_ROOT),

            "--checkpoint",
            str(checkpoint),

            "--output-dir",
            str(metric_out),

            "--split",
            "val",

            "--max-scenes",
            "200",

            "--seed",
            "20260917",

            "--num-workers",
            "2",

            "--decode-min-score",
            "0.001",

            "--operating-score",
            "0.05",

            "--topk",
            "100",

            "--nms-iou",
            "0.5",

            "--mask-threshold",
            "0.5",

            "--match-iou",
            "0.5",
        ],
        check=True,
    )


    subprocess.run(
        [
            sys.executable,
            str(ORACLE),

            "--project-root",
            str(PROJECT_ROOT),

            "--checkpoint",
            str(checkpoint),

            "--evaluator",
            str(EVALUATOR),

            "--output-dir",
            str(oracle_out),

            "--split",
            "val",

            "--max-scenes",
            "200",

            "--seed",
            "20260917",

            "--num-workers",
            "2",

            "--stride",
            "4",

            "--peak-kernel",
            "3",

            "--topk",
            "100",

            "--mask-threshold",
            "0.5",
        ],
        check=True,
    )


for step, checkpoint in NEW_SNAPSHOTS.items():

    evaluate_snapshot(
        step,
        checkpoint,
    )


# ============================================================
# Load all six checkpoints' metrics.
# ============================================================

steps = [
    250,
    500,
    750,
    1000,
    1250,
    1500,
]


records = {}


for step in steps:

    root = (
        EVAL_ROOT
        / f"opt_{step:04d}"
    )

    metric = json.loads(
        (
            root
            / "metrics"
            / "phase5c_metric_summary.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    oracle = json.loads(
        (
            root
            / "oracle"
            / "oracle_component_diagnostic.json"
        ).read_text(
            encoding="utf-8"
        )
    )


    ap = metric[
        "mask_ap"
    ]

    op = metric[
        "operating_point"
    ]

    sweep = metric[
        "validation_threshold_sweep"
    ]

    center = oracle[
        "center_head"
    ]

    geom = oracle[
        "oracle_gt_cell_geometry"
    ]

    mask = oracle[
        "oracle_gt_cell_mask"
    ]


    records[
        step
    ] = {
        "AP50":
            float(
                ap[
                    "mask_ap50"
                ]
            ),

        "AP75":
            float(
                ap[
                    "mask_ap75"
                ]
            ),

        "AP50_95":
            float(
                ap[
                    "mask_ap50_95"
                ]
            ),

        "BestF1":
            float(
                sweep[
                    "best_f1"
                ]
            ),

        "BestThreshold":
            float(
                sweep[
                    "best_threshold_by_f1"
                ]
            ),

        "PeakRecall8":
            float(
                center[
                    "top100_peak_recall_within_8px"
                ]
            ),

        "PeakRecall16":
            float(
                center[
                    "top100_peak_recall_within_16px"
                ]
            ),

        "GTCenterScore":
            float(
                center[
                    "gt_cell_score_stats"
                ][
                    "mean"
                ]
            ),

        "BoxIoUMean":
            float(
                geom[
                    "box_iou_stats"
                ][
                    "mean"
                ]
            ),

        "WidthRatio":
            float(
                geom[
                    "pred_to_gt_width_ratio_stats"
                ][
                    "median"
                ]
            ),

        "HeightRatio":
            float(
                geom[
                    "pred_to_gt_height_ratio_stats"
                ][
                    "median"
                ]
            ),

        "MaskIoUMean":
            float(
                mask[
                    "gt_box_crop_mask_iou_stats"
                ][
                    "mean"
                ]
            ),

        "MaskIoU50":
            float(
                mask[
                    "gt_box_crop_mask_iou_ge_0_50"
                ]
            ),

        "EmptyFP005":
            float(
                op[
                    "empty_scene_fp_per_image"
                ]
            ),
    }


print()
print(
    "=============================================================================================="
)

print(
    "PHASE 5C — EXTENDED ORIGINAL-PCC BASELINE LEARNING CURVE"
)

print(
    "=============================================================================================="
)

print(
    f"{'Updates':>8}"
    f"{'AP50':>10}"
    f"{'AP75':>10}"
    f"{'AP50:95':>11}"
    f"{'BestF1':>10}"
    f"{'PeakR@8':>10}"
    f"{'BoxIoU':>10}"
    f"{'MaskIoU':>10}"
)

print(
    "-" * 79
)


for step in steps:

    r = records[
        step
    ]

    print(
        f"{step:>8d}"
        f"{r['AP50']:>10.4f}"
        f"{r['AP75']:>10.4f}"
        f"{r['AP50_95']:>11.4f}"
        f"{r['BestF1']:>10.4f}"
        f"{r['PeakRecall8']:>10.4f}"
        f"{r['BoxIoUMean']:>10.4f}"
        f"{r['MaskIoUMean']:>10.4f}"
    )


print()
print(
    "----------------------------------------------------------------------------------------------"
)

print(
    f"{'Updates':>8}"
    f"{'GTScore':>10}"
    f"{'PeakR@16':>11}"
    f"{'W ratio':>10}"
    f"{'H ratio':>10}"
    f"{'Mask>=.5':>11}"
    f"{'BestThr':>10}"
    f"{'FP@.05':>10}"
)

print(
    "-" * 80
)


for step in steps:

    r = records[
        step
    ]

    print(
        f"{step:>8d}"
        f"{r['GTCenterScore']:>10.4f}"
        f"{r['PeakRecall16']:>11.4f}"
        f"{r['WidthRatio']:>10.4f}"
        f"{r['HeightRatio']:>10.4f}"
        f"{r['MaskIoU50']:>11.4f}"
        f"{r['BestThreshold']:>10.2f}"
        f"{r['EmptyFP005']:>10.2f}"
    )


# ============================================================
# Provisional best across all six snapshots.
# ============================================================

best_step = max(
    steps,
    key=lambda step: (
        records[
            step
        ][
            "AP50_95"
        ],
        records[
            step
        ][
            "BestF1"
        ],
    ),
)


best_checkpoint = (
    SNAPSHOT_DIR
    / (
        {
            250:
                "epoch_000_opt_000250.pt",

            500:
                "epoch_001_opt_000500.pt",

            750:
                "epoch_002_opt_000750.pt",

            1000:
                "epoch_003_opt_001000.pt",

            1250:
                "epoch_004_opt_001250.pt",

            1500:
                "epoch_005_opt_001500.pt",
        }[
            best_step
        ]
    )
)


print()
print(
    "Provisional best checkpoint:"
)

print(
    "  optimizer updates:",
    best_step,
)

print(
    "  AP50:95:",
    f"{records[best_step]['AP50_95']:.4f}",
)

print(
    "  BestF1:",
    f"{records[best_step]['BestF1']:.4f}",
)

print(
    "  checkpoint:",
    best_checkpoint,
)


# ============================================================
# Counterfactual diagnostic on new provisional best.
# ============================================================

cf_out = (
    EVAL_ROOT
    / "extended_best_counterfactual"
)


subprocess.run(
    [
        sys.executable,
        str(COUNTERFACTUAL),

        "--project-root",
        str(PROJECT_ROOT),

        "--checkpoint",
        str(best_checkpoint),

        "--evaluator",
        str(EVALUATOR),

        "--output-dir",
        str(cf_out),

        "--split",
        "val",

        "--max-scenes",
        "200",

        "--seed",
        "20260917",

        "--num-workers",
        "2",

        "--stride",
        "4",

        "--peak-kernel",
        "3",

        "--window-radius-feature",
        "2",
    ],
    check=True,
)


cf = json.loads(
    (
        cf_out
        / "counterfactual_center_diagnostic.json"
    ).read_text(
        encoding="utf-8"
    )
)


print()
print(
    "============================================================"
)

print(
    "EXTENDED-BEST COUNTERFACTUAL SUMMARY"
)

print(
    "============================================================"
)

print(
    "Exact GT-cell lift mean:",
    f"{cf['exact_gt_cell']['composite_minus_clean_stats']['mean']:+.4f}",
)

print(
    "Composite > clean:",
    f"{cf['exact_gt_cell']['fraction_composite_gt_clean']:.4f}",
)

print(
    "Local peak lift mean:",
    f"{cf['local_window_peak']['composite_minus_clean_stats']['mean']:+.4f}",
)

print(
    "Clean max peak mean:",
    f"{cf['scene_level']['clean_max_peak_stats']['mean']:.4f}",
)

print(
    "Effect max peak mean:",
    f"{cf['scene_level']['composite_effect_max_peak_stats']['mean']:.4f}",
)


record = {
    "phase":
        "5C",

    "paper_result":
        False,

    "experiment":
        "original_pcc_baseline_extended_to_1500_updates",

    "records":
        records,

    "provisional_best_step":
        best_step,

    "provisional_best_checkpoint":
        str(
            best_checkpoint
        ),

    "counterfactual":
        cf,
}


summary_path = (
    EVAL_ROOT
    / "baseline_learning_curve_1500_summary.json"
)


summary_path.write_text(
    json.dumps(
        record,
        indent=2,
    ),
    encoding="utf-8",
)


print()
print(
    "Summary:"
)

print(
    summary_path
)

print()
print(
    "========================================"
)

print(
    "PHASE 5C EXTENDED LEARNING CURVE: PASS"
)

print(
    "========================================"
)

print(
    "Stop training here for today."
)


EVALUATING BASELINE @ 1250 OPTIMIZER UPDATES
eval 0001/0200 scene=369b96d7adde8c0026a7 gt=1 decoded=100 op_pred=46 elapsed=0.1m
eval 0011/0200 scene=8aab26301d02d13c97c2 gt=3 decoded=100 op_pred=38 elapsed=0.2m
eval 0021/0200 scene=d7efc64a9774dd5c9202 gt=0 decoded=100 op_pred=63 elapsed=0.3m
eval 0031/0200 scene=fe667dc0752460a37390 gt=3 decoded=100 op_pred=47 elapsed=0.4m
eval 0041/0200 scene=11e8bd15bed065be3400 gt=2 decoded=99 op_pred=45 elapsed=0.5m
eval 0051/0200 scene=87b7a8761b585200d3ff gt=1 decoded=100 op_pred=78 elapsed=0.7m
eval 0061/0200 scene=a3cb9b50dca442394cbe gt=2 decoded=99 op_pred=38 elapsed=0.8m
eval 0071/0200 scene=25a9d6b2837276a9dbb7 gt=1 decoded=100 op_pred=66 elapsed=0.9m
eval 0081/0200 scene=6eefe1f0d26dfb5c9a9e gt=5 decoded=100 op_pred=68 elapsed=0.9m
eval 0091/0200 scene=22aeb0f719d57f84b0f7 gt=0 decoded=100 op_pred=88 elapsed=1.0m
eval 0101/0200 scene=d3df4c3febde0d7548bc gt=0 decoded=98 op_pred=98 elapsed=1.1m
eval 0111/0200 scene=858baec99120971a3e47 gt

## Checkpoint

In [56]:
# ============================================================
# DASO-Net — END-OF-DAY HANDOFF
# 2026-09-13
#
# PURPOSE
# -------
# Preserve the COMPLETE current Phase-5C development state
# before the Kaggle session is closed.
#
# Creates:
#
#   1. FULL package
#      - complete current code tree
#      - ALL dasonet_phase5b runs/checkpoints
#      - ALL dasonet_phase5c evaluations/diagnostics
#      - root configs/scripts
#      - six baseline snapshots: 250..1500
#      - rejected PCC-center experiments
#      - schedule probe / decoder diagnostics
#      - state summary / hashes / restore script
#
#   2. EMERGENCY RESUME package
#      - current code
#      - configs
#      - 1500 checkpoint
#      - history/status
#      - 1500 metrics/oracle/counterfactual
#      - handoff state
#
# Both ZIPs are integrity checked.
#
# IMPORTANT:
#   After PASS, DOWNLOAD BOTH ZIPs before closing Kaggle.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import sys
import textwrap
import zipfile
from datetime import datetime, timezone
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

WORK = Path(
    "/kaggle/working"
)

CODE_ROOT = (
    WORK
    / "DASO-Net-Phase5B-KaggleTraining"
)

TRAIN_RUN_ROOT = (
    WORK
    / "dasonet_phase5b"
    / "dasonet_phase5c_baseline_curve1000updates_v1"
)

SNAPSHOT_DIR = (
    TRAIN_RUN_ROOT
    / "snapshots"
)

EVAL_ROOT = (
    WORK
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
)

SUMMARY_1500 = (
    EVAL_ROOT
    / "baseline_learning_curve_1500_summary.json"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

BASE_TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

CONFIG_1500 = (
    WORK
    / "phase5c_baseline_curve1000updates_v1.json"
)

RUN_STATUS = (
    TRAIN_RUN_ROOT
    / "run_status.json"
)

HISTORY = (
    TRAIN_RUN_ROOT
    / "history.csv"
)

LATEST_CHECKPOINT = (
    SNAPSHOT_DIR
    / "epoch_005_opt_001500.pt"
)

METRIC_1500 = (
    EVAL_ROOT
    / "opt_1500"
    / "metrics"
    / "phase5c_metric_summary.json"
)

ORACLE_1500 = (
    EVAL_ROOT
    / "opt_1500"
    / "oracle"
    / "oracle_component_diagnostic.json"
)

CF_1500 = (
    EVAL_ROOT
    / "extended_best_counterfactual"
    / "counterfactual_center_diagnostic.json"
)


# Development evidence that must not disappear.
PCC_ABSOLUTE_COMPARISON = (
    WORK
    / "dasonet_phase5c"
    / "pcccenter1000_v2_evaluation"
    / "pcc_center_controlled_ablation_v2.json"
)

PCC_RELATIVE_COMPARISON = (
    WORK
    / "dasonet_phase5c"
    / "pcccenter_relative1000_v1_evaluation"
    / "pcc_center_three_way_comparison.json"
)

SCHEDULE_COMPARISON = (
    WORK
    / "dasonet_phase5c"
    / "scheduleprobe1000_evaluation_v1"
    / "schedule_capacity_comparison.json"
)


# ============================================================
# REQUIRED-STATE AUDIT
# ============================================================

required_paths = [
    CODE_ROOT,
    TRAIN_RUN_ROOT,
    EVAL_ROOT,
    SUMMARY_1500,
    DECODER,
    LAUNCHER,
    BASE_TRAINER,
    CONFIG_1500,
    RUN_STATUS,
    HISTORY,
    LATEST_CHECKPOINT,
    METRIC_1500,
    ORACLE_1500,
    CF_1500,
    PCC_ABSOLUTE_COMPARISON,
    PCC_RELATIVE_COMPARISON,
    SCHEDULE_COMPARISON,
]


for path in required_paths:

    if not path.exists():

        raise RuntimeError(
            "CRITICAL handoff input missing:\n"
            f"{path}"
        )


expected_snapshots = {
    250:
        SNAPSHOT_DIR
        / "epoch_000_opt_000250.pt",

    500:
        SNAPSHOT_DIR
        / "epoch_001_opt_000500.pt",

    750:
        SNAPSHOT_DIR
        / "epoch_002_opt_000750.pt",

    1000:
        SNAPSHOT_DIR
        / "epoch_003_opt_001000.pt",

    1250:
        SNAPSHOT_DIR
        / "epoch_004_opt_001250.pt",

    1500:
        SNAPSHOT_DIR
        / "epoch_005_opt_001500.pt",
}


for step, path in expected_snapshots.items():

    if not path.exists():

        raise RuntimeError(
            f"Missing required {step}-update snapshot:\n"
            f"{path}"
        )


# ============================================================
# HELPERS
# ============================================================

def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def human_bytes(
    size: int,
) -> str:

    units = [
        "B",
        "KiB",
        "MiB",
        "GiB",
        "TiB",
    ]

    value = float(
        size
    )

    for unit in units:

        if (
            value < 1024.0
            or unit == units[-1]
        ):

            return (
                f"{value:.2f} {unit}"
            )

        value /= 1024.0

    return (
        f"{size} B"
    )


IGNORE_NAMES = {
    "__pycache__",
    ".ipynb_checkpoints",
    ".pytest_cache",
    ".mypy_cache",
}


def ignore_copy(
    directory,
    names,
):

    ignored = []

    for name in names:

        if name in IGNORE_NAMES:
            ignored.append(
                name
            )

        if name.endswith(
            ".pyc"
        ):
            ignored.append(
                name
            )

    return ignored


def copy_directory(
    source: Path,
    destination: Path,
):

    shutil.copytree(
        source,
        destination,
        ignore=ignore_copy,
        dirs_exist_ok=True,
    )


def add_tree_to_zip(
    zip_path: Path,
    source_root: Path,
):

    with zipfile.ZipFile(
        zip_path,
        mode="w",
        allowZip64=True,
    ) as zf:

        files = sorted(
            p
            for p in source_root.rglob("*")
            if p.is_file()
        )

        for path in files:

            relative = path.relative_to(
                source_root
            )

            # Checkpoints and existing ZIPs are already effectively
            # compressed. Store them directly for speed/reliability.
            if path.suffix.lower() in {
                ".pt",
                ".pth",
                ".zip",
            }:

                compression = (
                    zipfile.ZIP_STORED
                )

            else:

                compression = (
                    zipfile.ZIP_DEFLATED
                )

            zf.write(
                path,
                arcname=str(
                    relative
                ),
                compress_type=compression,
            )


def verify_zip_basic(
    path: Path,
):

    with zipfile.ZipFile(
        path,
        "r",
    ) as zf:

        bad = zf.testzip()

        if bad is not None:

            raise RuntimeError(
                f"ZIP CRC failure: {bad}"
            )


# ============================================================
# LOAD CURRENT SCIENTIFIC STATE
# ============================================================

summary_1500 = json.loads(
    SUMMARY_1500.read_text(
        encoding="utf-8"
    )
)

metric_1500 = json.loads(
    METRIC_1500.read_text(
        encoding="utf-8"
    )
)

oracle_1500 = json.loads(
    ORACLE_1500.read_text(
        encoding="utf-8"
    )
)

cf_1500 = json.loads(
    CF_1500.read_text(
        encoding="utf-8"
    )
)

status = json.loads(
    RUN_STATUS.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# HARD CHECK: LATEST CHECKPOINT REALLY IS 1500
# ============================================================

import torch


latest_payload = torch.load(
    LATEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)


if int(
    latest_payload.get(
        "optimizer_step",
        -1,
    )
) != 1500:

    raise RuntimeError(
        "Latest handoff checkpoint is not optimizer step 1500."
    )


if int(
    latest_payload.get(
        "epoch",
        -1,
    )
) != 5:

    raise RuntimeError(
        "Latest handoff checkpoint is not epoch 5."
    )


if int(
    status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
) != 0:

    raise RuntimeError(
        "Current baseline status contains AMP skips."
    )


# ============================================================
# FROZEN DECODER CHECK
# ============================================================

decoder_sha = sha256_file(
    DECODER
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


if decoder_sha != EXPECTED_DECODER_SHA:

    raise RuntimeError(
        "Decoder hash changed unexpectedly.\n"
        f"Expected: {EXPECTED_DECODER_SHA}\n"
        f"Actual:   {decoder_sha}"
    )


# ============================================================
# PACKAGE PATHS
# ============================================================

STAMP = "20260913"

FULL_STAGE = (
    WORK
    / f"_DASO_NET_EOD_{STAMP}_FULL_STAGE"
)

EMERGENCY_STAGE = (
    WORK
    / f"_DASO_NET_EOD_{STAMP}_EMERGENCY_STAGE"
)

FULL_ZIP = (
    WORK
    / f"DASO-Net-Phase5C-EOD-{STAMP}-FULL.zip"
)

EMERGENCY_ZIP = (
    WORK
    / f"DASO-Net-Phase5C-EOD-{STAMP}-RESUME.zip"
)

FULL_SHA_FILE = (
    WORK
    / f"DASO-Net-Phase5C-EOD-{STAMP}-FULL.sha256.txt"
)

EMERGENCY_SHA_FILE = (
    WORK
    / f"DASO-Net-Phase5C-EOD-{STAMP}-RESUME.sha256.txt"
)

STATE_SIDECAR = (
    WORK
    / f"DASO-Net-Phase5C-EOD-{STAMP}-STATE.json"
)


for path in [
    FULL_STAGE,
    EMERGENCY_STAGE,
]:

    if path.exists():

        shutil.rmtree(
            path
        )


for path in [
    FULL_ZIP,
    EMERGENCY_ZIP,
    FULL_SHA_FILE,
    EMERGENCY_SHA_FILE,
    STATE_SIDECAR,
]:

    if path.exists():

        path.unlink()


FULL_STAGE.mkdir(
    parents=True
)

EMERGENCY_STAGE.mkdir(
    parents=True
)


full_workspace = (
    FULL_STAGE
    / "workspace"
)

full_handoff = (
    FULL_STAGE
    / "handoff"
)

full_workspace.mkdir(
    parents=True
)

full_handoff.mkdir(
    parents=True
)


# ============================================================
# FULL PACKAGE:
# COPY ALL CURRENT DASO-NET WORKING TREES
# ============================================================

print(
    "Copying complete current code tree..."
)

copy_directory(
    CODE_ROOT,
    full_workspace
    / CODE_ROOT.name,
)


print(
    "Copying ALL training/development run outputs..."
)

if (
    WORK
    / "dasonet_phase5b"
).exists():

    copy_directory(
        WORK
        / "dasonet_phase5b",
        full_workspace
        / "dasonet_phase5b",
    )


print(
    "Copying ALL Phase-5C evaluations/diagnostics..."
)

if (
    WORK
    / "dasonet_phase5c"
).exists():

    copy_directory(
        WORK
        / "dasonet_phase5c",
        full_workspace
        / "dasonet_phase5c",
    )


# ============================================================
# COPY ALL RELEVANT TOP-LEVEL WORKING FILES
#
# This intentionally captures configs, launch wrappers,
# previous frozen ZIPs, etc.
# ============================================================

excluded_prefixes = [
    f"DASO-Net-Phase5C-EOD-{STAMP}",
    f"_DASO_NET_EOD_{STAMP}",
]


top_level_copied = []


for path in sorted(
    WORK.iterdir()
):

    if not path.is_file():
        continue

    if any(
        path.name.startswith(
            prefix
        )
        for prefix in excluded_prefixes
    ):
        continue

    destination = (
        full_workspace
        / path.name
    )

    shutil.copy2(
        path,
        destination,
    )

    top_level_copied.append(
        path.name
    )


# ============================================================
# STATE RECORD
# ============================================================

ap = metric_1500[
    "mask_ap"
]

op = metric_1500[
    "operating_point"
]

sweep = metric_1500[
    "validation_threshold_sweep"
]

center = oracle_1500[
    "center_head"
]

geom = oracle_1500[
    "oracle_gt_cell_geometry"
]

mask = oracle_1500[
    "oracle_gt_cell_mask"
]


checkpoint_hashes = {
    str(step):
        {
            "path":
                str(path),

            "sha256":
                sha256_file(
                    path
                ),

            "bytes":
                path.stat().st_size,
        }

    for step, path in expected_snapshots.items()
}


state = {
    "handoff_version":
        "DASO-Net-Phase5C-EOD-20260913-v1",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "project":
        "DASO-Net",

    "current_phase":
        "Phase 5C",

    "status":
        (
            "Original-PCC baseline training successfully "
            "extended to 1500 optimizer updates."
        ),

    "latest_resume_checkpoint":
        {
            "path_in_original_session":
                str(
                    LATEST_CHECKPOINT
                ),

            "optimizer_step":
                1500,

            "epoch":
                5,

            "sha256":
                checkpoint_hashes[
                    "1500"
                ][
                    "sha256"
                ],
        },

    "all_baseline_snapshots":
        checkpoint_hashes,

    "runtime_frozen_for_current_curve":
        {
            "gpus":
                "2x Tesla T4",

            "native_tile":
                1024,

            "batch_per_gpu":
                1,

            "anchor_pair_batch_per_gpu":
                2,

            "gradient_accumulation":
                4,

            "effective_synthetic_batch":
                8,

            "synthetic_workers_per_rank":
                2,

            "anchor_workers_per_rank":
                1,

            "amp":
                "FP16",

            "amp_initial_scale":
                32,

            "amp_skips":
                0,

            "learning_rate":
                3e-4,

            "warmup_optimizer_steps":
                50,

            "post_warmup_schedule":
                "constant 3e-4 for current learning-curve diagnostic",
        },

    "current_1500_validation_diagnostic":
        {
            "mask_AP50":
                float(
                    ap[
                        "mask_ap50"
                    ]
                ),

            "mask_AP75":
                float(
                    ap[
                        "mask_ap75"
                    ]
                ),

            "mask_AP50_95":
                float(
                    ap[
                        "mask_ap50_95"
                    ]
                ),

            "best_F1":
                float(
                    sweep[
                        "best_f1"
                    ]
                ),

            "best_threshold":
                float(
                    sweep[
                        "best_threshold_by_f1"
                    ]
                ),

            "operating_recall_at_0_05":
                float(
                    op[
                        "recall"
                    ]
                ),

            "empty_FP_per_image_at_0_05":
                float(
                    op[
                        "empty_scene_fp_per_image"
                    ]
                ),

            "GT_center_score_mean":
                float(
                    center[
                        "gt_cell_score_stats"
                    ][
                        "mean"
                    ]
                ),

            "peak_recall_8px":
                float(
                    center[
                        "top100_peak_recall_within_8px"
                    ]
                ),

            "peak_recall_16px":
                float(
                    center[
                        "top100_peak_recall_within_16px"
                    ]
                ),

            "oracle_box_IoU_mean":
                float(
                    geom[
                        "box_iou_stats"
                    ][
                        "mean"
                    ]
                ),

            "oracle_GT_box_mask_IoU_mean":
                float(
                    mask[
                        "gt_box_crop_mask_iou_stats"
                    ][
                        "mean"
                    ]
                ),

            "counterfactual_exact_lift_mean":
                float(
                    cf_1500[
                        "exact_gt_cell"
                    ][
                        "composite_minus_clean_stats"
                    ][
                        "mean"
                    ]
                ),

            "counterfactual_composite_gt_clean_fraction":
                float(
                    cf_1500[
                        "exact_gt_cell"
                    ][
                        "fraction_composite_gt_clean"
                    ]
                ),
        },

    "learning_curve_conclusion":
        (
            "Not plateaued at 1500 updates. "
            "1500 is the current provisional best development checkpoint. "
            "Continue original-PCC baseline training tomorrow rather than "
            "redesigning architecture/losses."
        ),

    "pcc_center_decision":
        {
            "status":
                "REJECTED",

            "absolute_variant":
                (
                    "Reduced false positives but substantially degraded "
                    "AP, recall, counterfactual lift and mask learning."
                ),

            "relative_only_variant":
                (
                    "Recovered some performance but remained below "
                    "the original-PCC baseline."
                ),

            "final_development_rule":
                (
                    "Do not use detection-level PCC-center in the "
                    "current DASO-Net design. Preserve original "
                    "feature/embedding PCC."
                ),
        },

    "decoder":
        {
            "sha256":
                decoder_sha,

            "local_peak_kernel":
                3,

            "status":
                "validated and retained",
        },

    "scientific_scope":
        {
            "primary_task":
                "instance segmentation",

            "target":
                (
                    "visually resolvable suspected microplastic-like "
                    "particles / suspected surface plastic contamination"
                ),

            "rgb_claim_limit":
                (
                    "RGB does not chemically identify polymer type "
                    "or all microplastics."
                ),

            "deployment":
                "ordinary smartphone RGB",

            "external_test_tuning":
                False,

            "RSC_2025":
                (
                    "external detection test only; never train/tune"
                ),

            "Tioman_2025":
                (
                    "limited FTIR-linked metadata anchor; "
                    "not normal training"
                ),

            "Talanta_2026":
                (
                    "qualitative food/smartphone transfer only "
                    "without invented quantitative GT"
                ),
        },

    "frozen_hashes":
        {
            "phase1_morphology_manifest":
                (
                    "01722d67038b9ca44f9b0b11f5bac7a"
                    "3e588862bc2a86152cf817edaa3de16ee"
                ),

            "phase2_fruitbg_manifest":
                (
                    "d0108ac720cb03dd565cc6c1fb792762e"
                    "7afea808ee596f4b7b2be82e2f223fe"
                ),

            "renderer_v3_1_json":
                (
                    "80d1572e58f84750a975dbed5811b2ad5"
                    "6c1c4cc51bdf615a5e33df675aeec32"
                ),

            "renderer_py":
                (
                    "e1d74610a144e19791f23abaa2679a070"
                    "0189c5651f73bc33743aa4c24985a46"
                ),

            "phase5b_frozen_zip":
                (
                    "74338288f033a1dec3e0f1d66c271f65"
                    "80be3c1f1a4caf5e1e986a11ffde7983"
                ),

            "decoder_current":
                decoder_sha,
        },

    "required_kaggle_input_tomorrow":
        (
            "/kaggle/input/datasets/abinavprabhuv/"
            "dasonet-project/DASO-Net-Phase1A"
        ),

    "next_step_tomorrow":
        (
            "Restore handoff. Verify 1500 checkpoint and decoder hash. "
            "Review/continue the ORIGINAL-PCC baseline from optimizer "
            "step 1500 with PCC-center disabled. Do not touch external "
            "test datasets for model selection."
        ),

    "do_not_do":
        [
            "Do not resurrect manual RealSpike/calibration.",
            "Do not enable PCC-center.",
            "Do not overwrite the validated decoder without evidence.",
            "Do not tune on RSC/Tioman/Talanta.",
            "Do not restart training from zero when 1500 resume exists.",
        ],

    "top_level_files_preserved":
        top_level_copied,
}


STATE_PATH = (
    full_handoff
    / "PHASE5C_EOD_STATE.json"
)


STATE_PATH.write_text(
    json.dumps(
        state,
        indent=2,
    ),
    encoding="utf-8",
)


STATE_SIDECAR.write_text(
    json.dumps(
        state,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# HUMAN-READABLE PROJECT HANDOFF
# ============================================================

handoff_md = r"""
# DASO-Net End-of-Day Handoff — 2026-09-13

## Current position

Phase 5C development is active.

The original DASO-Net architecture, original feature/embedding PCC,
InstanceSegLoss and CRPA are currently retained.

Detection-level PCC-center was tested in two controlled variants and
REJECTED. Do not reactivate it unless fundamentally new evidence appears.

The original-PCC baseline was trained to 1500 optimizer updates.

Latest resumable snapshot:

`workspace/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapshots/epoch_005_opt_001500.pt`

The six preserved learning-curve snapshots are:

- 250 updates
- 500 updates
- 750 updates
- 1000 updates
- 1250 updates
- 1500 updates

## 1500-update development result

Synthetic validation diagnostic, frozen deterministic 200-scene subset:

- Mask AP50: 0.1794
- Mask AP75: 0.0334
- Mask AP50:95: 0.0610
- Best validation F1: 0.2815
- Best diagnostic threshold: 0.30
- Peak recall <= 8 px: 0.7125
- Peak recall <= 16 px: 0.8725
- Oracle box IoU mean: 0.5275
- Oracle GT-box mask IoU mean: 0.4345
- GT-box mask IoU >= 0.50: 0.4400
- Empty-scene FP/image at threshold 0.05: 60.5143

Counterfactual center behavior at 1500:

- Clean GT-cell score mean: 0.0127
- Composite GT-cell score mean: 0.1900
- Exact composite-clean lift mean: +0.1773
- Composite > clean: 0.9675
- Local peak lift mean: +0.2244

Interpretation: learning has NOT plateaued at 1500.

## Training state

Current development recipe:

- 2 x Tesla T4
- native 1024
- synthetic batch/GPU = 1
- anchor pair batch/GPU = 2
- accumulation = 4
- effective synthetic batch = 8
- synthetic workers/rank = 2
- anchor workers/rank = 1
- AMP FP16
- initial GradScaler scale = 32
- zero AMP skipped updates
- LR = 3e-4
- warm-up = 50 optimizer updates
- current learning-curve diagnostic uses constant 3e-4 after warm-up

Current validated decoder SHA256:

`9799eb1be568e3d6e0ce82b7c99df7a0ed05d271cd44ff24b612526cc18537d1`

## Tomorrow

Restore this package into `/kaggle/working`.

Attach the frozen project dataset:

`/kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A`

Verify:

- decoder SHA
- 1500 checkpoint SHA
- optimizer_step == 1500
- epoch == 5
- PCC-center remains disabled

Then continue development from the 1500-update original-PCC checkpoint.
Do NOT retrain from zero.

Do not use RSC, Tioman or Talanta for model selection or tuning.

## Scientific scope

DASO-Net is an RGB screening system for visually resolvable suspected
microplastic-like surface particles / suspected surface plastic contamination.

It does not chemically identify polymer type and must not claim detection of
all microplastics.

Primary task: instance segmentation.

Deployment target: ordinary smartphone RGB.

## Frozen major assets

Morphology manifest SHA256:
`01722d67038b9ca44f9b0b11f5bac7a3e588862bc2a86152cf817edaa3de16ee`

FruitBG manifest SHA256:
`d0108ac720cb03dd565cc6c1fb792762e7afea808ee596f4b7b2be82e2f223fe`

Renderer v3.1 JSON SHA256:
`80d1572e58f84750a975dbed5811b2ad56c1c4cc51bdf615a5e33df675aeec32`

Renderer Python SHA256:
`e1d74610a144e19791f23abaa2679a0700189c5651f73bc33743aa4c24985a46`

Phase-5B frozen ZIP SHA256:
`74338288f033a1dec3e0f1d66c271f6580be3c1f1a4caf5e1e986a11ffde7983`

Current decoder SHA256:
`9799eb1be568e3d6e0ce82b7c99df7a0ed05d271cd44ff24b612526cc18537d1`
"""


(
    full_handoff
    / "PROJECT_HANDOFF.md"
).write_text(
    textwrap.dedent(
        handoff_md
    ).strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# TOMORROW RESTORE SCRIPT INSIDE PACKAGE
# ============================================================

restore_script = r'''
from __future__ import annotations

import hashlib
import json
import shutil
from pathlib import Path

import torch


CURRENT = Path.cwd()

PACKAGE_ROOT = Path(__file__).resolve().parents[1]

WORKSPACE = (
    PACKAGE_ROOT
    / "workspace"
)

HANDOFF = (
    PACKAGE_ROOT
    / "handoff"
)

STATE = json.loads(
    (
        HANDOFF
        / "PHASE5C_EOD_STATE.json"
    ).read_text(
        encoding="utf-8"
    )
)

TARGET = Path(
    "/kaggle/working"
)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


if not WORKSPACE.exists():
    raise RuntimeError(
        f"Packaged workspace missing: {WORKSPACE}"
    )


for source in WORKSPACE.iterdir():

    destination = (
        TARGET
        / source.name
    )

    if source.is_dir():

        shutil.copytree(
            source,
            destination,
            dirs_exist_ok=True,
        )

    else:

        shutil.copy2(
            source,
            destination,
        )


decoder = (
    TARGET
    / "DASO-Net-Phase5B-KaggleTraining"
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

expected_decoder = (
    STATE[
        "decoder"
    ][
        "sha256"
    ]
)

actual_decoder = sha256_file(
    decoder
)

if actual_decoder != expected_decoder:

    raise RuntimeError(
        "Restored decoder SHA mismatch."
    )


checkpoint = (
    TARGET
    / "dasonet_phase5b"
    / "dasonet_phase5c_baseline_curve1000updates_v1"
    / "snapshots"
    / "epoch_005_opt_001500.pt"
)

payload = torch.load(
    checkpoint,
    map_location="cpu",
    weights_only=False,
)

if int(
    payload.get(
        "optimizer_step",
        -1,
    )
) != 1500:

    raise RuntimeError(
        "Restored checkpoint is not optimizer step 1500."
    )

if int(
    payload.get(
        "epoch",
        -1,
    )
) != 5:

    raise RuntimeError(
        "Restored checkpoint is not epoch 5."
    )


expected_checkpoint = (
    STATE[
        "latest_resume_checkpoint"
    ][
        "sha256"
    ]
)

actual_checkpoint = sha256_file(
    checkpoint
)

if actual_checkpoint != expected_checkpoint:

    raise RuntimeError(
        "Restored 1500-checkpoint SHA mismatch."
    )


print(
    "============================================================"
)

print(
    "DASO-Net EOD RESTORE: PASS"
)

print(
    "============================================================"
)

print(
    "Decoder SHA: PASS"
)

print(
    "1500 checkpoint SHA: PASS"
)

print(
    "Optimizer step: 1500"
)

print(
    "Epoch: 5"
)

print()

print(
    "NEXT:"
)

print(
    STATE[
        "next_step_tomorrow"
    ]
)
'''


(
    full_handoff
    / "restore_tomorrow.py"
).write_text(
    textwrap.dedent(
        restore_script
    ).strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# REQUIRED INPUT NOTE
# ============================================================

(
    full_handoff
    / "REQUIRED_KAGGLE_INPUTS.txt"
).write_text(
    (
        "Tomorrow attach the existing persistent Kaggle dataset:\n\n"
        "/kaggle/input/datasets/abinavprabhuv/"
        "dasonet-project/DASO-Net-Phase1A\n\n"
        "The immutable project dataset itself is not duplicated "
        "inside this handoff ZIP because it already exists as a "
        "persistent Kaggle input dataset.\n"
    ),
    encoding="utf-8",
)


# ============================================================
# FULL FILE MANIFEST
#
# Manifest intentionally excludes itself.
# ============================================================

manifest_entries = []


for path in sorted(
    p
    for p in FULL_STAGE.rglob("*")
    if p.is_file()
):

    relative = path.relative_to(
        FULL_STAGE
    )

    if str(
        relative
    ).startswith(
        "handoff/FULL_FILE_MANIFEST"
    ):
        continue

    manifest_entries.append(
        {
            "path":
                str(
                    relative
                ),

            "bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


manifest = {
    "manifest_version":
        1,

    "file_count":
        len(
            manifest_entries
        ),

    "total_uncompressed_bytes":
        sum(
            x["bytes"]
            for x in manifest_entries
        ),

    "files":
        manifest_entries,
}


MANIFEST_PATH = (
    full_handoff
    / "FULL_FILE_MANIFEST.json"
)


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)


MANIFEST_SHA = sha256_file(
    MANIFEST_PATH
)


(
    full_handoff
    / "FULL_FILE_MANIFEST.sha256.txt"
).write_text(
    (
        f"{MANIFEST_SHA}  "
        "FULL_FILE_MANIFEST.json\n"
    ),
    encoding="utf-8",
)


# Completion marker is included in ZIP.
(
    full_handoff
    / "HANDOFF_COMPLETE.marker"
).write_text(
    (
        "DASO-Net Phase 5C end-of-day handoff complete.\n"
        "Latest resumable optimizer step: 1500\n"
        "PCC-center: rejected/disabled\n"
        f"Decoder SHA256: {decoder_sha}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# BUILD FULL ZIP
# ============================================================

print()
print(
    "Building FULL handoff ZIP..."
)


add_tree_to_zip(
    FULL_ZIP,
    FULL_STAGE,
)


verify_zip_basic(
    FULL_ZIP
)


FULL_ZIP_SHA = sha256_file(
    FULL_ZIP
)


FULL_SHA_FILE.write_text(
    (
        f"{FULL_ZIP_SHA}  "
        f"{FULL_ZIP.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# VERIFY FULL ZIP AGAINST MANIFEST
# ============================================================

print(
    "Verifying FULL ZIP file hashes..."
)


with zipfile.ZipFile(
    FULL_ZIP,
    "r",
) as zf:

    archived_names = set(
        zf.namelist()
    )

    for entry in manifest_entries:

        rel = entry[
            "path"
        ]

        if rel not in archived_names:

            raise RuntimeError(
                f"File missing from FULL ZIP: {rel}"
            )

        h = hashlib.sha256()

        with zf.open(
            rel,
            "r",
        ) as f:

            for block in iter(
                lambda: f.read(
                    4 * 1024 * 1024
                ),
                b"",
            ):

                h.update(
                    block
                )

        if (
            h.hexdigest()
            != entry[
                "sha256"
            ]
        ):

            raise RuntimeError(
                "FULL ZIP SHA verification failed for:\n"
                f"{rel}"
            )


# ============================================================
# BUILD EMERGENCY RESUME PACKAGE
# ============================================================

em_workspace = (
    EMERGENCY_STAGE
    / "workspace"
)

em_handoff = (
    EMERGENCY_STAGE
    / "handoff"
)

em_workspace.mkdir(
    parents=True
)

em_handoff.mkdir(
    parents=True
)


# Complete current code.
copy_directory(
    CODE_ROOT,
    em_workspace
    / CODE_ROOT.name,
)


# Required config files / wrappers from workspace root.
for path in sorted(
    WORK.iterdir()
):

    if not path.is_file():
        continue

    if (
        path.name.startswith(
            "phase5"
        )
        or
        path.name.startswith(
            "launch_phase"
        )
    ):

        shutil.copy2(
            path,
            em_workspace
            / path.name,
        )


# Minimal resumable training structure.
em_run_root = (
    em_workspace
    / "dasonet_phase5b"
    / "dasonet_phase5c_baseline_curve1000updates_v1"
)

(
    em_run_root
    / "snapshots"
).mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copy2(
    LATEST_CHECKPOINT,
    em_run_root
    / "snapshots"
    / LATEST_CHECKPOINT.name,
)


for source in [
    RUN_STATUS,
    HISTORY,
]:

    shutil.copy2(
        source,
        em_run_root
        / source.name,
    )


# Minimal evaluation evidence.
em_eval_root = (
    em_workspace
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
)

copy_directory(
    EVAL_ROOT
    / "opt_1500",
    em_eval_root
    / "opt_1500",
)

copy_directory(
    EVAL_ROOT
    / "extended_best_counterfactual",
    em_eval_root
    / "extended_best_counterfactual",
)


for source in [
    SUMMARY_1500,
]:

    shutil.copy2(
        source,
        em_eval_root
        / source.name,
    )


# Copy handoff metadata.
for source in [
    STATE_PATH,
    full_handoff
    / "PROJECT_HANDOFF.md",
    full_handoff
    / "REQUIRED_KAGGLE_INPUTS.txt",
    full_handoff
    / "restore_tomorrow.py",
    full_handoff
    / "HANDOFF_COMPLETE.marker",
]:

    shutil.copy2(
        source,
        em_handoff
        / source.name,
    )


# Emergency manifest.
em_entries = []


for path in sorted(
    p
    for p in EMERGENCY_STAGE.rglob("*")
    if p.is_file()
):

    rel = path.relative_to(
        EMERGENCY_STAGE
    )

    em_entries.append(
        {
            "path":
                str(
                    rel
                ),

            "bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


(
    em_handoff
    / "EMERGENCY_FILE_MANIFEST.json"
).write_text(
    json.dumps(
        {
            "manifest_version":
                1,

            "file_count":
                len(
                    em_entries
                ),

            "files":
                em_entries,
        },
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "Building emergency RESUME ZIP..."
)


add_tree_to_zip(
    EMERGENCY_ZIP,
    EMERGENCY_STAGE,
)


verify_zip_basic(
    EMERGENCY_ZIP
)


EMERGENCY_ZIP_SHA = sha256_file(
    EMERGENCY_ZIP
)


EMERGENCY_SHA_FILE.write_text(
    (
        f"{EMERGENCY_ZIP_SHA}  "
        f"{EMERGENCY_ZIP.name}\n"
    ),
    encoding="utf-8",
)


# ============================================================
# FINAL EXTERNAL VERIFICATION
# ============================================================

with zipfile.ZipFile(
    FULL_ZIP,
    "r",
) as zf:

    required_inside_full = [
        (
            "workspace/"
            "DASO-Net-Phase5B-KaggleTraining/"
            "src/dasonet/model/decoding.py"
        ),
        (
            "workspace/"
            "dasonet_phase5b/"
            "dasonet_phase5c_baseline_curve1000updates_v1/"
            "snapshots/epoch_005_opt_001500.pt"
        ),
        (
            "workspace/"
            "dasonet_phase5c/"
            "baseline_curve1000updates_v1/"
            "baseline_learning_curve_1500_summary.json"
        ),
        "handoff/PHASE5C_EOD_STATE.json",
        "handoff/PROJECT_HANDOFF.md",
        "handoff/restore_tomorrow.py",
        "handoff/HANDOFF_COMPLETE.marker",
    ]

    names = set(
        zf.namelist()
    )

    for required in required_inside_full:

        if required not in names:

            raise RuntimeError(
                "Critical file absent from FULL ZIP:\n"
                f"{required}"
            )


with zipfile.ZipFile(
    EMERGENCY_ZIP,
    "r",
) as zf:

    required_inside_emergency = [
        (
            "workspace/"
            "dasonet_phase5b/"
            "dasonet_phase5c_baseline_curve1000updates_v1/"
            "snapshots/epoch_005_opt_001500.pt"
        ),
        (
            "workspace/"
            "DASO-Net-Phase5B-KaggleTraining/"
            "scripts/train_phase5b.py"
        ),
        "handoff/PHASE5C_EOD_STATE.json",
        "handoff/restore_tomorrow.py",
    ]

    names = set(
        zf.namelist()
    )

    for required in required_inside_emergency:

        if required not in names:

            raise RuntimeError(
                "Critical file absent from emergency ZIP:\n"
                f"{required}"
            )


# ============================================================
# CLEAN TEMP STAGING
#
# ZIPs + state sidecar remain.
# ============================================================

shutil.rmtree(
    FULL_STAGE
)

shutil.rmtree(
    EMERGENCY_STAGE
)


# ============================================================
# FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "DASO-NET END-OF-DAY HANDOFF: PASS"
)

print(
    "============================================================"
)

print()

print(
    "Latest model checkpoint:"
)

print(
    LATEST_CHECKPOINT
)

print(
    "  optimizer step: 1500"
)

print(
    "  epoch:          5"
)

print(
    "  SHA256:",
    checkpoint_hashes[
        "1500"
    ][
        "sha256"
    ],
)

print()

print(
    "Validated decoder SHA256:"
)

print(
    decoder_sha
)

print()

print(
    "FULL PACKAGE:"
)

print(
    FULL_ZIP
)

print(
    "  size:",
    human_bytes(
        FULL_ZIP.stat().st_size
    ),
)

print(
    "  SHA256:",
    FULL_ZIP_SHA,
)

print()

print(
    "EMERGENCY RESUME PACKAGE:"
)

print(
    EMERGENCY_ZIP
)

print(
    "  size:",
    human_bytes(
        EMERGENCY_ZIP.stat().st_size
    ),
)

print(
    "  SHA256:",
    EMERGENCY_ZIP_SHA,
)

print()

print(
    "STATE SIDECAR:"
)

print(
    STATE_SIDECAR
)

print()

print(
    "Full archive CRC:                PASS"
)

print(
    "Full per-file SHA verification:  PASS"
)

print(
    "Emergency archive CRC:           PASS"
)

print(
    "1500 checkpoint verification:    PASS"
)

print(
    "Decoder verification:            PASS"
)

print(
    "PCC-center decision preserved:   REJECTED"
)

print()

print(
    "============================================================"
)

print(
    "CRITICAL — BEFORE CLOSING THIS NOTEBOOK"
)

print(
    "============================================================"
)

print(
    "Download BOTH ZIP files to your computer."
)

print(
    "Also download the two .sha256.txt files if convenient."
)

print(
    "Do not close Kaggle until the downloads have completed."
)

print()


# Clickable links in Kaggle/Jupyter where supported.
try:

    from IPython.display import display, FileLink

    display(
        FileLink(
            str(
                FULL_ZIP
            )
        )
    )

    display(
        FileLink(
            str(
                EMERGENCY_ZIP
            )
        )
    )

    display(
        FileLink(
            str(
                FULL_SHA_FILE
            )
        )
    )

    display(
        FileLink(
            str(
                EMERGENCY_SHA_FILE
            )
        )
    )

except Exception as exc:

    print(
        "Clickable FileLink display unavailable:",
        exc,
    )

Copying complete current code tree...
Copying ALL training/development run outputs...
Copying ALL Phase-5C evaluations/diagnostics...

Building FULL handoff ZIP...
Verifying FULL ZIP file hashes...
Building emergency RESUME ZIP...

DASO-NET END-OF-DAY HANDOFF: PASS

Latest model checkpoint:
/kaggle/working/dasonet_phase5b/dasonet_phase5c_baseline_curve1000updates_v1/snapshots/epoch_005_opt_001500.pt
  optimizer step: 1500
  epoch:          5
  SHA256: 258b9d4457c367373c07f7e3468445591fe824dc5ceef27e8cd338fc4b76c6f5

Validated decoder SHA256:
9799eb1be568e3d6e0ce82b7c99df7a0ed05d271cd44ff24b612526cc18537d1

FULL PACKAGE:
/kaggle/working/DASO-Net-Phase5C-EOD-20260913-FULL.zip
  size: 398.19 MiB
  SHA256: 7478783afe8aedd8459ab8779499e2972816148e06b3387ff5baa7beaf1a1bda

EMERGENCY RESUME PACKAGE:
/kaggle/working/DASO-Net-Phase5C-EOD-20260913-RESUME.zip
  size: 13.78 MiB
  SHA256: 2cd490ed9b5c7333d43c89ef01587cad5c0a6a75ff86e6ae586059f1e93b660c

STATE SIDECAR:
/kaggle/working/DASO-Net-Phase

/kaggle/working/DASO-Net-Phase5C-EOD-20260913-FULL.zip

/kaggle/working/DASO-Net-Phase5C-EOD-20260913-RESUME.zip

/kaggle/working/DASO-Net-Phase5C-EOD-20260913-FULL.sha256.txt

/kaggle/working/DASO-Net-Phase5C-EOD-20260913-RESUME.sha256.txt

## Storage policy

The trainer intentionally writes only `best.pt`, `last.pt`, `history.csv`, configs and status files. It does **not** cache the 40,000 rendered scenes or dump full predictions, keeping persistent output far below Kaggle's ~19 GB allowance.
